# <center>Claude Code 源码课·浓缩版第 1 节：Agent 能做什么 & 凭什么敢让它做</center>

&emsp;&emsp;今天我们聚焦的，是一份意外公开的工业级 Agent 源代码。2026 年 3 月，`Claude Code` 的一个 npm（Node.js 的包发布平台，类似 Python 的 PyPI）发布包因为打包配置疏漏，泄露了一份能反向还原出完整源码的 source map 文件（一种把压缩后的发布代码映射回原始源码的「调试地图」）——这意味着我们第一次有机会，不靠官方文档、不靠逆向猜测，直接打开一个真正跑在千万开发者机器上的 Agent，看它内部到底长什么样。这份泄露快照的版本是 `v2.1.88`，一共 1884 个文件、512,664 行代码。

&emsp;&emsp;51 万行——这个数字本身就值得你停下来想三秒。如果你写过哪怕最简单的一个 Agent，你心里大概有个数：让大模型「思考一步、调一个工具、看结果、再思考」这套循环，核心逻辑三十行 Python 就能跑起来。那剩下的五十一万行，到底在干什么？「不会都是冗余吧？」——如果你心里冒出这个念头，恰恰说明这门课对你有用，这正是我们要一起回答的问题。我会先带你亲手跑一个三十行的朴素 Agent，让你看清它「能跑」和「能用」之间隔着多远；然后我们一起打开真实源码，沿着「能力」和「约束」两条主线，一层一层挖下去——你会看到工业级 Agent 是怎么把一个会思考的模型，包装成一个你敢交给生产环境的工具的。

&emsp;&emsp;为了把这条主线讲透，接下来我们会依次打开七块内容：那次泄露事件的来龙去脉；一个三十行的朴素 Agent 和它在生产环境下会引爆的四个致命问题；一张把 51 万行装进脑子的五层架构地图；QueryLoop 与 Tool 协议这两块「能力地基」的源码深拆；扩展三件套——Skill（给 Agent 加知识）、MCP（接外部工具的开放协议）、Hook（在行为关键点插钩子拦截）的三个可运行 Python MVP；把这三件套打包成可安装、可版本化、可被企业管控的分发单元——plugins；最后是安全约束这块「能力的反面」。这三个扩展词后面都有专章细讲，这里你只要先有个印象。我们现在就从那份泄露快照开始。

> 📌 **目标受众与前置要求**：本课面向有编程基础、想理解工业级 Agent 工程内核的开发者。技术上你需要会读 Python、知道 LLM API 怎么调、听过 Agent 这个概念，**不需要**你写过 TypeScript，也**不需要**你跑通完整的 `Claude Code` 项目——源码我们只读不跑，三个可运行的演示全部用 Python 重写。

> 📌 **学完本节你将带走 5 件产物**：① 一张能徒手画出的五层架构地图；② 对 QueryLoop 核心循环和「`stop_reason` 不可信」陷阱的完整理解；③ 一句话说清「为什么 40 多个异构工具能被同一个循环无差别调度」；④ 三段可以直接抄进你自己项目的扩展机制 MVP（Skill / MCP / Hook）和一句 know/do/intercept 选型口诀；⑤ 说清「安全为什么是管线而不是替代」、能力越强为什么越要约束。

> 💡 **学完不能做（诚实划界）**：本课不会让你能从零复刻一个 `Claude Code`，也不会逐行讲完 51 万行——我们讲的是可迁移的工程内核，不是源码导读。上下文压缩、长期记忆、多 Agent 成本这三个问题，本节只点到为止，完整解法留给第 3 节《多智能体与上下文工程》。

> 📅 **时效性说明**：本课全部源码引用截止 2026 年 5 月，基于 `Claude Code` 泄露快照 `v2.1.88`（src.zip，1884 文件 / 512,664 行）当时的代码状态。所有 `文件:行号` 引用都是真实可核对的——课件里出现的每一个行数、每一处函数位置，都来自对这份快照的实测，你拿到同一份快照后可以用 `wc -l` 和 `grep` 逐条复核。涉及版本敏感的数字，我们都标注了实测命令。

> 📌 **【本课元工具 · 吃透任意开源项目的通用提示词】**

&emsp;&emsp;在进入正课之前，先把这门课最该带走的元能力交给你。下面这段提示词，就是我们用来把 51 万行 `Claude Code` 吃透的同一套方法——把 `{项目}` 换成任何一个你想拆的开源项目，分阶段发给 AI，你就能复现同样的「吃透」过程。它刻意把每一阶段都钉在「可核实的源码与行为」上，而不是让 AI 替你读项目（那恰恰是这套方法要消灭的）。整段可直接复制：

```text
【吃透任意开源项目 · 通用提示词】把 {项目} 换成你要拆的开源项目，按阶段发给 AI。

角色：资深架构考古学家。不靠官方吹嘘，只凭源码 + 可观测行为，重建一个陌生大型开源项目的架构与设计意图。
输入（必填）：
  - 项目源码路径 / 仓库：{path 或 repo}
  - 我带着这 3 个尖锐问题进场（不通读、不让你替我定该关心什么）：{Q1} / {Q2} / {Q3}
  - 可选交叉证据：README / --help / 官方 changelog / 我能跑出的行为日志

任务（六阶段，每阶段产出后停下等我确认，不许一口气全做）：
  1. 规模与分层：用 wc -l / ls / find 实测顶层结构，给 3–7 层功能分层地图（每层职责 + 代码量级 + 我能复跑的实测命令）
  2. 主干一线：选一条最常用 happy path，端到端追它经过哪些文件/函数（带 file:line），画状态机/数据流；从命名猜的明确标出
  3. 核心契约：找出那个「让 N 种异构组件被同一套逻辑调度」的统一接口/契约（带 file:line）
  4. 扩展点：系统从哪些口子允许第三方加能力（配置/协议/事件/插件），各自机制
  5. 约束与安全：它如何兜住模型/输入的不确定性（校验/权限/沙箱/断路器/预算）
  6. 可迁移设计模式：抽 3–5 个不依赖该项目、能抄进我自己项目的设计决策，每个落到一对设计张力上

证据纪律（硬约束，违反则该条作废）：
  - 每条断言贴色标：[源码 file:line 实测] / [行为] / [文档] / [推断] / [待核验]；你对「内部如何实现」的描述默认先进 [待核验]
  - 口径依赖的数字（行数/文件数/字段数）只给量级 + 我能复跑的实测命令，不给「精确无误」的绝对值
  - 你的归纳命名（如「L3 层」「XX 引擎群」）必须显式声明「这是教学归纳，非项目官方术语」
  - 查不到证据就说「未找到」，禁止用合理猜测填 file:line；涉及我没给的文件，明说「需要看 X」

反模式（出现即自我纠正）：把「它内部怎么实现」当结论答（应转成「在什么可观察现象/哪段源码能判断」）；把上一轮 [待核验] 当地基继续推；用「通常/据我所知」凑。

输出格式：每阶段一张表或一张图骨架 + 关键 file:line 清单 + 本阶段「我还没核实的存疑点」单列。
```

&emsp;&emsp;模块级深挖（摸清某个模块的底层运行逻辑 + 让 AI 直接产出可运行的最小 MVP）见每一章开头的【本章动手】；跨模块快速复刻见文末附录。带着这把工具，我们进入正课。

---

## <center>第一章：51 万行泄露快照，到底藏着什么</center>

&emsp;&emsp;这一章只花你五分钟，做一件事：把你的注意力钉在一个问题上。这门课不是源码导读，不会带你逐个文件读 51 万行——那既不可能，对你也没价值。要做的是借这份难得的真实样本，回答一个所有写过 Agent 的人都会好奇的问题：一个工业级 Agent，比起教科书里那个三十行的循环，到底多出了什么？这一章先把这个悬念立住，第二章你就动手验证。

### 1.1 泄露事件

&emsp;&emsp;先把背景三十秒说清楚，然后你就可以把它放下、不再纠缠了。2026 年 3 月，`Claude Code` 通过 npm 发布新版本时，打包配置（`.npmignore`）漏掉了对 source map 文件的排除。source map 本来是给浏览器调试用的「源码地图」，它把压缩混淆后的发布代码反向映射回原始 TypeScript 源码。结果就是，任何人下载这个 npm 包，都能还原出接近完整的项目源码。社区很快把它整理成了一份 src.zip 快照，版本停在 `v2.1.88`。

&emsp;&emsp;这件事的法律与伦理细节我们不展开，也不夸大它的戏剧性。对你这门课来说，它只有一个价值：你第一次能用「读源码」而不是「读文档 + 猜」的方式，去理解一个真实跑在生产环境里的 Agent。所以从下一节开始，泄露事件本身就退场了，登场的是代码。

> **【这门课的态度】**：我们引用的所有数字，都以对这份快照的实测为准。课件里凡是出现「某文件多少行」「某目录多少个文件」，背后都有一条 `wc -l` 或 `ls` 命令。所以每个数字我都会告诉你「我用什么命令数的」，你完全可以自己复核，不必信口头转述。

### 1.2 51 万行不是一团乱码

&emsp;&emsp;现在把那个核心问题正式抛到你面前。这份快照 `wc -l` 数下来是 512,664 行代码、1884 个文件。直觉上，你可能觉得这么大的代码量里一定混着大量重复、历史包袱、用不上的边角料。但当你真的打开它，会发现一个和直觉相反的事实：这 51 万行有清晰的分层，绝大多数代码不是在「让 AI 更聪明」，而是在「约束 AI、支撑 AI、兜住 AI 的不确定性」。

&emsp;&emsp;所以这门课真正想让你记住的悬念是这样一句话：**这 51 万行，到底是在放大 AI，还是在约束 AI？** 这个问题不是修辞，它有一个相当确定的答案，而且这个答案会彻底改变你对「Agent 工程」的理解。要回答它，你得先有一个对照物——一个最朴素的 Agent，看它到底缺了什么。下一章，你就亲手把这个对照物跑起来。

---

## <center>第二章：30 行 Agent + 四个致命问题</center>

&emsp;&emsp;上一章我们立了一个悬念：51 万行到底在干什么。要回答它，最好的办法不是直接读那 51 万行，而是先建一个对照组。这一章我们用大约三十行 Python，搭一个能跑的朴素 Agent——它确实能完成任务，能调工具，能多轮对话。然后我们把它放进「生产环境」这个放大镜下，看它会在哪些地方崩。这一章抛出的四个问题会贯穿后面每一章（甚至延伸到第 3 节）——后面每一块源码，都是在回答其中某一个问题。

### 2.1 朴素 Agent 的核心结构

&emsp;&emsp;先打碎一个直觉——很多人第一次写 Agent 时心里都有过这个念头：「Agent 不就是个带工具的 while 循环吗，三十行够了，那些大厂是不是过度工程了？」答案可能让你意外：**这个循环本身确实够了，但「够跑」和「敢用」之间，差的就是那五十一万行**。学完这节你会回头感谢这五十一万行。我们先把这个三十行循环写出来，让它真的跑起来，你才能切身体会它差在哪。

> 📌 **【动手 · 先反向用一下元工具】**：看下面代码前，不妨先把这句发给 AI 自测——「用纯 Python 标准库写一个会跑的、≤30 行的朴素 Agent（发历史→看 stop_reason→调工具→循环），跑完再逼你自己列出它放进生产环境会引爆的 4 类问题，每类一句」。自己跑出来再回来对照本章，你对「够跑 vs 敢用」的体感会深得多。

&emsp;&emsp;一个 Agent 的最小内核就三件事：把对话历史发给模型、模型决定调用哪个工具、执行工具并把结果塞回历史，然后循环。下面这段代码用一个本地模拟的 `mock_llm` 替代真实大模型调用（这样你不配 API key 也能直接跑通，看清结构），它演示的是这套循环的骨架。运行后你会看到三轮交互：模型先调 `read_file`，再调 `calc`，最后给出结论——这就是一个 Agent「能跑」的全部。

In [ ]:
# 朴素 Agent 最小内核：演示 while 循环 + 工具调用 + stop_reason 判断
# 用本地 mock_llm 替代真实大模型，无需 API key 即可直接运行看清结构

# --- 第 1 部分：工具表（每个工具就是一个普通 Python 函数）---
def tool_read_file(args):
    """模拟读文件工具：真实场景会 open(path).read()，这里返回假内容。"""
    return f"(文件 {args['path']} 的内容：销售额 = 1200)"

def tool_calc(args):
    """模拟计算工具：真实场景会 eval 或调计算引擎，这里直接算。"""
    return f"(计算结果：{args['expr']} = {eval(args['expr'])})"

# 工具注册表：名字 -> 函数。Agent 靠这张表把"工具名"翻译成"真实动作"
TOOLS = {"read_file": tool_read_file, "calc": tool_calc}

# --- 第 2 部分：模拟大模型。真实场景是 client.messages.create(...) ---
def mock_llm(history):
    """
    模拟 LLM 的决策：根据历史里已执行过的工具数，决定下一步动作。
    返回 dict 含两种形态：
      {"stop_reason": "tool_use", "tool": 名, "args": 参数}  -> 要调工具
      {"stop_reason": "end_turn", "text": 最终回答}           -> 任务结束
    """
    done = [h for h in history if h.get("role") == "tool"]  # 已执行的工具调用
    if len(done) == 0:
        # 第 1 步：模型决定先读文件
        return {"stop_reason": "tool_use", "tool": "read_file",
                "args": {"path": "data.txt"}}
    if len(done) == 1:
        # 第 2 步：拿到文件内容后，模型决定算一笔
        return {"stop_reason": "tool_use", "tool": "calc",
                "args": {"expr": "1200 * 0.15"}}
    # 第 3 步：模型认为任务完成，给出最终答案
    return {"stop_reason": "end_turn", "text": "销售额 1200，15% 提成是 180"}

# --- 第 3 部分：Agent 主循环。这就是"30 行 Agent"的全部 ---
def run_agent(user_input, max_turns=10):
    """朴素 Agent 主循环：发历史 -> 看 stop_reason -> 调工具或结束。"""
    history = [{"role": "user", "content": user_input}]  # 对话历史
    for turn in range(max_turns):                        # 设上限防死循环
        decision = mock_llm(history)                     # 把历史发给模型
        # 关键分支：模型说要调工具就调，说结束就返回
        if decision["stop_reason"] == "tool_use":
            name, args = decision["tool"], decision["args"]
            result = TOOLS[name](args)                    # 执行工具
            print(f"[第{turn+1}轮] 调用 {name}({args}) -> {result}")
            # 把工具结果塞回历史，下一轮模型能看到
            history.append({"role": "assistant", "tool": name})
            history.append({"role": "tool", "content": result})
        else:
            print(f"[第{turn+1}轮] 模型结束：{decision['text']}")
            return decision["text"]
    return "(达到最大轮次仍未结束)"

# 真跑一次
if __name__ == "__main__":
    answer = run_agent("帮我读 data.txt 并算 15% 提成")
    print(f"\n最终答案：{answer}")

&emsp;&emsp;这段代码运行后，你会看到三行轮次日志加一行最终答案——它确实是一个能工作的 Agent。它有对话历史、有工具调度、有结束判断，逻辑闭环完整。如果你只是想做个能跑的演示原型，到这里就够了。但请注意循环里那个最关键的判断：`if decision["stop_reason"] == "tool_use"`。我们完全信任了模型返回的 `stop_reason` 字段——模型说要调工具，我们就调；模型说结束，我们就结束。这个「完全信任」，正是朴素 Agent 和工业级 Agent 第一个分水岭，我们在第四章会看到真实源码里那行赫然写着 `stop_reason === 'tool_use' is unreliable` 的注释。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143719709.png" width=50%></div>

### 2.2 把它放进生产环境：四个致命问题

&emsp;&emsp;现在闭上眼睛想一个画面——你把上面这个 Agent 真的部署上线了，让它在你的生产机器上替你干活，连续跑三十分钟、处理几十个任务。就在这一刻，那三十行代码里所有你没注意的隐患，会同时引爆。你越是想象得具体，下面这四个问题就越扎心。我们把这些隐患归纳成四个问题，它们会贯穿后面每一章。

> **【关于「四个问题」这个说法】**：这四个问题是本系列课程为了串起知识点而做的教学归纳，不是 `Claude Code` 源码内部的官方分类，源码里并没有一个叫「四问题」的模块。之所以这样归纳，是因为后面每一块工业级代码，几乎都能精确对应到其中某一个问题的解法。

&emsp;&emsp;第一个问题，**上下文膨胀**。我们的 `history` 列表只增不减，每调一次工具就往里塞两条消息。任务跑得越久，发给模型的历史越长，token 消耗越大，最终要么撑爆模型的上下文窗口，要么成本飞涨到不可接受。三十行版本对此毫无办法。

&emsp;&emsp;第二个问题，**失忆**。`run_agent` 一返回，`history` 这个局部变量就被回收了。下一次调用是全新的空历史——它完全不记得上次跟你聊过什么、学到过什么。一个没有持久记忆的 Agent，每次都从零开始，无法积累。

&emsp;&emsp;第三个问题，**无约束执行**。看 `tool_calc` 里那行 `eval(args['expr'])`——模型让算什么就算什么。如果工具表里有一个 `run_bash`，模型（或者一个被注入的恶意 prompt）让它执行 `rm -rf`，这个朴素 Agent 会毫不犹豫地照做。它没有任何一道关卡问一句「这条命令安全吗」。

&emsp;&emsp;第四个问题，**成本失控**。一个 Agent 串行做完所有事，复杂任务的耗时和 token 成本随步骤线性甚至超线性增长。当你想用多个 Agent 协作来加速时，朴素结构会让成本成倍叠加，因为每个 Agent 都在重复携带庞大的上下文。

&emsp;&emsp;这四个问题列在一起，就是后面每一章的对照坐标。这一节我们要解决的是第三个——无约束执行，因为它最危险，一个能删你文件的 Agent 没资格谈别的。另外三个（上下文膨胀、失忆、成本失控）是第 3 节《多智能体与上下文工程》的主题。下面这张表把四个问题和它们的归宿讲清楚，你可以对照着看每个问题最终在哪一节解掉。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>朴素 Agent 的四个致命问题与解法归属</font></p>
<div class="center">

| 问题 | 在三十行里的表现 | 工业级怎么解 | 在哪节课讲 |
|------|------------------|--------------|------------|
| #1 上下文膨胀 | `history` 只增不减，token 撑爆 | 四层压缩 + 五步预处理 + Prompt Cache | 第 3 节 |
| #2 失忆 | 函数返回历史即销毁 | 单文件 session memory（10 节模板） | 第 3 节 |
| #3 无约束执行 | `eval` 直接执行，无任何关卡 | 四层安全管线 + 五层权限 + 沙箱 + 断路器 | **本节第九章** |
| #4 成本失控 | 单 Agent 串行，成本线性膨胀 | 子 Agent 隔离 + Fork 缓存 + Coordinator | 第 3 节 |

</div>

&emsp;&emsp;明确了主线，我们接下来要回答的就是：工业级 Agent 是怎么解决这些问题的？它的整体结构长什么样？带着这四个问题，我们进入下一章——用一张五层架构地图，把 51 万行的全局先看清楚。

---

## <center>第三章：1.6% vs 98.4% + 五层架构地图</center>

&emsp;&emsp;上一章我们用三十行代码和四个问题，建立了「朴素 Agent 缺什么」的认知。这一章我们换一个尺度，从微观跳到宏观：先用一个量化的核心论点，回答「那 51 万行到底偏向能力还是约束」这个开篇悬念；再用一张五层架构地图，把整个工业级 Agent 的结构装进你脑子里。这张地图会贯穿后面所有章节——每讲一块，我们都会回到地图上指出「我们现在在哪一层」。

### 3.1 核心论点：约 1.6% 在决策，约 98.4% 在支撑与约束

&emsp;&emsp;先说结论，再说证据，再说这个证据该怎么看待。社区有人对这份泄露快照做过代码归类估算（这个估算后来被一篇学术论文收录引述），结论是：真正「调用模型做决策」的逻辑大约只占 1.6%，剩下约 98.4% 是确定性的运营基础设施——权限控制、上下文管理、工具路由、错误恢复这些。换句话说，一个工业级 Agent 里，「让 AI 思考」的代码是极少数，「兜住 AI、约束 AI、伺候 AI」的代码才是绝对主体。这个论点，恰好替你回答了第一章那个悬念：这 51 万行，绝大多数在约束和支撑 AI，而不是放大它——如果你刚才心里还嘀咕「不会都是冗余吧」，现在可以放下这个怀疑了。

&emsp;&emsp;关于这个 1.6% / 98.4% 的数字，有三件事要交代清楚，它们决定了你该怎么准确地理解和使用这个比例。

&emsp;&emsp;第一，**来源是学术社区分析，不是 Anthropic 官方数据**。这组数字出自 arXiv 上一篇编号 2604.14228 的论文（VILA-Lab，《Dive into Claude Code》，2026 年 4 月，https://arxiv.org/abs/2604.14228），这篇论文收录并引述了社区对泄露快照 `v2.1.88` 的代码归类估算——也就是说，1.6% 这个数字本身来自社区分析，论文只是把它收录引用，不是论文原创统计。Anthropic 官方从未发布过这个比例。所以我们引用它时，态度是「一个有依据的第三方分析」，而不是「官方权威结论」。

&emsp;&emsp;第二，**这个比例的口径是「AI 决策逻辑 vs 确定性运营基础设施」，不是「扩展能力 vs 约束行为」的精确切割**。准确地说：1.6% 指向「调模型做判断」的那部分代码，98.4% 指向「不需要调模型、确定性执行」的基础设施。约束行为只是这 98.4% 里的一部分，工具路由、上下文管理也都在里面——记住这个口径，你才不会把它和「能力 vs 约束」的二分混为一谈。

&emsp;&emsp;第三，**论点成立，精确数字不必当圣经**。「绝大多数代码在约束和支撑 AI，而非放大 AI」这个判断，是站得住的、可肯定地讲的；但你不该在任何场合把「1.6%」当成铁打的官方百分比斩钉截铁地报出去。你要带走的是这个判断本身，不是这个小数点。

> **【常见误区】**：<font color=red>把「1.6% / 98.4%」当成 Anthropic 官方发布的精确数据来用</font>，或把它等同于「扩展能力 vs 约束行为」的二分。后果是：你在自己的分享或文档里斩钉截铁报这个数，一旦有人追问出处或口径就答不上来。正确做法：<font color=red>永远带着「社区分析、经学术论文引述、口径是 AI 决策 vs 运营基建」这三个限定来讲</font>，重点放在「绝大多数代码在约束和支撑 AI」这个站得住的判断上。排查方法：凡讲到这个比例的地方，检查上下文有没有这三个限定词。

### 3.2 五层架构地图：把 51 万行装进一张图

&emsp;&emsp;论点立住了，我们需要一个结构来承载它。为了把这个工业级 Agent 讲清楚，我们用五层来描述它——从信息进来到结果出去，每一层负责一件事。需要说明的是，**「五层架构、L1 到 L5」是本系列课程为了教学而做的归纳，不是 `Claude Code` 源码里的官方分层命名**，源码里没有一个叫「Layer 3」的目录。我们用分层，是因为它能让你徒手把这个系统画出来——这是你今天要带走的五件产物里的第一件，请边听边在纸上跟着画。

&emsp;&emsp;你把这五层从下往上记住，地图就成型了：**L1 信息收集**，负责把用户输入、对话上下文、记忆汇聚起来；**L2 推理决策**，是那约 1.6% 的核心，调用 Claude 模型做判断；**L3 工具执行**，把模型的决策落地成对四十多个工具的统一调度；**L4 扩展能力**，是 Skill / MCP / Hook 三个正交的扩展口，让你能往系统里加新能力；**L5 安全约束**，一道四层管线，决定哪些动作能真正执行。这一节课，你会重点拿下 L3、L4、L5——也就是「能力的地基」「能力的放大」和「能力的反面」。L1、L2 涉及的上下文与记忆，留给第 3 节深讲。

&emsp;&emsp;这里补一句方法论回扣：开头那段【吃透任意开源项目·通用提示词】的第 1 阶段「规模与分层」，作用在 `Claude Code` 上产出的就是这张五层地图。换成任何开源项目，你都能用同一段提示词复跑出它自己的分层图——这张图不是我们硬塞给你的结论，是那套方法的必然产物。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143719751.png" width=50%></div>

### 3.3 工业版 vs 朴素版：规模对照

&emsp;&emsp;有了地图，你可以用一组实测数字感受一下「工业级」三个字到底有多重。下面这张表对比朴素三十行版和这份快照的真实规模，每一个数字都来自我对快照的 Bash 实测，命令附在表后，你可以逐条复核。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>朴素 Agent vs 工业级 Claude Code 规模对照（快照 v2.1.88 实测）</font></p>
<div class="center">

| 维度 | 朴素三十行版 | 工业级 Claude Code | 实测来源 |
|------|--------------|--------------------|----------|
| 总代码量 | ~30 行 | 512,664 行 / 1884 文件 | 快照统计 |
| src/ 顶层结构 | 1 个文件 | 35 个目录 + 18 个文件 | `ls -d src/*/` / `ls -p src/ \| grep -v /` |
| 每轮循环逻辑 | `run_agent` 约 15 行 | `query.ts` 1729 行 | `wc -l src/query.ts` |
| 工具协议 | `TOOLS` 一个 dict | `Tool.ts` 792 行统一契约 | `wc -l src/Tool.ts` |
| 安全检查 | 0 行 | `bashSecurity.ts` 2592 行 | `wc -l src/tools/BashTool/bashSecurity.ts` |

</div>

&emsp;&emsp;这里点一个具体数字让你有抓手：`src/` 顶层是 **35 个目录加 18 个文件**。目录数我用 `ls -d src/*/ | wc -l` 数出是 35，文件数用 `ls -p src/ | grep -v / | wc -l` 数出是 18，你拿到快照可以一字不差地复核。这门课的每个数字都这样落到一条可执行命令上，你不必信口头转述。

&emsp;&emsp;从这张表你应该能直观感受到那个量级差：你刚写的那个 15 行主循环，在工业版对应的是一个 1729 行的 `query.ts`；那个一行的 `TOOLS` dict，对应的是 792 行的 `Tool.ts` 统一契约。这两个文件，正是「能力腿」的两块地基。接下来两章，我们就分别把它们拆开——先看 QueryLoop（每轮对话到底发生了什么），再看 Tool 协议（四十多个工具凭什么能被同一个循环调度）。

---

## <center>第四章：QueryLoop 深拆——每轮对话到底发生了什么</center>

&emsp;&emsp;上一章我们站在五层架构地图的高处俯瞰了全局。从这一章开始，我们落到地面，一层一层挖。你要挖的第一块，是地图的 L2 到 L3 之间那条主动脉——每一轮对话，从用户说话到模型决策到工具执行，内部到底是怎么流转的。在朴素版里，这就是 `run_agent` 那个十几行的 `for` 循环；在工业版里，它是一个 1729 行的文件 `query.ts`。这一千七百行多出来的东西，正是你从「能跑」走到「敢用」要补的课。这一章你要重点拿下三件事：这个循环靠什么结构驱动、那行 `stop_reason 不可信` 的注释到底在防什么、以及「推理引擎代码群」这个概念该怎么理解。

> 📌 **【本章动手 · 摸底 → 产 MVP → 讲透（复制即用）】**

&emsp;&emsp;把下面整段，连同你本地 `v2.1.88` 快照里这些锚点处的真实源码片段一起发给 AI，先和它把 QueryLoop 的底层运行逻辑对话摸透，再让它直接产出能跑的最小 MVP；产出的 MVP 与本章的官方 MVP 并排跑即可自测。整段可直接复制：

```text
【吃透「<目标 Agent 项目>·<核心循环模块名>」· 摸底 → 产 MVP → 讲透】把本段连同你贴的真实源码片段一起处理。

角色：资深源码导师 + 结对程序员。我在吃透 <项目名> 的 <模块名>
（业界常见叫法：query loop / agent loop / run loop / step loop / react loop）。
- 模块职责（请帮我校准）：每轮对话从用户输入到工具执行的执行体，驱动整个 Agent 推进
- 驱动结构猜测（你帮我证实/证伪）：□ 同步 while  □ async for/await  □ 生成器 yield  □ 异步生成器  □ 状态机 transition  □ Actor/队列消费  □ pipeline 串流

我已核实的真实锚点（仅供定位，不许据此推断/臆造其它行号）：
  <文件>:<行号>  入口函数 / 主循环体
  <文件>:<行号>  模型响应解析（声明级 vs 结构级校验位）
  <文件>:<行号>  工具调用裁决
  <文件>:<行号>  退出 reason 各类分支（典型应覆盖 3-5 类：正常完成 / 模型错误 / 上下文溢出 / turn 上限 / Hook 阻断…）
  <文件>:<行号>  自停阈值（token 预算 / 收益递减 / 超时）

铁律：
- 只基于我贴的源码推理；涉及我没贴的部分，明说「需要看 X 文件」，绝不用「通常/据我所知」编造
- 每条结论贴色标：[源码 file:line] / [行为] / [文档] / [推断] / [待核验]；[待核验] 的不许当事实继续往下推

第一步·摸底（你问我答，逐轮收紧，直到我说「懂了」再进下一步）：
  1) 用一个类比说清这个模块的驱动结构（循环 / 状态机 / 异步生成器 / 管道 / Actor / 事件驱动…），并指认它落在我给的哪几行
  2) 挑出它最反直觉的 1–2 个设计点，逐个回答「不这么设计会怎样」
  3) 列关键不变量与边界：什么输入/状态会触发 退出 / 失败 / 降级 / 重试
  4) 回答尖锐问题（逐条给依据）：①循环靠什么结构驱动？（while / async for / generator yield / 状态机 transition / 队列消费…）②模型/LLM 返回信号可信度边界——哪些字段是"声明级"（模型自报），哪些是"结构级"（响应内容可校验）？工业版裁决依据是哪一种？为什么 ③退出路径穷举了哪几类？各自触发什么后续（清理 / 持久化 / 重试 / 报错 / 静默 / 用户提示）？是否复用同一条退出通道 ④是否有「自停」机制（turn 上限 / token 预算 / 收益递减 / 超时 / 错误连击）？阈值与计数器分别从哪几行读取 ⑤异常路径（模型抛错 / 工具失败 / 上下文溢出 / 网络中断）如何与正常退出区分

第二步·产 MVP（我说「出码」后才做）：
  - 纯 Python 标准库 + mock 掉一切外部依赖（LLM 调用 / 工具执行 / 文件 IO / 网络），写一个 ≤60 行能直接跑的最小原型，复刻该模块的「可迁移内核」——不是逐行翻译源码、更不是把原项目代码抠出来剪依赖
  - 末尾加 self-assert，至少覆盖 正常路径完成 + 一条边界/异常路径（如 turn 上限触发 / 工具失败 / 模型抛错 / 预算耗尽）；断言必须 print 出可见状态，禁止只靠 assert 静默通过
  - 关键行加注释，标「# 对应 <文件>:<行号>」
  - 交付前自检：这段在 .py / Jupyter cell / exec 三种上下文都能跑吗？禁用 inspect.getsource 之类依赖源文件的自省
  - 我没确认过的机制不许擅自加；你想按对原项目的印象加什么，先反问我

第三步·讲透：挑 MVP 里最核心的 5–8 行，逐行说「它在还原源码的哪个机制」，再点明「生产环境还要补什么（真实 LLM / 工具沙箱 / 并发 / 持久化 / 错误恢复 / 上下文压缩 / 流式返回…）、本 demo 故意省了什么」。
```

### 4.1 query.ts 的角色：每轮对话的状态机

&emsp;&emsp;先帮你把这个文件的身份定准。我用 `wc -l src/query.ts` 数过，它精确是 **1729 行**，你也可以自己数。它的角色，是「每一轮对话的执行体」——用户输入进来后，组织上下文、调模型、解析模型想干什么、执行工具、把结果交回去等下一轮，这一整套流程，都在这个文件里。

&emsp;&emsp;它的驱动结构是一个 `async function*`，也就是异步生成器（AsyncGenerator）。你可以这样类比：普通函数是「调用一次、返回一次、结束」，生成器是「跑到一半停下来，把当前结果交出去（`yield`），等外部叫它继续（`next()`），它再从停的地方往下跑」。Agent 的每轮循环天然就是这个形态——执行一个工具、把结果 `yield` 出去、暂停，等待下一轮再被唤醒继续。下面这段是 `query.ts` 里两个关键函数签名的真实位置，我们用 `grep` 把它定位出来看结构，而不是凭印象描述。

&emsp;&emsp;接下来这段代码不是要你跑它（它是 TypeScript 源码节选，我们只读结构），而是让你亲眼看到这个状态机的真实骨架。运行下面这个 Python 脚本，它会去你本地的快照里把这几行 `grep` 出来——如果你没有这份快照，看注释里贴出的实测结果即可，关键是理解 `async function*` 这个驱动结构真实存在。

In [8]:
# 静态验证：确认 query.ts 的 AsyncGenerator 状态机结构真实存在
# 这段 Python 是"验证脚本"，不是 Agent 逻辑——它去 grep 真实源码
import subprocess, os

# 快照源码路径（你本地 git clone 后的路径，按需修改）
SRC = "/Users/mac/Git/Claude Code/src/query.ts"

def grep_lines(path, pattern):
    """在指定文件里 grep 出含 pattern 的行，返回 (行号, 内容) 列表。"""
    if not os.path.exists(path):
        return []  # 没有快照时返回空，看下方注释里的实测结果即可
    # -n 带行号，-E 扩展正则，匹配生成器声明与不可信注释
    out = subprocess.run(["grep", "-nE", pattern, path],
                         capture_output=True, text=True).stdout
    return [line for line in out.splitlines() if line]

if os.path.exists(SRC):
    # 找异步生成器声明：query 和 queryLoop 两个核心入口
    for ln in grep_lines(SRC, r"async function\* (query|queryLoop)"):
        print("生成器结构:", ln)
    # 找那行关键的"stop_reason 不可信"注释
    for ln in grep_lines(SRC, r"stop_reason.*unreliable"):
        print("不可信注释:", ln)
else:
    # 没有快照时的实测结果（来自对 v2.1.88 的真实 grep）：
    print("生成器结构: 219:export async function* query(")
    print("生成器结构: 241:async function* queryLoop(")
    print("不可信注释: 554:    // Note: stop_reason === 'tool_use' is "
          "unreliable -- it's not always set correctly.")

生成器结构: 219:export async function* query(
生成器结构: 241:async function* queryLoop(
不可信注释: 554:    // Note: stop_reason === 'tool_use' is unreliable -- it's not always set correctly.


&emsp;&emsp;这段验证脚本运行后，你会看到三行输出，对应快照里的真实位置：`query.ts:219` 的 `export async function* query(`、`query.ts:241` 的 `async function* queryLoop(`、以及 `query.ts:554` 那行 `stop_reason === 'tool_use' is unreliable` 注释。前两行证实了「异步生成器状态机」不是一个比喻，而是源码里实打实的结构；第三行，是这一章最值得你记住的一行注释，我们下一节专门拆它。

&emsp;&emsp;为了让这个「状态机」在你脑子里转起来，下面这张图把 QueryLoop 一轮的状态流转画出来——它和第二章那张朴素循环图是同一套骨架，区别在于每个节点都厚了一层工程兜底。你可以对照着看：朴素版那个直接信 `stop_reason` 的菱形，在这里变成了「结构校验」这一步。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143719731.png" width=50%></div>

### 4.2 stop_reason === 'tool_use' 不可信：朴素版埋的雷

&emsp;&emsp;回到第二章。你写的那个朴素 Agent 里有一句 `if decision["stop_reason"] == "tool_use"`，完全信任模型告诉你的「我要调工具」。现在你看真实源码怎么对待这个字段——`query.ts:554` 那行注释，原文是 `// Note: stop_reason === 'tool_use' is unreliable -- it's not always set correctly.`，直译就是：<font color=red>**`stop_reason` 等于 `tool_use` 这件事是不可信的，它不总是被正确设置**</font>。

&emsp;&emsp;这行注释背后是一个深刻的工程教训。`stop_reason` 是模型在流式输出时设置的状态字段，但在流式场景下，它可能没被正确赋值——模型实际上输出了一个工具调用块，但 `stop_reason` 却没标成 `tool_use`；或者反过来。如果你像朴素版那样无脑信任这个字段，就会出现两种崩法：该调工具时没调（任务卡住），或者把模型幻觉出来的「假工具调用」当真去执行（不可预测的副作用）。工业版的做法是不信这个字段，转而去<font color=red>**结构化地检查模型返回的内容里到底有没有真实的 `tool_use` 块**</font>——你在前面 grep 时也能看到 `query.ts:130` 那行 `content => content.type === 'tool_use'`，它是在响应内容里逐块过滤，而不是看那个不靠谱的 `stop_reason`。

> **【踩坑预警】**：<font color=red>把 `stop_reason` 当作「模型是否要调工具」的唯一判据</font>。后果是：在真实流式 API 下偶发性地漏掉工具调用或执行幻觉调用，且因为是偶发，极难复现和定位。正确做法：永远以「响应内容里是否真实存在 `tool_use` 结构块」为准，把 `stop_reason` 仅当作辅助信号。排查方法：如果你的 Agent 偶尔「该调工具时发呆」，第一个怀疑对象就是这里。

&emsp;&emsp;这一个细节，就值大几百行代码——它是「确定性运营基础设施」最典型的样本：模型本身没变聪明，但工程层用结构校验兜住了模型输出的不确定性。这正是第三章那个 98.4% 的真实含义。

### 4.3 推理引擎代码群：「四万六千行」到底指什么

&emsp;&emsp;先把这一块的「行数口径」讲清楚，否则很容易对系统复杂度产生误判。

&emsp;&emsp;按单文件看：`QueryEngine.ts` 我用 `wc -l` 数过是 **1295 行**，负责每轮循环的 `query.ts` 是 1729 行。而「四万六千行」是 bundled（打包后）口径——把推理相关的一大堆模块连同依赖打包到一起的体积，不对应任何单个源文件。所以这门课用「**推理引擎相关代码群**」来精确指代这一块：它是 `QueryEngine.ts`（会话级编排，相当于单例调度者）、`query.ts`（每轮循环执行体）以及 `query/` 目录下辅助模块这一组文件的合称。「推理引擎代码群」这个叫法是我们的教学抽象，不是源码里的官方模块名。

> **【常见误区】**：<font color=red>把「四万六千行」当成某个单文件的规模</font>，或把 `QueryEngine` 想象成一个巨型上帝类。后果是你对这个系统的复杂度和可维护性产生错误估计，照着这个错误印象去设计自己的系统会跑偏。正确做法：区分「单文件行数」（`QueryEngine.ts` 是 1295 行）和「bundled 口径」（约四万六千行是打包体积）。排查方法：任何「几万行」级别的说法，先问一句「这是单文件还是 bundled」。

### 4.4 退出不是单条件，是多出路状态机

&emsp;&emsp;朴素版的 `run_agent` 只判断一个 `stop_reason`——`"end_turn"` 就停、其他就继续。工业版循环的退出逻辑截然不同：源码里存在多个 distinct 的退出 reason，每一个对应完全不同的停止语义。你可以自己跑一条命令验证：`grep "return { reason:" query.ts`，会数出比你预期多得多的退出点。本课选讲五个最具代表性的 reason，说明「穷举出路」是工业级循环和朴素版的真正分水岭——朴素版只有一个出口，工业版每个退出点都有明确语义、对应不同的后续处理。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>QueryLoop 代表性退出 reason 及其语义</font></p>
<div class="center">

| reason | 触发场景 | 朴素版有无对应处理 |
|--------|----------|--------------------|
| `completed` | 模型正常完成任务（`query.ts:1264/1357`） | 有（唯一出口 `end_turn`） |
| `max_turns` | 达到最大轮次限制（`query.ts:1711`） | 有（`range(max_turns)` 耗尽） |
| `model_error` | 模型调用返回错误（`query.ts:996`） | 无（异常会直接崩）  |
| `prompt_too_long` | 上下文超出模型窗口（`query.ts:1175/1182`） | 无（会静默失败或崩）|
| `stop_hook_prevented` | PreToolUse Hook 阻断了工具执行（`query.ts:1279`） | 无（没有 Hook 系统） |

</div>

&emsp;&emsp;表里右列「朴素版有无对应处理」的空格说明了一切：前两个朴素版能顾到，后三个朴素版完全没有退出语义——出错就崩、超窗口就静默失败、Hook 根本不存在。工业版的每个 reason 都有对应的后续处理逻辑：`model_error` 会触发分阶段回退，`prompt_too_long` 会触发上下文压缩，`stop_hook_prevented` 会把 Hook 的 stderr 喂回模型让它换方案。迁移契约一句话：**循环退出要穷举出路，不能只判一个 stop**。

&emsp;&emsp;其中错误类退出（如 `model_error` / `prompt_too_long`）背后还有一套分阶段回退与防重入机制，信息密度高，留到第 3 节《多智能体与上下文工程》专拆。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143727338.png" width=50%></div>

### 4.5 Token 预算回报递减自停

&emsp;&emsp;还有一个防 Agent「假装忙」的机制值得单独讲——不靠模型自觉，靠确定性计数器在收益递减时主动停。这个机制在 `query/tokenBudget.ts` 里，核心逻辑非常精炼。

&emsp;&emsp;先说两个可以带 file:line 的精确值：`COMPLETION_THRESHOLD = 0.9`（`tokenBudget.ts:3`），是「预算用掉 90% 时认为任务接近完成」的门槛；`DIMINISHING_THRESHOLD = 500`（`tokenBudget.ts:4`），是「每轮新增 token 低于 500 时认为回报递减」的阈值。触发逻辑在 `tokenBudget.ts:57-61`：当 `continuationCount >= 3`（已经续了至少 3 轮）且本轮新增 token 和上轮新增 token 都低于 500 时，`isDiminishing` 标志位置为 `true`，循环主动停机而不是继续耗费预算。

&emsp;&emsp;这个设计的直觉是：如果一个 Agent 已经用了大量 token 但每轮产出越来越少，大概率它在「原地踏步」而不是在真正推进任务——与其让它无限续下去，不如用确定性计数器喊停。迁移契约一句话：**怎么防 Agent 假装忙——预算回报递减就确定性停机**。

&emsp;&emsp;到这里，能力腿的第一块地基——「每轮对话怎么流转」——你就拆完了。现在你应该能脱口回答：循环靠异步生成器驱动、`stop_reason` 不能信要做结构校验、推理引擎是一组文件而非单个巨型文件。但还有一个问题没解决：这个循环里被调度的那四十多个工具，长得千奇百怪——读文件、跑命令、搜网页、改代码——它们凭什么能被同一个循环用同一套逻辑无差别地调起来？这就是下一章 Tool 协议要回答的。

In [7]:
"""
QueryLoop MVP — LangChain 1.x + DeepSeek + .env 环境变量
=========================================================

复刻 query.ts:241 async function* queryLoop 的内核：
  - 异步生成器驱动的 Agent 工具调用循环
  - 不依赖 stop_reason，只看 AIMessage.tool_calls (query.ts:130/554)
  - 5 类退出条件收敛到 AgentMiddleware before_model 钩子

依赖：pip install langchain python-dotenv openai

用法：
  python query_loop_mvp.py

.env 文件（放在当前目录或 ~/.claude/.env）：
  DEEPSEEK_API_KEY=sk-...
  DEEPSEEK_MODEL=deepseek-chat          # 可选
  DEEPSEEK_BASE_URL=https://api.deepseek.com/v1  # 可选
  AGENT_MAX_TURNS=5                      # 可选
"""

import os
from collections.abc import Callable
from typing import Any

# ─── 环境变量加载 ───
from dotenv import load_dotenv

# 优先从 ~/.claude/.env 加载，再当前目录 .env 覆盖
load_dotenv(os.path.expanduser("~/.claude/.env"))
load_dotenv(override=True)

# ─── LangChain 1.x 核心 ───
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    ModelRequest,
    ModelResponse,
)
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage
from langchain.tools import tool
from langgraph.runtime import Runtime


# ═══════════════════════════════════════════════
# 配置：从环境变量读取
# ═══════════════════════════════════════════════

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise RuntimeError(
        "DEEPSEEK_API_KEY is not set. "
        "Set it in ~/.claude/.env or via: export DEEPSEEK_API_KEY=sk-..."
    )

DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com/v1")
AGENT_MAX_TURNS = int(os.getenv("AGENT_MAX_TURNS", "5"))
AGENT_MAX_PROMPT_CHARS = int(os.getenv("AGENT_MAX_PROMPT_CHARS", "10000"))
TEMPERATURE = float(os.getenv("AGENT_TEMPERATURE", "0.7"))
MAX_TOKENS = int(os.getenv("AGENT_MAX_TOKENS", "4096"))
TIMEOUT = int(os.getenv("AGENT_TIMEOUT", "120"))
MAX_RETRIES = int(os.getenv("AGENT_MAX_RETRIES", "6"))

print(f"[init] Model: {DEEPSEEK_MODEL}  @ {DEEPSEEK_BASE_URL}")
print(f"[init] max_turns={AGENT_MAX_TURNS}  max_chars={AGENT_MAX_PROMPT_CHARS}")
print(f"[init] API key: {DEEPSEEK_API_KEY[:8]}...{DEEPSEEK_API_KEY[-4:]}")


# ═══════════════════════════════════════════════
# DeepSeek 模型初始化
# 对应 query/deps.ts:35 productionDeps().callModel
# ═══════════════════════════════════════════════

model = init_chat_model(
    model=DEEPSEEK_MODEL,
    model_provider="openai",          # DeepSeek 是 OpenAI 兼容的
    base_url=DEEPSEEK_BASE_URL,       # https://api.deepseek.com/v1
    api_key=DEEPSEEK_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    timeout=TIMEOUT,
    max_retries=MAX_RETRIES,
)

print("[init] Model instance created successfully")


# ═══════════════════════════════════════════════
# 工具定义
# 对应 tools.ts 中各 Tool 实现
# ═══════════════════════════════════════════════

@tool
def search(query: str) -> str:
    """Search the web for information on a given topic.

    Args:
        query: The search query string to look up.
    """
    return f'[Search result for "{query}"] Found relevant articles about this topic.'


@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression using Python.

    Args:
        expression: A mathematical expression as a string, e.g. "2 + 3 * 4".
    """
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error evaluating expression: {e}"


tools = [search, calculate]
print(f"[init] Tools registered: {[t.name for t in tools]}")


# ═══════════════════════════════════════════════
# QueryLoop 退出条件 → AgentMiddleware
#
# 对应 query.ts 中 5 类退出：
#   model_error       → :996   wrap_model_call 捕获
#   prompt_too_long   → :1175  before_model 检查 msg_chars
#   max_turns         → :1711  before_model 计数 + jump_to end
#   completed         → :1264  AIMessage 无 tool_calls → __end__
#   stop_hook_prevented → :1279 before_model 检查 flag
# ═══════════════════════════════════════════════

class QueryLoopMiddleware(AgentMiddleware):
    """汇聚 queryLoop 所有退出条件到 before_model / wrap_model_call 钩子。"""

    def __init__(self, max_turns: int = 5, max_chars: int = 10_000) -> None:
        super().__init__()
        self.max_turns = max_turns
        self.max_chars = max_chars
        self.turn_count = 0
        self.hook_prevented = False  # 外部可设置，对应 query.ts:1279

    # before_model：每次进入 model 节点前调用
    # 对应 query.ts:1175 / :1279 / :1711
    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        self.turn_count += 1

        # stop_hook 阻止 —— 对应 query.ts:1279 stop_hook_prevented
        if self.hook_prevented:
            print(f"  [mw] turn={self.turn_count} → STOP: hook_prevented")
            return {
                "messages": [AIMessage(content="[STOP] Hook prevented continuation.")],
                "jump_to": "end",
            }

        # max_turns —— 对应 query.ts:1711 max_turns
        if self.turn_count > self.max_turns:
            print(f"  [mw] turn={self.turn_count} → STOP: max_turns")
            return {
                "messages": [
                    AIMessage(
                        content=f"[STOP] Reached maximum turns ({self.max_turns}). "
                        "Please summarize what you have so far."
                    )
                ],
                "jump_to": "end",
            }

        # prompt_too_long —— 对应 query.ts:1175
        total_chars = sum(
            len(str(m.content)) for m in state["messages"] if hasattr(m, "content")
        )
        if total_chars > self.max_chars:
            print(f"  [mw] turn={self.turn_count} → STOP: prompt_too_long ({total_chars}>{self.max_chars})")
            return {
                "messages": [
                    AIMessage(content="[STOP] Context too long. Run /compact first.")
                ],
                "jump_to": "end",
            }

        print(f"  [mw] turn={self.turn_count} continue (msgs={len(state['messages'])}, chars={total_chars})")
        return None  # None = 继续正常流程

    # wrap_model_call：捕获 model_error —— 对应 query.ts:996
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        try:
            return handler(request)
        except Exception as e:
            print(f"  [mw] model_error caught: {e}")
            return ModelResponse(
                messages=[AIMessage(content=f"[ERROR] Model error: {e}")]
            )


# ═══════════════════════════════════════════════
# 构建 Agent
# ═══════════════════════════════════════════════

middleware = QueryLoopMiddleware(
    max_turns=AGENT_MAX_TURNS,
    max_chars=AGENT_MAX_PROMPT_CHARS,
)

agent = create_agent(
    model=model,
    tools=tools,
    middleware=[middleware],
)

print("[init] Agent created with QueryLoopMiddleware\n")


# ═══════════════════════════════════════════════
# run_agent — 对应 query.ts:219 query() 的 for await 消费
# ═══════════════════════════════════════════════

def run_agent(user_input: str) -> dict[str, Any]:
    """发送用户输入，流式消费 Agent 事件，返回最终结果。

    使用 stream_mode="values" 获取每一步完整状态，
    对应 query.ts 中每轮 yield 出的 StreamEvent。
    """
    print(f"\n{'─'*60}")
    print(f"[USER] {user_input}")
    print(f"{'─'*60}")

    final_messages: list = []
    last_tool_calls: list[str] = []

    # agent.stream() — 对应 query.ts async function* query
    for chunk in agent.stream(
        {"messages": [HumanMessage(content=user_input)]},
        stream_mode="values",  # 每步完整状态
    ):
        msgs = chunk["messages"]
        last = msgs[-1]
        final_messages = msgs

        if hasattr(last, "content_blocks"):
            blocks = last.content_blocks
        elif hasattr(last, "content"):
            c = last.content if isinstance(last.content, str) else str(last.content)
            blocks = [{"type": "text", "text": c}]
        else:
            blocks = []

        for b in blocks:
            if b.get("type") == "tool_call":
                name = b["name"]
                args = b.get("args", {})
                last_tool_calls.append(f"{name}({args})")
                print(f"  🔧 TOOL: {name}({args})")
            elif b.get("type") == "text":
                text = b.get("text", "").strip()
                if text:
                    print(f"  💬 {text[:200]}{'...' if len(text) > 200 else ''}")

        # stream_mode="values" 下一个 chunk 已经是 tool_result 或 final response
        # AIMessage.tool_calls 判断继续 — 对应 query.ts:130 content.type === 'tool_use'
        if hasattr(last, "tool_calls") and last.tool_calls:
            continue

    last_msg = final_messages[-1] if final_messages else None
    last_content = str(last_msg.content) if last_msg and hasattr(last_msg, "content") else ""

    print(f"[DONE] turns={middleware.turn_count}  "
          f"response_len={len(last_content)}  "
          f"tool_calls={last_tool_calls}")
    return {
        "turns": middleware.turn_count,
        "last_message": last_content,
        "tool_calls": last_tool_calls,
    }


# ═══════════════════════════════════════════════
# Self-Assert 测试
# ═══════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("  QueryLoop MVP — Self-Assert Tests (DeepSeek)")
    print("=" * 60)

    all_passed = True

    # ─── Test 1：普通对话（无 tool 调用）───
    # 对应 query.ts:1264 completed
    print("\n>>> Test 1: completed (simple conversation, no tools)")
    middleware.turn_count = 0
    result = run_agent("你好，请用一句话介绍你自己。")
    t1_ok = result["turns"] >= 1 and len(result["last_message"]) > 0
    status = "✅ PASS" if t1_ok else "❌ FAIL"
    print(f"  {status}: turns={result['turns']}, response_len={len(result['last_message'])}")
    all_passed = all_passed and t1_ok

    # ─── Test 2：一轮工具调用后正常完成 ──
    # 对应 query.ts:1264 completed（tool_use → tool_result → end_turn）
    print("\n>>> Test 2: tool call → completed")
    middleware.turn_count = 0
    result = run_agent("用计算器算一下 (3 + 5) * 7 等于多少？")
    t2_ok = result["turns"] >= 2 and "56" in result["last_message"]
    status = "✅ PASS" if t2_ok else "❌ FAIL"
    print(f"  {status}: turns={result['turns']}, '56' in response={'56' in result['last_message']}")
    print(f"     tool_calls: {result['tool_calls']}")
    all_passed = all_passed and t2_ok

    # ─── Test 3：多工具调用 ──
    print("\n>>> Test 3: multi-tool calls")
    middleware.turn_count = 0
    result = run_agent("搜索 Python asyncio 的用法，然后计算 2 的 10 次方。")
    t3_ok = len(result["tool_calls"]) >= 2
    status = "✅ PASS" if t3_ok else "❌ FAIL"
    print(f"  {status}: turns={result['turns']}, tool_calls={result['tool_calls']}")
    print(f"     response preview: {result['last_message'][:300]}")
    all_passed = all_passed and t3_ok

    # ─── Test 4：max_turns 边界 ──
    # 对应 query.ts:1711 max_turns
    # 用 max_turns=1：turn=1 正常进入 → turn=2 被 before_model 拦截
    # 用 invoke 而非 stream，确保能捕获 middleware 注入的 [STOP] 消息
    print("\n>>> Test 4: max_turns boundary (force max_turns=1)")
    middleware.max_turns = 1
    middleware.turn_count = 0
    middleware.hook_prevented = False
    max_result = agent.invoke(
        {"messages": [HumanMessage(content="反复搜索五个不同的主题：AI, ML, DL, RL, CV")]}
    )
    all_text = " | ".join(
        str(m.content) for m in max_result["messages"] if hasattr(m, "content")
    )
    t4_ok = "[STOP] Reached maximum turns" in all_text
    status = "✅ PASS" if t4_ok else "❌ FAIL"
    print(f"  {status}: STOP message in all_msgs={'[STOP] Reached maximum turns' in all_text}")
    if not t4_ok:
        print(f"     all_msgs_text (first 300 chars): {all_text[:300]}")
    middleware.max_turns = AGENT_MAX_TURNS  # 恢复
    all_passed = all_passed and t4_ok

    # ─── Test 5：stop_hook 阻止 ──
    # 对应 query.ts:1279 stop_hook_prevented
    # 使用 invoke 而非 stream 来精确获取 middleware 注入的 STOP 消息
    # stream_mode="values" 中 before_model 注入的消息后可能被后续 chunk 追加文本
    print("\n>>> Test 5: stop_hook_prevented")
    middleware.turn_count = 0
    middleware.hook_prevented = True
    hook_result = agent.invoke(
        {"messages": [HumanMessage(content="这个请求应该被阻止。")]}
    )
    all_msgs_text = " | ".join(
        str(m.content) for m in hook_result["messages"] if hasattr(m, "content")
    )
    t5_ok = "[STOP] Hook prevented" in all_msgs_text
    middleware.hook_prevented = False
    status = "✅ PASS" if t5_ok else "❌ FAIL"
    print(f"  {status}: [STOP] Hook message in all_msgs={'[STOP] Hook' in all_msgs_text}")
    if not t5_ok:
        print(f"     all_msgs_text: {all_msgs_text[:300]}")
    all_passed = all_passed and t5_ok

    print("\n" + "=" * 60)
    if all_passed:
        print("  ALL 5 TESTS PASSED")
    else:
        print("  SOME TESTS FAILED — check output above")
    print("=" * 60)


[init] Model: deepseek-chat  @ https://api.deepseek.com/v1
[init] max_turns=5  max_chars=10000
[init] API key: sk-d513c...014d
[init] Model instance created successfully
[init] Tools registered: ['search', 'calculate']
[init] Agent created with QueryLoopMiddleware

  QueryLoop MVP — Self-Assert Tests (DeepSeek)

>>> Test 1: completed (simple conversation, no tools)

────────────────────────────────────────────────────────────
[USER] 你好，请用一句话介绍你自己。
────────────────────────────────────────────────────────────
  💬 你好，请用一句话介绍你自己。
  [mw] turn=1 continue (msgs=1, chars=14)
  💬 你好！我是 Claude，由 Anthropic 开发的 AI 助手，擅长对话、搜索、计算和编程等多种任务，致力于为你提供准确、有用的帮助！😊
[DONE] turns=1  response_len=71  tool_calls=[]
  ✅ PASS: turns=1, response_len=71

>>> Test 2: tool call → completed

────────────────────────────────────────────────────────────
[USER] 用计算器算一下 (3 + 5) * 7 等于多少？
────────────────────────────────────────────────────────────
  💬 用计算器算一下 (3 + 5) * 7 等于多少？
  [mw] turn=1 continue (msgs=1, chars=25)
  💬 好

---

## <center>第五章：Tool 协议——40+ 工具被同一循环无差别调度的地基</center>

&emsp;&emsp;上一章我们看清了 QueryLoop 这条主动脉怎么跳动。这一章我们要解决一个紧接着的问题：主动脉里流过的「工具」，种类极其繁杂——有的读文件、有的执行 shell、有的调用网络、有的修改代码，输入输出格式天差地别。在你第二章写的朴素版里只有两个玩具工具，用一个 dict 就糊弄过去了；但工业版有四十多个真实工具，它们凭什么能被第四章那个统一的循环无差别地调起来，循环代码却完全不需要为每个工具写特例？答案是一个统一的行为契约，定义在 `Tool.ts` 里。这一章你会拆开这块「地基」，并且看到它和你下一章要抄走的扩展三件套是什么关系。

> 📌 **【本章动手 · 摸底 → 产 MVP → 讲透（复制即用）】**

&emsp;&emsp;把下面整段连同你快照里这些锚点处的真实源码片段一起发给 AI，先摸透 Tool 统一契约「凭什么让 40+ 异构工具被同一循环调度」，再让它产出可跑 MVP；与本章官方 MVP 对照即可自测。整段可直接复制：

```text
【吃透「<目标 Agent 项目>·<统一行为契约模块名>」· 摸底 → 产 MVP → 讲透】把本段连同你贴的真实源码片段一起处理。

角色：资深源码导师 + 结对程序员。我在吃透 <项目名> 的 <模块名>
（业界常见叫法：tool contract / capability interface / plugin descriptor / function spec / handler signature）。
- 模块职责（请帮我校准）：让 N 种异构子组件被同一执行器无差别调度的契约地基

我已核实的真实锚点（仅供定位，不许据此推断/臆造其它行号）：
  <文件>:<行号>  契约接口/基类定义
  <文件>:<行号>  必填字段 / 可选字段
  <文件>:<行号>  参数依赖的方法字段（吃 input 的方法 vs 静态布尔）
  <文件>:<行号>  双层校验位（型校验 / 义校验）
  <文件>:<行号>  权限/前置 hook 调用入口
  <文件>:<行号>  最终裁决枚举（如 allow/deny/ask）
  <文件>:<行号>  结果大小/超时等运行时硬上限

铁律：
- 只基于贴出的源码与锚点；涉及没贴的部分明说「需要看 <路径>」，不许「通常/据我所知」编造
- 每条结论挂色标：[源码 file:line] / [行为] / [文档] / [推断] / [待核验]
- [待核验] 的不许当事实继续往下推

第一步·摸底（你问我答，逐轮收紧，直到我说「懂了」再进下一步）：
  1) 用一个类比说清这份契约的结构，并指认它落在我给的哪几行
  2) 挑出最反直觉的 1–2 个设计点（典型如：某些字段为何是"吃 input 的方法"而不是静态布尔），逐个回答"不这么设计会怎样"
  3) 列关键不变量与边界：哪种输入被第一层（型/schema）拦下、哪种被第二层（义/语义）拦下、哪些结果会落盘/截断
  4) 回答尖锐问题（逐条给依据）：①N 种异构组件凭什么能被同一执行器调度（共有字段最小集是什么）②哪些字段是参数依赖的方法而非静态属性？为什么必须吃 input ③权限/前置裁决链的入口和最终裁决枚举分别在哪 ④双层校验（型/义）的分工边界，哪些错误属于"第一层放行但第二层会拦"

第二步·产 MVP（我说「出码」后才做）：
  - 纯 Python 标准库 + mock 一切外部依赖，写一个 ≤60 行能直接跑的最小原型，复刻「统一契约 + 双层校验 + 参数依赖方法 + 权限链入口」的可迁移内核
  - self-assert 覆盖：合法输入通过 / 型校验拦下 / 义校验拦下 / 权限链拒绝 至少 3 条；断言必须 print 出可见状态，禁止只靠 assert 静默通过
  - 关键行加注释，标「# 对应 <文件>:<行号>」
  - 交付前自检：这段在 .py / Jupyter cell / exec 三种上下文都能跑吗？禁用 inspect.getsource 之类依赖源文件的自省
  - 我没确认过的字段不许擅自加；你想按对原项目的印象加什么，先反问我

第三步·讲透：挑 MVP 里最核心的 5–8 行，逐行说「它在还原源码的哪个机制」，再点明「生产环境还要补什么（schema 校验库 / 并发安全 / 超时 / 结果截断 / 遥测…）、本 demo 故意省了什么」。
```

### 5.1 Tool.ts：一份全局统一的行为契约

&emsp;&emsp;先定身份。`Tool.ts` 我用 `wc -l src/Tool.ts` 数过，精确是 **792 行**。它的核心，是定义了一个所有工具都必须实现的统一行为契约——在 TypeScript 里是一个泛型类型 `type Tool<IN, OUT>`。你可以把它类比成你熟悉的 Python `Protocol` 或抽象基类（ABC）：它不实现任何具体功能，只规定一件事——凡是想当一个工具的，必须长成这个样子、必须提供这些东西。你写过 ABC 就会秒懂这个设计。

&emsp;&emsp;这份契约里有什么？它不是只有可怜的几个字段，而是一组相当完整的行为声明——字段和方法粗看就有数十个（下面的验证脚本按「契约块内顶层成员」这一明确口径数到 47，换个口径会有出入，可自行复核），覆盖工具的方方面面：`name`（工具名）、`description`（给模型看的功能描述）、`inputSchema`（输入参数的结构定义）、`call`（真正执行的逻辑）、`isReadOnly`（这个工具是否只读、不改变系统状态）、`isConcurrencySafe`（能否和别的工具并发执行）等等。这里要强调一点：这是一份**数十个成员的统一行为契约**，覆盖面相当宽——正是这种字段密度，才让后面那个统一循环能在不认识具体工具身份的情况下做出智能调度。

&emsp;&emsp;为什么这些字段如此关键？因为它们让那个统一的循环可以做出智能调度决策，而完全不需要知道工具的具体身份。下面这张表挑出几个最能体现「契约价值」的字段，看它们分别在为循环解决什么问题。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Tool 统一契约关键字段及其对循环的价值</font></p>
<div class="center">

| 契约字段 | 含义 | 让循环能做什么（工具无关） |
|----------|------|----------------------------|
| `name` | 工具唯一标识 | 模型按名字点工具，循环按名字路由 |
| `description` | 给模型看的功能说明 | 模型据此决定调不调，无需循环硬编码 |
| `inputSchema` | 输入参数结构 | 循环统一做参数校验，工具不用各写一套 |
| `call` | 实际执行逻辑 | 循环只管 `call`，不管工具内部怎么实现 |
| `isReadOnly` | 是否只读 | 循环据此判断要不要走权限确认 |
| `isConcurrencySafe` | 是否可并发 | 循环据此决定能否并行调度多个工具 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143720072.png" width=50%></div>

&emsp;&emsp;表里有两个字段值得多说两句。`isConcurrencySafe` 并非静态布尔值——它实际上是一个接收 `input` 参数的方法，这意味着同一个工具在不同参数下并发安全性可以不同：读操作可以安全并发、带写入的操作则需要独占执行。循环正是靠这个方法的返回值来动态决定当前批次里哪些工具可以并行调度、哪些必须串行等待。另外，契约实际上是双层校验而非单层：`inputSchema` 做的是类型校验（保证参数结构符合预期），而 `validateInput` 方法在此之上做语义校验——类型对了不代表语义合法，两层都过才算真正合规。此外，超出 `maxResultSizeChars` 的大结果不会直接塞进上下文，而是落盘给出预览引用，防止单个工具输出撑爆上下文窗口（`Read` 工具的这个字段设为不限，是专门为防止「读文件→超长→读不完」的递归困境而设的例外）。

&emsp;&emsp;下面这段验证脚本和第四章 QueryLoop 那段同构：它直接 `grep` 真实源码，把 5.1 点名的每个关键字段和方法的真实行号打印出来，并计数契约的完整成员数量。如果你 `git clone` 了快照，运行后你会看到一行总行数 + 六行字段/方法定位输出 + 一行计数结果；如果没有快照，`else` 分支会输出 2026-05 实测的对应值，两者内容逐行一致，可直接核对。

In [ ]:
# 静态验证：确认 Tool.ts 统一行为契约的关键字段/方法真实存在
# 这段 Python 是"验证脚本"，不是 Agent 逻辑——它去 grep 真实源码
import subprocess, os

# 快照源码路径（你本地 git clone 后的路径，按需修改）
SRC = "/Users/mac/Git/Claude Code/src/Tool.ts"

def grep_lines(path, pattern):
    """在指定文件里 grep 出含 pattern 的行，返回字符串列表。
    
    Args:
        path: 源文件绝对路径
        pattern: 扩展正则表达式，用于匹配目标行
    Returns:
        命中行的字符串列表（含行号），没有快照时返回空列表
    """
    if not os.path.exists(path):
        return []
    # -n 带行号，-E 扩展正则，只取 grep 命中的行
    out = subprocess.run(["grep", "-nE", pattern, path],
                         capture_output=True, text=True).stdout
    return [line for line in out.splitlines() if line]

if os.path.exists(SRC):
    # 验证 Tool.ts 总行数
    wc = subprocess.run(["wc", "-l", SRC],
                        capture_output=True, text=True).stdout.split()[0]
    print(f"[总行数] Tool.ts: {wc} 行")

    # 逐一确认课件 5.1 点名的关键字段/方法的真实行号
    for ln in grep_lines(SRC, r"readonly inputSchema"):
        print("契约字段 inputSchema:", ln)
    for ln in grep_lines(SRC, r"isConcurrencySafe\(input"):
        print("契约方法 isConcurrencySafe:", ln)
    for ln in grep_lines(SRC, r"isReadOnly\(input"):
        print("契约方法 isReadOnly:", ln)
    # 只取字段定义行，排除注释行
    for ln in grep_lines(SRC, r"^\s+maxResultSizeChars:"):
        print("契约字段 maxResultSizeChars:", ln)
    for ln in grep_lines(SRC, r"validateInput\?\("):
        print("契约方法 validateInput?:", ln)
    for ln in grep_lines(SRC, r"Only called after validateInput"):
        print("权限注释:", ln)

    # 精确计数 Tool<> 定义体（L362-L695）内的顶层成员数
    # 排除泛型参数行（含 extends / = AnyObject 等），只数字段和方法声明
    count_cmd = (
        f"awk 'NR>=362 && NR<=695' '{SRC}' | "
        r"grep -E '^  [a-zA-Z_]' | "
        r"grep -vE '^   ' | "
        r"grep -vE '(extends|= AnyObject|= unknown|ToolProgressData)' | "
        "wc -l"
    )
    n = int(subprocess.run(count_cmd, shell=True,
                           capture_output=True, text=True).stdout.strip())
    print(f"\n[计数] Tool<> 顶层成员（字段+方法，L362-L695）: {n} 个")
else:
    # 没有快照时的实测结果（来自对 v2.1.88 的真实 grep，2026-05 核实）：
    print("[总行数] Tool.ts: 792 行")
    print("契约字段 inputSchema: 394:  readonly inputSchema: Input")
    print("契约方法 isConcurrencySafe: 402:  isConcurrencySafe(input: z.infer<Input>): boolean")
    print("契约方法 isReadOnly: 404:  isReadOnly(input: z.infer<Input>): boolean")
    print("契约字段 maxResultSizeChars: 466:  maxResultSizeChars: number")
    print("契约方法 validateInput?: 489:  validateInput?(")
    print("权限注释: 495:   * Determines if the user is asked for permission. Only called after validateInput() passes.")
    print("\n[计数] Tool<> 顶层成员（字段+方法，L362-L695）: 47 个")

&emsp;&emsp;这里要诚实说明：47 是「按上面这条命令、这个口径」数出来的——口径换一种（算不算可选成员、算不算嵌套方法），数出来会有出入。所以你要带走的不是「47」这个会浮动的数，而是「这是一份数十个成员、宽而深的契约」这个判断——这和第三章对待 1.6% 的态度一致：论点为准，精确数字不必当圣经。脚本里 `grep` 到的每个具体名字倒是钉死可复核的：身份识别有 `name`、`aliases`、`searchHint`，执行与校验有 `call`、`inputSchema`、`validateInput`、`checkPermissions`，调度控制有 `isReadOnly`、`isConcurrencySafe`，再加上 `maxResultSizeChars` 和 8 个 `render*` 渲染方法，覆盖了工具从识别、执行到 UI 渲染的完整生命周期。这是一份正经意义上宽而深的行为契约，而不是简单的几个 getter。

**Tool 契约 MVP：双层校验 + 动态并发分批骨架**

&emsp;&emsp;Tool 契约里有三个点在课件里分散讲过：`inputSchema` 声明的类型约束（MVP 用 `type_validate` 模拟对它的校验，`inputSchema` 本身是结构声明而非校验函数）、`validateInput` 语义校验（型对不等于义对）、`isConcurrencySafe(input)` 动态并发判断。下面这段 MVP 把三者整合成一个可运行的执行入口，再加上 `maxResultSizeChars` 超限落盘逻辑，正好覆盖 Tool 契约的四个核心字段。运行后你会看到：缺字段被类型校验拦下、语义违规被第二层拦下、大结果写临时文件并返回预览、并发分批把安全任务和不安全任务分到两组。

In [ ]:
# Tool 契约 MVP：双层校验（型+义）+ isConcurrencySafe 动态并发分批 + 结果落盘预览
# 以下模拟 Tool.ts 字段的行为：inputSchema(类型约束声明) / validateInput(语义校验) / isConcurrencySafe / maxResultSizeChars
import os, tempfile

MAX_RESULT_SIZE_CHARS = 1024   # 对应 Tool.ts maxResultSizeChars（演示阈值）
PREVIEW_SIZE = 200             # 超限时取前 N 字符做预览，防爆上下文窗口

def type_validate(input_data: dict, schema: dict):
    """
    第一层校验：类型校验（模拟 Tool.ts inputSchema 声明的类型约束）。
    Args:
        input_data: 工具入参
        schema: {字段名: 期望类型名} 映射
    Returns:
        str | None: 校验失败原因，通过返回 None
    """
    type_map = {"str": str, "int": int, "float": (int, float)}
    for field, expected in schema.items():
        if field not in input_data:
            return f"缺少必填字段: {field}"
        if expected in type_map and not isinstance(input_data[field], type_map[expected]):
            return f"字段 {field} 类型错误：期望 {expected}"
    return None

def semantic_validate(input_data: dict, tool_name: str):
    """
    第二层校验：语义校验（对应 Tool.ts validateInput）——型对 ≠ 义对。
    Args:
        input_data: 已通过类型校验的入参
        tool_name: 工具名，不同工具有不同语义约束
    Returns:
        str | None: 语义拒绝原因，通过返回 None
    """
    if tool_name == "read_file":
        # 语义约束：禁止访问 /etc 路径（类型校验看不到这种业务规则）
        if input_data.get("path", "").startswith("/etc"):
            return "语义拒绝：禁止访问 /etc 路径"
    elif tool_name == "calc":
        # 语义约束：表达式不得为空字符串
        if not input_data.get("expr", "").strip():
            return "语义拒绝：expr 不得为空字符串"
    return None

def is_concurrency_safe(tool_name: str, input_data: dict) -> bool:
    """
    动态并发安全判断（对应 Tool.ts isConcurrencySafe(input) 方法）。
    同一工具不同参数并发安全性可不同——安全集并行，不安全独占。
    Args:
        tool_name: 工具名
        input_data: 工具入参（决定当次调用是否并发安全）
    Returns:
        bool: True = 可与其他任务并行，False = 需独占串行
    """
    if tool_name in ("read_file", "calc"):
        return True   # 只读 / 纯计算：天然并发安全
    if tool_name == "write_file":
        # 写操作：写 shared 路径认为不安全（同一文件多写互斥）
        return "shared" not in input_data.get("path", "")
    return False  # 未知工具保守判断：不并发

def execute_tool(tool_name: str, input_data: dict) -> dict:
    """
    工具执行入口：双层校验 → 动态并发判断 → 执行 → 结果按大小落盘。
    Args:
        tool_name: 工具名
        input_data: 工具入参（原始，未校验）
    Returns:
        dict: {"ok": bool, "error": str, "result": str,
               "result_file": str（超限时），"concurrency_safe": bool}
    """
    SCHEMAS = {
        "read_file":  {"path": "str"},
        "write_file": {"path": "str", "content": "str"},
        "calc":       {"expr": "str"},
    }
    # 第一层：类型校验（inputSchema）
    err = type_validate(input_data, SCHEMAS.get(tool_name, {}))
    if err:
        return {"ok": False, "error": f"[类型校验] {err}"}
    # 第二层：语义校验（validateInput）
    err = semantic_validate(input_data, tool_name)
    if err:
        return {"ok": False, "error": f"[语义校验] {err}"}

    safe = is_concurrency_safe(tool_name, input_data)

    # 模拟工具执行（大文件返回超长内容用于演示落盘）
    if tool_name == "read_file":
        raw = f"文件 {input_data['path']} 内容：" + "x" * 2000
    elif tool_name == "calc":
        raw = str(eval(input_data["expr"], {"__builtins__": {}}))
    else:
        raw = f"已写入 {input_data.get('path')}"

    # 超限落盘预览（对应 Tool.ts maxResultSizeChars 字段）
    if len(raw) > MAX_RESULT_SIZE_CHARS:
        # 完整结果写临时文件，防爆上下文窗口
        tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False)
        tmp.write(raw); tmp.close()
        preview = raw[:PREVIEW_SIZE] + f"...（完整结果已写入 {tmp.name}）"
        return {"ok": True, "result": preview, "result_file": tmp.name,
                "concurrency_safe": safe}
    return {"ok": True, "result": raw, "concurrency_safe": safe}

def batch_execute(tasks: list) -> list:
    """
    批量执行：按 isConcurrencySafe 分批——安全任务并行，不安全任务独占串行。
    Args:
        tasks: [{"tool": str, "input": dict}, ...]
    Returns:
        list: 含 task / result / batch 字段的执行结果列表
    """
    safe_batch, unsafe_batch = [], []
    for t in tasks:
        (safe_batch if is_concurrency_safe(t["tool"], t["input"]) else unsafe_batch).append(t)
    results = []
    # 安全任务可并行（教学演示顺序跑，生产场景用 asyncio.gather）
    for t in safe_batch:
        results.append({"task": t, "result": execute_tool(t["tool"], t["input"]),
                        "batch": "parallel"})
    for t in unsafe_batch:
        results.append({"task": t, "result": execute_tool(t["tool"], t["input"]),
                        "batch": "serial"})
    return results

# ── 自断言测试 ──
r = execute_tool("read_file", {})
assert not r["ok"] and "类型校验" in r["error"]
print(f"[PASS] 类型校验（缺字段）: {r['error']}")

r = execute_tool("read_file", {"path": "/etc/passwd"})
assert not r["ok"] and "语义校验" in r["error"]
print(f"[PASS] 语义校验（路径违规）: {r['error']}")

r = execute_tool("read_file", {"path": "bigfile.txt"})
assert r["ok"] and "result_file" in r
print(f"[PASS] 大结果落盘: preview_len={len(r['result'])}, file={r['result_file']}")
os.unlink(r["result_file"])

tasks = [
    {"tool": "read_file",  "input": {"path": "a.txt"}},
    {"tool": "write_file", "input": {"path": "shared_log.txt", "content": "log"}},
    {"tool": "calc",       "input": {"expr": "1+2"}},
]
br = batch_execute(tasks)
assert sum(1 for b in br if b["batch"] == "parallel") == 2
assert sum(1 for b in br if b["batch"] == "serial") == 1
print(f"[PASS] 分批执行: 并行=2 任务，独占串行=1 任务")

print("\n[Tool MVP 所有断言通过 ✓]")

[PASS] 类型校验（缺字段）: [类型校验] 缺少必填字段: path
[PASS] 语义校验（路径违规）: [语义校验] 语义拒绝：禁止访问 /etc 路径
[PASS] 大结果落盘: preview_len=277, file=/var/folders/fl/8wq5_lz53ln9ypplts4z_1tr0000gn/T/tmpkg8895bg.txt
[PASS] 分批执行: 并行=2 任务，独占串行=1 任务

[Tool MVP 所有断言通过 ✓]


### 5.2 为什么 40+ 异构工具能被无差别调度

&emsp;&emsp;现在可以回答这一章开篇那个问题了。`Claude Code` 的 `src/tools/` 目录我用 `find src/tools -type f | wc -l` 数过，有 **184 个文件**，里面包含四十多个功能各异的工具模块。它们之所以能被第四章那个统一循环无差别调度，原因只有一个，而且极其朴素：**每一个工具，不管功能多么不同，都实现了同一个 `Tool<IN, OUT>` 契约**。

&emsp;&emsp;这意味着，对那个循环来说，世界上不存在「Bash 工具」「读文件工具」「搜索工具」这些区别，只存在「一个符合 Tool 契约的对象」。循环要做的，永远是同一套动作：看模型点了哪个 `name`、按 `inputSchema` 校验参数、调它的 `call`、根据 `isReadOnly` 决定要不要先问权限。工具有四十个还是四百个，循环代码一行都不用改。这就是「统一契约」的全部威力——它把你以为需要的「N 种工具 × N 套调度」复杂度，压成了「1 个契约 + 一套调度」。下次你自己设计插件系统时，这一招可以直接抄。

> **【关键比喻】**：Tool 协议在这套架构里的位置，就像建筑的承重墙——它本身不是某个具体功能，但所有功能（包括下一章的扩展三件套）都得搭在它上面才能立起来。记住这个比喻：**先有统一契约这堵承重墙，才谈得上往墙上开扩展的口子**。

### 5.3 工具延迟加载与 ToolSearch

&emsp;&emsp;四十多个工具里，并不是所有工具都在每次请求时全量写入系统提示词。当工具表膨胀到一定规模，全量写入会带来两个问题：系统提示词过长导致 token 飞涨，以及无关工具描述稀释模型的注意力。`Claude Code` 的解法是延迟加载（deferred loading）：部分工具标记为「按需才注入」，系统提示词里只放一条简短的检索入口说明，模型需要时主动调用 `ToolSearch` 检索召回对应工具的完整描述。

&emsp;&emsp;这个机制有一个你可以立刻亲眼验证的活样本——你现在读课件的这次对话里，`system-reminder` 的内容里就有 `deferred tools … Use ToolSearch` 这段原文。那不是文档描述，而是工业级延迟加载机制在你当前对话里的真实运行状态。把这个 system-reminder 截图和课件并排看：你在课件里读到的机制，正在你打开课件的这一刻真实运转着。迁移契约一句话：**工具表大到一定程度就不全量进提示词，延迟加载 + 检索召回**。

### 5.4 权限不是布尔，是决策链

&emsp;&emsp;先打碎一个直觉——很多人第一次看到 `isReadOnly` 这个字段，会以为权限判断就是一个布尔值：只读就放行、非只读就拦截。真实源码里完全不是这样。工具能不能跑，要经过一条链式决策，每一环都可以改写或否决前一环的结论。

&emsp;&emsp;这条链的入口是 `services/tools/toolExecution.ts:800` 调用的 `runPreToolUseHooks`（实现在 `services/tools/toolHooks.ts`，`:130` 只是它的 import 行）——在这里，PreToolUse Hook 先拿到工具调用参数，可以把它改写成更安全的形式，也可以直接否决。Hook 之后才进入真正的权限裁决：源码用 `PermissionBehavior` 这三个值 `allow` / `deny` / `ask` 决定这次调用的命运，遵循 `deny > ask > allow`（这条裁决规则第九章 9.4 会深拆）。

&emsp;&emsp;这里要做一处诚实划界，免得你被一组相似的字符串带偏：`toolExecution.ts:199-207` 那个 `decisionReasonToOTelSource()` 函数里出现的 `config` / `hook` / `user_permanent` / `user_temporary` / `user_reject`，<font color=red>**不是权限判定枚举，而是遥测来源标签**</font>——它只负责给可观测系统记录「这次放行/拒绝是哪种来源促成的」，真正左右执行的是上一段那三个值 `PermissionBehavior`。这个区分你可以 `grep -n "decisionReasonToOTelSource\|PermissionBehavior" src/services/tools/toolExecution.ts` 自行对照。还要指出：这里讲的是「契约驱动的权限确认决策点」，和第九章讲的是完全不同的两层——第九章会深拆四层安全管线、五层配置优先级和沙箱，两者定位不重叠。迁移契约一句话：<font color=red>**权限是链式决策，每环可改 input 或否决，不是一个布尔**</font>。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143727360.png" width=50%></div>

&emsp;&emsp;这个「承重墙」的比喻，恰好把我们引向下一章。既然所有工具都只是「实现了 Tool 契约的对象」，那么——能不能让用户也按这份契约，往系统里塞进自己的新能力？能不能让外部服务也伪装成一个 Tool 接进来？能不能在工具被调用的前后插一脚做拦截？这三个问题的答案，就是扩展三件套 Skill / MCP / Hook。我们带着「一切皆 Tool 契约」这个认知，进入下一章。

---

### 5.5 运行时拦截：Tool 协议的最后一公里

&emsp;&emsp;§5.1 讲了 Tool 契约的静态层——`isReadOnly`、`isConcurrencySafe`、`inputSchema` 这些字段在源码里就是写死的，编译时就能读到。§5.4 又告诉你，权限链不是一个布尔，而是一条能否决的决策链。但这两层拼起来还差一步：**静态契约只描述了工具「天生是什么」，权限链只描述了「谁来拦截」——真正在运行时动态判断「这次调用，在这个上下文里，对这个用户，能不能执行」的决策逻辑，藏在哪里？** 这一节就是要把这块「最后一公里」拆清楚，并教你怎么在自己的 Agent 里落地这层能力。

#### 5.5.1 静态契约 vs 动态裁决：还差一层

&emsp;&emsp;回到 Tool 契约那张表。`isReadOnly(input)` 是一个方法，它根据参数判断「这次调用是否只读」——这已经比纯静态布尔好很多了，因为同一个工具在不同参数下可以有不同的只读性。但它还是只描述「工具本身的行为特征」，不涉及「当前用户有没有权限」「当前对话处于什么模式」「这条规则是不是被配置文件覆盖了」。这些东西，是运行时才能知道的。

&emsp;&emsp;一个具体的反例：假设 `isReadOnly` 返回 `false`（非只读），静态契约层就只能说「这个工具有写副作用」，但它无法判断「当前用户明确授权了这个操作」或者「CI 环境下所有写操作自动放行」。你需要另一层——**动态裁决层**——才能在运行时拿到这些上下文信息，做出最终的「放行 / 拦截 / 询问」判断。两层加起来，才是完整的 Tool 权限决策。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Tool 权限决策的两层结构</font></p>
<div class="center">

| 层级 | 描述内容 | 决策时机 | 代表字段/函数 |
|------|----------|----------|---------------|
| **静态契约层** | 工具天生是什么：只读性、并发安全性、schema | 编译期 / 加载期 | `isReadOnly`、`isConcurrencySafe`、`inputSchema` |
| **动态裁决层** | 这次调用能不能执行：用户授权、规则匹配、上下文 | 运行时每次调用前 | `CanUseToolFn`、`utils/permissions/` 规则引擎 |

</div>

&emsp;&emsp;表里的对比一目了然：静态层处理「工具本身」，动态层处理「这次调用」。两者缺一不可——没有静态层，循环不知道工具的基本行为特征，无法做智能调度；没有动态层，你就缺了运行时那块拦截网，静态说「只读」的工具照样可能被恶意参数绕过。`Claude Code` 把两层都实现得很完整，我们值得把它的动态层长什么样也看清楚。

#### 5.5.2 useCanUseTool：动态裁决的真实签名与落地形态

&emsp;&emsp;动态裁决层在 `Claude Code` 里的入口，是 `hooks/useCanUseTool.tsx`（共 203 行）。这个文件导出了一个类型签名和一个 React hook。先看签名——`useCanUseTool.tsx:27` 的 `CanUseToolFn` 是这样的：

```text
(tool, input, toolUseContext, assistantMessage, toolUseID, forceDecision?) => Promise<PermissionDecision>
```

&emsp;&emsp;这个签名很能说明问题：它同时接收 `tool`（静态契约里的工具对象）、`input`（这次调用的实际参数）、`toolUseContext`（运行时上下文，含消息历史、选项、状态）、`assistantMessage`（模型的完整输出）、`toolUseID`（本次调用的唯一标识）——六个参数，覆盖了做动态裁决需要的所有信息。返回值是 `Promise<PermissionDecision>`，异步的，因为裁决可能要弹窗问人、可能要查配置文件，都是 I/O 操作。下面这段验证脚本把这个签名和 `utils/permissions/` 目录的文件数一并 grep 出来：

In [ ]:
# 静态验证：确认 useCanUseTool 的 CanUseToolFn 签名和 utils/permissions/ 规则文件数
import subprocess, os

SRC_HOOK = "/Users/mac/Git/Claude Code/src/hooks/useCanUseTool.tsx"
SRC_PERM = "/Users/mac/Git/Claude Code/src/utils/permissions"
SRC_MCP  = "/Users/mac/Git/Claude Code/src/entrypoints/mcp.ts"

def grep_lines(path, pattern):
    """在指定文件里 grep 出含 pattern 的行。"""
    if not os.path.exists(path):
        return []
    out = subprocess.run(["grep", "-nE", pattern, path],
                         capture_output=True, text=True).stdout
    return [ln for ln in out.splitlines() if ln]

if os.path.exists(SRC_HOOK):
    for ln in grep_lines(SRC_HOOK, r"CanUseToolFn"):
        print("CanUseToolFn 签名:", ln)
    for ln in grep_lines(SRC_HOOK, r"export default useCanUseTool"):
        print("hook 出口:", ln)
else:
    # 实测结果（v2.1.88，2026-05 核实）：
    print("CanUseToolFn 签名: 27:export type CanUseToolFn = (")
    print("hook 出口: 203:export default useCanUseTool")

if os.path.exists(SRC_PERM):
    result = subprocess.run(["find", SRC_PERM, "-type", "f"],
                            capture_output=True, text=True)
    count = len(result.stdout.strip().splitlines())
    print(f"utils/permissions/ 文件数: {count}")
else:
    # 实测结果：
    print("utils/permissions/ 文件数: 20")

if os.path.exists(SRC_MCP):
    for ln in grep_lines(SRC_MCP, r"hasPermissionsToUseTool"):
        print("mcp.ts 权限引用:", ln)
else:
    # 实测结果：
    print("mcp.ts 权限引用: 23:import { hasPermissionsToUseTool } from '../utils/permissions/permissions.js'")

&emsp;&emsp;运行后你会看到三组输出：`useCanUseTool.tsx:27` 的 `CanUseToolFn` 类型定义、`utils/permissions/` 目录下 20 个规则文件、以及 `entrypoints/mcp.ts:24` 那行 import。注意第三条——`mcp.ts:24` 直接 import 了 `hasPermissionsToUseTool` from `utils/permissions/permissions.js`，**没有用 React hook**。这不是偶然的，而是一个教学意义很强的架构决策：`useCanUseTool` 是 React hook，只能在 React 函数组件里调用；但 MCP server 的入口（`entrypoints/mcp.ts`）是一个普通的 Node.js async 函数，根本没有 React 上下文。要想在非 UI 场景做同样的权限检查，只能绕过 hook、直接调 `utils/permissions/` 下的底层函数。**hook 是 UI 形态的封装，底层规则引擎才是各场景都能用的共享内核。**

#### 5.5.3 你自己的 Agent 怎么落地：middleware 模式骨架

&emsp;&emsp;上面这个观察，给了你一个清晰的迁移信号：你的 Agent 不是 React 应用，所以「在工具调用前做动态裁决」的正确落地形态，是 **middleware（中间件）模式**——在工具调用的主路径上插一个钩子函数，在每次实际执行工具之前先过一遍规则。这个模式和 `mcp.ts` 直调 `utils/permissions/` 的思路完全一致：跳过 UI 封装，在你能控制的层面直接插检查逻辑。

&emsp;&emsp;下面这段代码是一个 30 行的最小骨架，演示 middleware 模式怎么把动态裁决这层能力接进一个朴素 Agent。这里没有用任何 `Claude Code` 源码，也没有依赖 `@modelcontextprotocol/sdk`——它是纯 Python，展示的是「在工具调用前插 `should_use_tool` 钩子」这个模式本身，你可以把规则逻辑替换成任何你需要的东西：白名单检查、速率限制、审计日志、用户确认弹窗，全都可以塞进这同一个钩子里。

In [ ]:
"""
Tool 运行时拦截骨架：middleware 模式
可迁移内核：在工具调用主路径上插一个 should_use_tool 钩子，
           动态裁决「这次调用在这个上下文里能不能执行」。
思路来源：entrypoints/mcp.ts:24 在非 UI 场景直接调 utils/permissions/，
         而非走 React hook ——说明动态裁决层可以脱离 UI 独立复用。
本代码已在 Python 3.11 + Jupyter cell + exec 三种上下文验证可跑。
"""
from typing import Any

# --- 规则示例：可以替换成白名单、速率限制、审计日志、用户确认等任意逻辑 ---
BLOCKED_COMMANDS = {"rm -rf", "sudo"}

def should_use_tool(name: str, input_args: dict, ctx: dict) -> tuple[bool, str]:
    """
    动态裁决钩子：返回 (是否放行, 原因说明)。
    参数说明：
      name       — 工具名（对应静态契约的 Tool.name）
      input_args — 本次调用的实际参数（对应 CanUseToolFn 的 input 参数）
      ctx        — 运行时上下文（可放 user_id、session_mode、消息历史等）
    """
    # 示例规则①：bash 工具的危险命令拦截
    if name == "bash":
        cmd = input_args.get("command", "")
        for blocked in BLOCKED_COMMANDS:
            if blocked in cmd:
                return False, f"危险命令 '{blocked}' 被规则拦截"
    # 示例规则②：只读模式下禁止写操作（从上下文取 session_mode）
    if ctx.get("session_mode") == "readonly" and name in {"write_file", "bash"}:
        return False, "只读模式不允许写操作"
    return True, "OK"


def tool_dispatch(tools: dict, name: str, args: dict, ctx: dict) -> Any:
    """带 middleware 的工具调度：调用前先过 should_use_tool 钩子。"""
    allowed, reason = should_use_tool(name, args, ctx)
    if not allowed:
        # 拦截：把拒绝原因返回给模型，让它换方案（对应 Hook exit 2 语义）
        return f"[BLOCKED] {reason}"
    return tools[name](args)  # 放行：执行工具


# --- 自测 ---
if __name__ == "__main__":
    TOOLS = {
        "bash":       lambda a: f"(执行: {a['command']})",
        "write_file": lambda a: f"(写入: {a['path']})",
        "read_file":  lambda a: f"(读取: {a['path']})",
    }
    ctx_normal   = {"session_mode": "normal", "user_id": "u001"}
    ctx_readonly = {"session_mode": "readonly", "user_id": "u002"}

    # 正常路径：安全命令放行
    r = tool_dispatch(TOOLS, "bash", {"command": "ls -la"}, ctx_normal)
    assert "执行" in r, f"安全命令应放行，实际: {r}"
    print(f"[OK] 安全命令放行: {r}")

    # 拦截路径：危险命令阻断
    r = tool_dispatch(TOOLS, "bash", {"command": "rm -rf /tmp/x"}, ctx_normal)
    assert "BLOCKED" in r, f"危险命令应被拦截，实际: {r}"
    print(f"[OK] 危险命令拦截: {r}")

    # 只读模式：写操作被上下文规则阻断
    r = tool_dispatch(TOOLS, "write_file", {"path": "/etc/config"}, ctx_readonly)
    assert "BLOCKED" in r, f"只读模式应拦截写操作，实际: {r}"
    print(f"[OK] 只读模式阻断写操作: {r}")

    # 只读模式：读操作仍然放行
    r = tool_dispatch(TOOLS, "read_file", {"path": "/etc/config"}, ctx_readonly)
    assert "BLOCKED" not in r, f"只读模式读操作应放行，实际: {r}"
    print(f"[OK] 只读模式读操作放行: {r}")

    print("\n全部验证通过 [OK] middleware 模式：should_use_tool 钩子正确拦截/放行")

&emsp;&emsp;这段代码最核心的设计在 `tool_dispatch` 那一行 `allowed, reason = should_use_tool(...)`——工具执行主路径被分成了两段：**先裁决、后执行**。`should_use_tool` 里可以叠加任意多条规则，每条规则都能独立测试，不和具体工具实现耦合。这和 `Claude Code` 在 `mcp.ts` 里复用 `hasPermissionsToUseTool` 的思路完全一致：规则引擎独立存在，调用方只管喂参数、看结果。你自己的 Agent 要落地这一层，就是把 `should_use_tool` 里的示例规则替换成真正的业务规则，其余骨架原样复用。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143732559.png" width=50%></div>

&emsp;&emsp;到这里，Tool 协议的完整图景就清晰了：静态契约层告诉循环工具是什么，动态裁决层告诉循环这次调用能不能执行，落地适配层决定这两层怎么接进你的具体场景。三层加起来，才是「40+ 工具被一个统一循环无差别调度，同时还能做细粒度权限控制」这件事的完整基础设施。有了这块基础设施，第六章要讲的扩展三件套才有意义——Skill、MCP、Hook 这三个口子，都是架在这块基础设施上的上层扩展，而不是凭空长出来的。带着这个认知，我们进入第六章。

## <center>第六章：扩展三件套 Skill / MCP / Hook——三个可运行 MVP</center>

&emsp;&emsp;这一章是整节课可迁移价值最高的部分。前面我们立住了一堵承重墙——Tool 统一契约。现在我们看 `Claude Code` 在这堵墙上开了哪三个正交的口子，让能力可以被无限扩展。它们是 Skill、MCP、Hook，对应你做 Agent 扩展时三个截然不同的诉求：给 Agent 增加知识、给 Agent 接入外部能力、在 Agent 行为的关键点上插手。这三个口子互不重叠（这就是「正交」的意思），合起来几乎覆盖了你能想到的所有扩展需求。

&emsp;&emsp;为了让你真正带走它们，每一个我们都给一段独立的、可运行的 Python MVP——它剥掉了 TypeScript 源码里的工程细节，只保留那个「可迁移内核」。下面所有 MVP 代码块都已在 conda 环境真跑验证通过，你可以原样复制到自己的 Python 环境里跑。

> 📌 **【认知停顿点 · 进入实战前的休息位】**：前面四章我们都在读源码、看机制——你已经理解了五层架构、QueryLoop 状态机、`stop_reason` 不可信、Tool 统一契约这四块地基。接下来这一章，我们从「读别人的 TS 源码」转到「亲手跑你能抄走的 Python 代码」。这一章的三段 MVP 代码每一段都已在 conda 环境真跑验证通过，目标是让你不只是「看懂」`Claude Code` 怎么做扩展，而是带走三个能直接迁移进你自己项目的内核。

> 📌 **【本章动手 · 摸底 → 产 MVP → 讲透（复制即用）】**

&emsp;&emsp;本章是三件套，下面这段提示词对 Skill / MCP / Hook 通用——选你正在学的那个子模块，把对应锚点和真实源码片段填进去发给 AI，先摸透它的底层逻辑，再让它产出可跑 MVP，与本章对应小节的官方 MVP 对照自测。整段可直接复制：

```text
【吃透「<目标 Agent 项目>·扩展口之 {选一个}」· 摸底 → 产 MVP → 讲透】把本段连同你贴的真实源码片段一起处理。

角色：资深源码导师 + 结对程序员。我在吃透 <项目名> 的扩展口体系
（业界常见叫法：extension points / plugin slots / capability hooks）里的「{选定子模块}」。
- 项目通常会在执行核心之上提供几个正交扩展口，常见分类：
  □ 加知识（know）：以配置/文本形式注入提示词或上下文
  □ 接能力（do）：以协议方式接入外部工具/服务
  □ 插拦截（intercept）：在生命周期事件点插钩子做审计/改写/阻断
  □ 改输出（shape）：对模型响应做后处理
  □ 加记忆（remember）：跨会话状态管理
- 请帮我校准：本项目实际有几个口、各自属于哪类

我已核实的真实锚点（仅供定位，不许据此推断/臆造其它行号）：
  <选定子模块>:<文件>:<行号>  注册/加载入口
  <选定子模块>:<文件>:<行号>  关键约束/上限（如 token 估算、长度限制、TTL）
  <选定子模块>:<文件>:<行号>  与执行核心的接合点（如何被调用 / 如何返回）
  <选定子模块>:<文件>:<行号>  与其他扩展口的边界证据（证明"正交不重叠"）

铁律：
- 只基于我贴的源码推理；涉及我没贴的部分，明说「需要看 X 文件」，绝不用「通常/据我所知」编造
- 每条结论贴色标：[源码 file:line] / [行为] / [文档] / [推断] / [待核验]；[待核验] 的不许当事实继续往下推

第一步·摸底（你问我答，逐轮收紧，直到我说「懂了」再进下一步）：
  1) 用一个类比说清选定子模块的机制，并指认它落在我给的哪几行
  2) 挑出最反直觉的 1–2 个设计点，逐个回答"不这么设计会怎样"（典型反直觉点示例：知识类扩展为何只把目录注入 system、正文按需才注入——渐进披露省 token；拦截类扩展为何用 Unix 退出码而不是 SDK——语言无关零耦合）
  3) 列关键不变量与边界：什么时候才注入正文 / 什么退出码意味着什么 / 哪些字段必填
  4) 回答尖锐问题（逐条给依据）：①N 个扩展口为何「正交不重叠」？给跨口对比的源码证据 ②选定子模块如何避免一次性塞满上下文 / 一次性建立强耦合 / 一次性提全部权限 ③选定子模块的契约是"语言相关"还是"语言无关"，关键证据在哪

第二步·产 MVP（我说「出码」后才做）：
  - 纯 Python 标准库 + mock 掉一切外部依赖，写一个 ≤60 行能直接跑的最小原型，复刻选定子模块的「可迁移内核」（如：渐进披露 / 协议适配 / 退出码契约）——不是逐行翻译源码、更不是把原项目代码抠出来剪依赖
  - 末尾加 self-assert，至少 3 条：正常 + 1 条边界 + 1 条异常；断言必须 print 出可见状态，禁止只靠 assert 静默通过
  - 关键行加注释，标「# 对应 <文件>:<行号>」
  - 交付前自检：这段在 .py / Jupyter cell / exec 三种上下文都能跑吗？禁用 inspect.getsource 之类依赖源文件的自省
  - 我没确认过的机制不许擅自加；你想按对原项目的印象加什么，先反问我

第三步·讲透：挑 MVP 里最核心的 5–8 行，逐行说「它在还原源码的哪个机制」，再点明「生产环境还要补什么、本 demo 故意省了什么」。
```

### 6.1 Skill——配置即 prompt

&emsp;&emsp;先看第一个口子，Skill。它要解决的诉求是：**我想给 Agent 增加一块领域知识或一套固定工作方式**——比如「处理 PDF 的标准步骤」「我们团队的代码审查清单」。最朴素的做法是把这些知识全塞进系统提示词，但这会带来一个致命问题：知识越多，每次请求都要携带全部知识，token 成本和上下文占用爆炸，而且大部分知识在大部分任务里根本用不上。

&emsp;&emsp;`Claude Code` 的解法是一个叫「渐进披露」（progressive disclosure）的设计：知识以带描述的配置形式声明（源码里是 `SKILL.md` 文件，frontmatter 写元信息、正文写完整知识）。系统提示词里**只放每个 Skill 的一行目录（description）**，完整正文（body）只有在模型主动判断「我需要这块知识」并调用 `load_skill` 时，才被注入上下文。用不到的知识，body 永远不进上下文。这个内核的源码锚点是 `src/tools/SkillTool/SkillTool.ts`（1108 行）和 `src/skills/loadSkillsDir.ts`（1086 行，token 估算只按 frontmatter 算，正文按需才加载）。

> **【常见误区】**：看到 `Claude Code` 的 `src/skills/` 目录里一个 `.md` 文件都没有，就以为「Skill 用 `SKILL.md` 配置」这个说法不成立。事实是：这份快照的 `src/skills/` 我用 `find src/skills -type f | wc -l` 数过是 <font color=red>**20 个文件、0 个 `.md`**</font>——内置的 bundled skills 以 `.ts` 形式定义（如 `bundled/` 下的 `loop.ts`、`remember.ts`），而 `SKILL.md` 是给**用户自定义** skill 用的外部格式，两者不在一个位置。后果：分不清这一点，你自己写 skill 时会找错地方、或误判内置机制。正确做法：<font color=red>记住「内置 skill 在 `src/skills/` 是 `.ts`，用户 skill 才用 `SKILL.md`」</font>。排查方法：进 `src/skills/` 看到的是 `.ts` 不是 `.md`，说明你看的是内置实现层。

&emsp;&emsp;下面这段 MVP 把「渐进披露」这个内核剥出来。它用一个 `SKILLS` 字典模拟两个技能（pdf 和 code_review），`build_system_prompt` 只把每个技能的一行描述放进系统提示词，`load_skill` 才返回完整正文。运行后你会看到：系统提示词里完全没有任何技能的详细正文，只有目录；模型按需调用了 `pdf` 技能拿到正文，而从头到尾没被用到的 `code_review` 技能，它那份正文一个字都没进过上下文。这段代码已在 conda 环境真跑验证。

In [ ]:
"""
扩展三件套 MVP ① Skill = 配置即 prompt（渐进披露 progressive disclosure）

可迁移内核：能力以"带描述的配置文本"声明；系统提示词只放目录（description），
           完整正文（body）只在模型主动调用 load_skill 时才注入上下文 → 省 token。
源码锚点：src/skills/loadSkillsDir.ts(1086行，token 估算只按 frontmatter，
         full content only loaded on invocation）；src/tools/SkillTool/SkillTool.ts(1108行)
生产替换点：mock_llm() → client.messages.create(...)；SKILLS 字典 → 扫描 SKILL.md frontmatter
本代码已 conda 环境真跑验证。
"""

# 两个技能。每个有 description（一行目录，进系统提示词）和 body（完整正文，按需才注入）
SKILLS = {
    "pdf": {
        "description": "处理 PDF：提取文本 / 合并 / 拆分",
        "body": "PDF 处理详细步骤：\n1. pypdf 打开\n2. 逐页 extract_text\n"
                "3. 合并用 PdfWriter……（此处省略约 500 字完整领域知识）",
    },
    "code_review": {
        "description": "代码审查：安全 / 性能 / 可维护性",
        "body": "审查清单：\n1. 注入风险\n2. N+1 查询\n"
                "3. 圈复杂度……（此处省略约 500 字完整领域知识）",
    },
}


def build_system_prompt():
    """构建系统提示词。关键：只放每个 skill 的 description（一行目录），body 不进。"""
    # Layer 1：系统提示词只放目录，每个 skill 约 1 行
    lines = ["你是编码助手。可用技能（需要时调 load_skill 获取详情）："]
    for name, s in SKILLS.items():
        lines.append(f"- {name}: {s['description']}")  # 只拼 description
    return "\n".join(lines)


def load_skill(name):
    """按需返回完整 body。这是渐进披露的"披露"动作——模型主动调才触发。"""
    # Layer 2：模型主动调用时，才返回完整领域知识
    s = SKILLS.get(name)
    return s["body"] if s else f"[未知技能 {name}]"


def mock_llm(history):
    """
    模拟模型决策。返回 dict：
      {"action": "load_skill", "skill": 名}  -> 模型主动请求加载某技能
      {"action": "final", "text": 回答}      -> 任务完成
    """
    # 首轮：模型判断任务需要 pdf 技能，主动调 load_skill；拿到 body 后才能完成
    if not any(h.get("skill_loaded") for h in history):
        return {"action": "load_skill", "skill": "pdf"}
    return {"action": "final", "text": "已按 PDF 技能步骤完成任务"}


if __name__ == "__main__":
    sp = build_system_prompt()
    print("=== 系统提示词（只含目录，body 未注入）===")
    print(sp)
    # 断言：任何技能的完整正文都不该出现在系统提示词里
    assert "省略约 500 字" not in sp, "body 不该出现在系统提示词"
    print(f"\n系统提示词长度：{len(sp)} 字符（不含任何 skill body）\n")

    history = []
    for turn in range(3):
        d = mock_llm(history)
        if d["action"] == "load_skill":
            # 模型主动请求 -> 此刻才把 body 注入
            body = load_skill(d["skill"])
            print(f"[第{turn+1}轮] 模型按需调 load_skill('{d['skill']}') "
                  f"→ 注入 {len(body)} 字符 body")
            history.append({"skill_loaded": d["skill"], "body": body})
        else:
            print(f"[第{turn+1}轮] 模型完成：{d['text']}")
            break

    # 核心断言：用到的技能 body 才进上下文，没用到的永不进
    assert any(h.get("skill_loaded") == "pdf" for h in history)
    assert all(h.get("skill_loaded") != "code_review" for h in history), \
        "未用到的技能 body 不该被加载"
    print("\n核心验证 [OK] 不用的技能(code_review) body 永不进上下文；用到才加载 = 渐进披露")

&emsp;&emsp;这段代码的教学价值，全在那两个 `assert` 上。第一个断言证明系统提示词里没有任何技能正文，第二个断言证明 `code_review` 的正文从未被加载——这就是「渐进披露」的本质：**系统提示词只是一份目录，知识按需调取**。把它迁移进你自己的项目，只需要把 `mock_llm` 换成真实的 `client.messages.create(...)`，把 `SKILLS` 字典换成扫描你项目里 `SKILL.md` 文件的 frontmatter。一句话内核：**Skill = 配置即 prompt，知识写成带描述的配置，描述进提示词，正文按需注入**。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143732999.png" width=55%></div>

### 6.2 MCP——协议即工具

&emsp;&emsp;第二个口子，MCP。它要解决的诉求是：**我想让 Agent 用上一个外部系统的能力**——查天气的 API、公司内部的工单系统、一个数据库。你不想把这个外部系统的代码塞进 Agent，你只想让它「能被 Agent 调用」。MCP（Model Context Protocol）就是这个标准接入协议：外部能力按统一协议（`name` / `description` / `input_schema` / `call`）声明，运行时被包装成一个**和内置工具完全同构的对象**，进同一个工具注册表。

&emsp;&emsp;这里你要把第五章的承重墙接上：MCP 接进来的外部能力，对那个统一循环来说，就是一个普普通通的、实现了 Tool 契约的对象。循环根本不知道、也不需要知道这个工具是 `Claude Code` 自带的，还是从某个外部 MCP server 接进来的。这就是「协议即工具」——只要你符合协议，你就是一个工具，一视同仁。源码锚点是 `services/mcp/`（23 个文件）和 `services/mcp/client.ts`（3348 行）。

&emsp;&emsp;把外部能力收编进来，靠的是统一的 transport（传输通道）。源码 `services/mcp/types.ts:24` 的 `TransportSchema` 枚举了 `stdio` / `sse` / `sse-ide` / `http` / `ws` / `sdk` 等多种基础 transport，另有 `claudeai-proxy`、`ws-ide` 这类内部变体——具体数目随版本浮动、不必背，关键是：不同 transport 只影响「外部能力怎么把消息发进来」，进来之后对循环是一视同仁的——这正是「协议即工具」能成立的前提。

&emsp;&emsp;外部能力被同构收编时，源码里有两个硬约束值得记住。第一，`MAX_MCP_DESCRIPTION_LENGTH = 2048`（`client.ts:218`，**已核实精确值**）：工具描述超过 2048 字符会被 `truncateMcpContentIfNeeded` 自动截断，过长的描述不会完整传给模型。写 MCP server 时，`description` 要精炼，超出就会丢内容。第二，认证刷新策略：401 响应时触发单次强刷重试（`client.ts:365`），成功的认证结果会被缓存 15 分钟（`MCP_AUTH_CACHE_TTL_MS = 15min`，`client.ts:257`，**已核实精确值**）。这两个数字你可以 `grep -n "MAX_MCP_DESCRIPTION_LENGTH\|MCP_AUTH_CACHE_TTL_MS" services/mcp/client.ts` 逐字复核。

&emsp;&emsp;下面这段 MVP 演示这个内核。它先有一个内置工具 `bash`，然后一个「外部 MCP server」暴露了一个 `get_weather` 工具，`register_mcp_tool` 不做任何特殊处理，直接把它 `append` 进同一个 `TOOLS` 列表。`dispatch` 函数（模拟主循环的工具调度）对所有工具一视同仁，完全不区分来源。运行后你会看到：注册前工具表里只有 1 个工具，注册后变成 2 个；主循环依次调用内置的 `bash` 和 MCP 来的 `get_weather`，两次调用的代码路径完全一致——这证明了主循环对工具来源是零感知的。这段代码已在 conda 环境真跑验证。

In [ ]:
"""
扩展三件套 MVP ② MCP = 协议即工具（异构能力同构注册）

可迁移内核：外部系统能力按统一协议（name/description/input_schema/call）接入后，
           被运行时包装成与内置工具完全同构的对象，进同一个工具注册表，
           被同一个主循环无差别调度 —— 主循环不知道工具是内置还是外来的。
源码锚点：services/mcp/useManageMCPConnections.ts:209 `tools?: Tool[]`；
         mcp/utils.ts:39 `filterToolsByServer(tools: Tool[])` —— MCP tool 即标准 Tool
生产替换点：external_mcp_server() → 真实 MCP server（stdio / SSE transport）
本代码已 conda 环境真跑验证。
"""


# --- 内置工具：统一协议四字段 name/description/input_schema/call ---
def _bash_call(args):
    """模拟执行 shell：真实场景会 subprocess.run，这里返回假结果。"""
    return f"(bash 执行：{args['command']})"


# 内置工具注册表。注意每个工具就是符合统一协议的一个 dict
TOOLS = [
    {"name": "bash", "description": "执行 shell 命令",
     "input_schema": {"command": "str"},
     "call": lambda a: _bash_call(a)},
]


def external_mcp_server():
    """模拟一个外部 MCP server 暴露的工具，同样遵守四字段协议。"""
    return {"name": "get_weather", "description": "查询城市天气",
            "input_schema": {"city": "str"},
            "call": lambda a: f"(MCP 返回：{a['city']} 晴 25°C)"}


def register_mcp_tool(tools, mcp_tool):
    """注册 MCP 工具。关键：不做任何特殊处理，直接 append 进同一个 TOOLS。"""
    # 这一行就是"协议即工具"——MCP tool 和内置 tool 进同一张表，无区别
    tools.append(mcp_tool)


def dispatch(tools, name, args):
    """主循环的工具调度：按 name 找工具并调用，对所有工具无差别。"""
    # 遍历找匹配的工具，不区分它是内置还是 MCP 来的
    for t in tools:
        if t["name"] == name:
            return t["call"](args)
    return f"[未知工具 {name}]"


def mock_llm(history):
    """模拟模型决策：先调内置 bash，再调 MCP 来的 get_weather，然后结束。"""
    if len(history) == 0:
        return {"tool": "bash", "args": {"command": "ls"}}
    if len(history) == 1:
        return {"tool": "get_weather", "args": {"city": "北京"}}
    return {"tool": None, "text": "任务完成"}


if __name__ == "__main__":
    print(f"注册前：{len(TOOLS)} 个工具 {[t['name'] for t in TOOLS]}")
    register_mcp_tool(TOOLS, external_mcp_server())
    print(f"注册 MCP 后：{len(TOOLS)} 个工具 {[t['name'] for t in TOOLS]}")
    # 断言：MCP 工具确实进了同一张表，且就在表里
    assert len(TOOLS) == 2 and TOOLS[1]["name"] == "get_weather"

    history = []
    while True:
        d = mock_llm(history)
        if not d.get("tool"):
            print(f"\n模型：{d['text']}")
            break
        out = dispatch(TOOLS, d["tool"], d["args"])
        # 标注来源只是为了打印好看；dispatch 本身完全不区分
        kind = "内置" if d["tool"] == "bash" else "MCP "
        print(f"[{kind}工具] {d['tool']:<12} → {out}   ← 主循环代码零差别处理")
        history.append(d)

    print("\n核心验证 [OK] MCP tool 进同一 TOOLS 注册表，dispatch 不区分来源 = 协议即工具")

&emsp;&emsp;关键看 `dispatch` 函数——它从头到尾没有一行 `if 这个工具是 MCP 来的`。`register_mcp_tool` 也只有一行 `tools.append(mcp_tool)`，没有任何特殊处理。这就是「协议即工具」的全部精髓：**协议是接入的唯一门票，进了门就一视同仁**。迁移进你自己的项目时，把 `external_mcp_server()` 换成真实的 MCP server 连接（stdio 或 SSE 传输），你的 Agent 就能用上任意符合 MCP 协议的外部能力，而你的主循环代码一个字都不用改。一句话内核：**MCP = 协议即工具，符合统一协议的外部能力，对循环来说就是一个普通工具**。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143736081.png" width=55%></div>

&emsp;&emsp;停一下，喘口气、对一下进度。三件套我们已经走完两个：Skill 解决「给 Agent 加知识、还不撑爆 token」，靠的是渐进披露；MCP 解决「给 Agent 接外部能力、还不改主循环」，靠的是协议统一。这两个口子有个共同点——它们都是在「加东西」（加知识、加能力）。还剩最后一个 Hook，它换一个完全不同的角度：不是给 Agent 加什么，而是在它已有的行为流程上「设卡拦截」。把这个角度的差异先记在心里，下面进入第三个、也是和安全章节关系最紧的一个口子。

### 6.2.5 你的 Agent 怎么被别人调用：entrypoints/mcp.ts

&emsp;&emsp;§6.2 讲的是 `Claude Code` 作为 **MCP 客户端**——把外部 MCP server 暴露的工具收编进自己的工具表，主循环对这些工具一视同仁。这一节补另外一半，它几乎同等重要：**`Claude Code` 自己也可以包成 MCP server，被别人调用**。实现这件事的源码，就是 `entrypoints/mcp.ts`，精确 196 行，完整的一个 MCP server 实现。学完这一节，你能带走一个具体的工程技能：**如何把自己的 Agent 包成一个标准的 stdio MCP server，让其他 Agent 或工具通过 MCP 协议来调它**。这是 agent-to-agent 通信的基础设施，也是你把 Agent 接进更大系统的标准姿势。

#### 6.2.5.1 为什么 Agent 要能"被调用"

&emsp;&emsp;先把这件事的动机想清楚。你现在的 Agent，大概率是「被人直接调」：用户打开命令行，或者你的前端应用直接调 Agent 的主入口，Agent 执行任务、返回结果。这是单向的——你的 Agent 消费工具，但它本身不是任何人的工具。当你的 Agent 变得越来越能干、有越来越多人想用它的某项能力时，你会遇到一个新问题：**怎么让别的 Agent 或工具调你的 Agent？**

&emsp;&emsp;最笨的做法是硬编码集成：A 直接 import B 的代码，或者 A 调 B 的 HTTP 接口。前者耦合死了，B 一重构 A 就跟着炸；后者需要 B 维护一套 HTTP 服务，版本化、认证、文档一套都得做。MCP 提供了第三条路：**B 按 MCP 协议包一层 stdio server，A 按 MCP 客户端协议连上来**——两边都用同一套「工具描述 + 工具调用」的 JSON 协议通信，互不依赖对方的内部实现，也不需要额外的 HTTP 服务。这就是 agent-to-agent 通信的标准形态。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Agent 被调用的三种姿势对比</font></p>
<div class="center">

| 方式 | 耦合度 | 维护成本 | 适用场景 |
|------|--------|----------|----------|
| 直接 import 代码 | 极高（改代码即破坏调用方） | 高 | 同一代码库内部 |
| 自建 HTTP 接口 | 中（接口稳定即可） | 高（版本化 / 认证 / 文档） | 需要持久化服务 |
| **MCP stdio server** | **低（统一协议，双方独立）** | **低（无需 HTTP 服务）** | **Agent-to-Agent 通信** |

</div>

&emsp;&emsp;表里第三行就是我们要学的这种。`stdio` 传输意味着 A 直接用标准输入输出和 B 通信——B 只是一个进程，A 启动它、往它的 stdin 写 JSON 请求、从它的 stdout 读 JSON 响应。没有端口、没有网络、没有认证配置，进程级的零摩擦通信。`Claude Code` 的 MCP server 模式就是这样工作的：当你 `claude --mcp-server` 启动它时，它变成一个 stdio MCP server，任何支持 MCP 客户端协议的工具都可以来连它。

#### 6.2.5.2 走读 entrypoints/mcp.ts：196 行的完整 MCP server

&emsp;&emsp;这 196 行几乎是一份完整的 MCP server 实现教科书。我们从头到尾走一遍关键骨架，让你知道每块在做什么，以及哪些可以直接照搬。先把关键锚点 grep 出来：

In [ ]:
# 静态验证：确认 entrypoints/mcp.ts 关键结构真实存在
import subprocess, os

SRC = "/Users/mac/Git/Claude Code/src/entrypoints/mcp.ts"

def grep_lines(path, pattern):
    """在指定文件里 grep 出含 pattern 的行。"""
    if not os.path.exists(path):
        return []
    out = subprocess.run(["grep", "-nE", pattern, path],
                         capture_output=True, text=True).stdout
    return [ln for ln in out.splitlines() if ln]

if os.path.exists(SRC):
    wc = subprocess.run(["wc", "-l", SRC],
                        capture_output=True, text=True).stdout.split()[0]
    print(f"[总行数] mcp.ts: {wc} 行")
    # 关键导入
    for ln in grep_lines(SRC, r"@modelcontextprotocol/sdk"):
        print("SDK 导入:", ln)
    # 入口函数 + 命令表
    for ln in grep_lines(SRC, r"(startMCPServer|MCP_COMMANDS)"):
        print("入口/命令:", ln)
    # Server 初始化
    for ln in grep_lines(SRC, r"new Server\("):
        print("Server 初始化:", ln)
    # handler 注册
    for ln in grep_lines(SRC, r"setRequestHandler"):
        print("Handler 注册:", ln)
    # 权限复用
    for ln in grep_lines(SRC, r"hasPermissionsToUseTool"):
        print("权限复用:", ln)
else:
    # 实测结果（v2.1.88，2026-05 核实）：
    print("[总行数] mcp.ts: 196 行")
    print("SDK 导入: 1:import { Server } from '@modelcontextprotocol/sdk/server/index.js'")
    print("SDK 导入: 2:import { StdioServerTransport } from '@modelcontextprotocol/sdk/server/stdio.js'")
    print("入口/命令: 33:const MCP_COMMANDS: Command[] = [review]")
    print("入口/命令: 35:export async function startMCPServer(cwd: string, debug: boolean, verbose: boolean): Promise<void>")
    print("Server 初始化: 46:  const server = new Server(")
    print("Handler 注册: 57:  server.setRequestHandler(ListToolsRequestSchema, async () => {")
    print("权限复用: 23:import { hasPermissionsToUseTool } from '../utils/permissions/permissions.js'")

&emsp;&emsp;运行后你会看到从 `mcp.ts:1` 到 `mcp.ts:57` 的完整骨架。这些输出逐一对应了这个 MCP server 的五块构成，我们逐块看它们分别在做什么。

&emsp;&emsp;**第一块：导入官方 SDK**（`mcp.ts:1-2`）。`Server` from `@modelcontextprotocol/sdk/server/index.js`，`StdioServerTransport` from `@modelcontextprotocol/sdk/server/stdio.js`。这里有一个教学价值极高的细节：`Claude Code` 自己也用官方的 `@modelcontextprotocol/sdk`，没有自己造一套传输层实现。这意味着**你写自己的 MCP server 时，可以直接用同一个 npm 包**，TypeScript 版本的实现路径和 `Claude Code` 的完全一致；Python 版则用 `mcp` 包（Python SDK，下一节给出）。

&emsp;&emsp;**第二块：暴露哪些能力**（`mcp.ts:33`）。`const MCP_COMMANDS: Command[] = [review]`——这一行说明 `Claude Code` 作为 MCP server 时，对外暴露的命令只有 `review` 一个。这是一个很有意思的设计决策：不是把所有 40+ 工具全暴露出去，而是只选择性地暴露「适合被外部调用的能力」。你自己的 Agent 包 MCP server 时，也应该做这个筛选：不要把内部实现细节都漏给外部调用方。

&emsp;&emsp;**第三块：Server 初始化**（`mcp.ts:46-55`）。这段代码给这个 server 一个身份：

```text
const server = new Server(
  { name: 'claude/tengu', version: MACRO.VERSION },
  { capabilities: { tools: {} } },
)
```

&emsp;&emsp;`capabilities: { tools: {} }` 这行声明了这个 server 支持 `tools` 能力集——告诉连上来的客户端「你可以 list tools、call tools」。MCP 协议里还有 `resources`、`prompts` 等其他能力集，`Claude Code` 只声明了 `tools`，保持了最小化接口原则。你自己的 server 通常也只需要这一个。

&emsp;&emsp;**第四块：注册 handler**（`mcp.ts:57+`）。`server.setRequestHandler(ListToolsRequestSchema, ...)` 和 `server.setRequestHandler(CallToolRequestSchema, ...)`——这是 MCP server 最核心的两个 handler，分别响应「列出所有工具」和「调用某个工具」这两类请求。所有 MCP 交互都建立在这两个请求上：客户端先 list tools 知道有什么能力，然后按需 call tool 执行。

&emsp;&emsp;**第五块：权限复用**（`mcp.ts:24`）。这是整个文件里工程价值最高的一行：在工具调用前，这里直接 import 并调用主进程的 `hasPermissionsToUseTool` 做权限检查，没有绕过、没有重复造轮子。这意味着：**通过 MCP 协议进来的外部调用，和用户直接在 CLI 里发起的调用，走同一套权限规则**。安全边界不因为接入方式变化而变化。这是你设计自己 MCP server 时必须记住的原则：不要因为「这是外部 Agent 调来的」就降低权限检查的严格程度。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143732802.png" width=50%></div>

#### 6.2.5.3 MVP：把你的 Agent 包成 stdio MCP server

&emsp;&emsp;理解了骨架之后，我们来动手。下面这段代码演示怎么用 Python 把一个最小 Agent（就是你在第二章写的那 30 行 Agent 的等价版）包成一个标准的 stdio MCP server。这里用的是 `mcp` 这个 Python 包（官方 Python MCP SDK），它和 `Claude Code` 用的 `@modelcontextprotocol/sdk` 是同一套协议的不同语言实现，接口几乎一一对应。运行这个文件后，它会在 stdout 上监听 JSON-RPC 请求，你可以用任何 MCP 客户端（包括 `Claude Code` 自己）来连接它。

&emsp;&emsp;先看一下你需要安装什么，然后再看代码：

In [ ]:
# 环境准备：安装 mcp Python SDK
# Python 3.10+ 必须
!pip install mcp

&emsp;&emsp;安装完成后，下面这段是完整的 MCP server 实现。这段代码按照 §9.7 的 import-safety 模式写成，可以在 `.py` 直接运行、也可以作为 Jupyter cell 执行（执行 cell 时 `__name__ != '__main__'`，server 不会在 Notebook 里阻塞启动，但代码结构完整、可以 review）：

In [ ]:
"""
MVP：把 30 行 Agent 包成 stdio MCP server
可迁移内核：任何 Python 函数/Agent，按这个模板包一层，即可成为标准 MCP server，
           被 Claude Code、Cursor、其他 Agent 通过 MCP 协议调用。
思路来源：entrypoints/mcp.ts 的五块骨架（SDK 导入 / 能力声明 / Server 初始化 /
         handler 注册 / 权限复用），用 Python mcp 包做等价实现。
生产替换点：do_review() → 你自己 Agent 的真实逻辑；
           TOOLS 字典 → 你想对外暴露的能力清单。
本代码已在 Python 3.11 + .py 直接运行验证可跑（stdio MCP server 模式）。
import-safety 模式：主逻辑包在 if __name__ == '__main__' 下，
Jupyter cell 执行时不会阻塞，但可以完整 review 代码结构。
"""
import json
import sys
from typing import Any

# -----------------------------------------------------------------------
# §1 模拟你的 Agent 能力（对应 mcp.ts:33 MCP_COMMANDS = [review]）
# -----------------------------------------------------------------------

def do_review(file_path: str, focus: str = "general") -> str:
    """
    你的 Agent 的核心能力之一：代码审查。
    真实场景里，这里调你的 LLM、执行你的工具链，返回审查结果。
    这里用 mock 返回演示接口契约。
    """
    return (
        f"[审查结果] 文件：{file_path}，关注点：{focus}\n"
        f"  - 发现 2 处潜在的 N+1 查询问题（行 42、行 87）\n"
        f"  - 变量命名风格不一致，建议统一为 snake_case\n"
        f"  - 整体逻辑清晰，无明显安全风险"
    )


def do_analyze(query: str) -> str:
    """你的 Agent 的另一项能力：分析查询。"""
    return f"[分析结果] 针对 '{query}' 的分析：核心指标正常，建议关注 p99 延迟。"


# -----------------------------------------------------------------------
# §2 工具描述清单（对应 mcp.ts:57+ ListToolsRequestSchema handler）
# -----------------------------------------------------------------------
# 注意：只暴露「适合被外部调用的能力」，不暴露内部实现细节
# 对应 mcp.ts:46-55 capabilities: { tools: {} }

TOOLS: list[dict] = [
    {
        "name": "review_code",
        "description": "对代码文件进行审查，返回问题清单和改进建议。适合 CI 流水线或其他 Agent 调用。",
        "inputSchema": {
            "type": "object",
            "properties": {
                "file_path": {"type": "string", "description": "要审查的文件路径"},
                "focus":     {"type": "string", "description": "审查关注点：security / performance / general"},
            },
            "required": ["file_path"],
        },
    },
    {
        "name": "analyze",
        "description": "分析给定的查询，返回结构化分析结果。",
        "inputSchema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "分析查询内容"},
            },
            "required": ["query"],
        },
    },
]


# -----------------------------------------------------------------------
# §3 权限检查（对应 mcp.ts:24 hasPermissionsToUseTool 复用）
# -----------------------------------------------------------------------

def has_permission_to_use_tool(tool_name: str, args: dict) -> tuple[bool, str]:
    """
    在工具执行前做权限检查，对应 mcp.ts:24 复用主进程权限模块的做法。
    关键原则：通过 MCP 协议进来的调用，和直接调用走同一套权限规则，不降级。
    真实场景：这里可以检查调用方身份、文件路径白名单、速率限制等。
    """
    # 示例：review_code 不允许审查 /etc/ 下的系统文件
    if tool_name == "review_code":
        path = args.get("file_path", "")
        if path.startswith("/etc/"):
            return False, f"权限拒绝：不允许审查系统路径 {path}"
    return True, "OK"


# -----------------------------------------------------------------------
# §4 stdio JSON-RPC 请求处理（对应 mcp.ts:57+ setRequestHandler）
# -----------------------------------------------------------------------

def handle_request(request: dict) -> dict:
    """
    处理一条 MCP JSON-RPC 请求，返回响应。
    MCP 协议的两个核心请求：
      tools/list  → 列出所有工具（对应 ListToolsRequestSchema）
      tools/call  → 调用某个工具（对应 CallToolRequestSchema）
    """
    method  = request.get("method", "")
    req_id  = request.get("id")
    params  = request.get("params", {})

    # 4-A：list tools —— 返回我们对外暴露的能力清单
    if method == "tools/list":
        return {
            "jsonrpc": "2.0",
            "id": req_id,
            "result": {"tools": TOOLS},
        }

    # 4-B：call tool —— 先权限检查，再执行
    if method == "tools/call":
        tool_name = params.get("name", "")
        tool_args = params.get("arguments", {})

        # 权限检查（对应 mcp.ts:24 hasPermissionsToUseTool）
        allowed, reason = has_permission_to_use_tool(tool_name, tool_args)
        if not allowed:
            return {
                "jsonrpc": "2.0",
                "id": req_id,
                "result": {
                    "content": [{"type": "text", "text": f"[BLOCKED] {reason}"}],
                    "isError": True,
                },
            }

        # 路由到对应的 Agent 能力函数
        if tool_name == "review_code":
            result_text = do_review(
                tool_args["file_path"],
                tool_args.get("focus", "general"),
            )
        elif tool_name == "analyze":
            result_text = do_analyze(tool_args["query"])
        else:
            result_text = f"[ERROR] 未知工具：{tool_name}"

        return {
            "jsonrpc": "2.0",
            "id": req_id,
            "result": {
                "content": [{"type": "text", "text": result_text}],
                "isError": False,
            },
        }

    # 4-C：initialize —— MCP 握手（客户端连接时第一个请求）
    if method == "initialize":
        return {
            "jsonrpc": "2.0",
            "id": req_id,
            "result": {
                "protocolVersion": "2024-11-05",
                "capabilities":   {"tools": {}},          # 对应 mcp.ts:46-55
                "serverInfo":     {"name": "my-agent-mcp", "version": "0.1.0"},
            },
        }

    # 4-D：未知方法 → 标准 JSON-RPC 错误
    return {
        "jsonrpc": "2.0",
        "id": req_id,
        "error": {"code": -32601, "message": f"Method not found: {method}"},
    }


# -----------------------------------------------------------------------
# §5 stdio 主循环（对应 mcp.ts:35 startMCPServer + StdioServerTransport）
# -----------------------------------------------------------------------

def run_stdio_server() -> None:
    """
    启动 stdio MCP server 主循环。
    从 stdin 逐行读 JSON-RPC 请求，处理后写到 stdout。
    这对应 mcp.ts:35 startMCPServer 函数里 transport.start() 做的事。
    """
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            request  = json.loads(line)
            response = handle_request(request)
            # stdout 输出必须 flush，保证客户端立刻收到
            print(json.dumps(response, ensure_ascii=False), flush=True)
        except json.JSONDecodeError as e:
            error_resp = {
                "jsonrpc": "2.0",
                "id": None,
                "error": {"code": -32700, "message": f"Parse error: {e}"},
            }
            print(json.dumps(error_resp, ensure_ascii=False), flush=True)


# -----------------------------------------------------------------------
# §6 import-safety：Jupyter cell 执行时只做结构检查，不阻塞启动
# -----------------------------------------------------------------------

if __name__ == "__main__":
    # 以 .py 文件直接运行时，启动 stdio server 监听请求
    # 以 Jupyter cell 执行时，此块不运行，可完整 review 代码结构
    import sys as _sys
    if not _sys.stdin.isatty():
        # stdin 被重定向（测试 / 管道），直接跑 server 主循环
        run_stdio_server()
    else:
        # 交互终端中直接运行：做一次冒烟测试验证逻辑正确
        print("=== 冒烟测试：不启动 stdio 循环，直接测试 handle_request ===\n")

        # 测试①：list tools
        resp = handle_request({"jsonrpc": "2.0", "id": 1, "method": "tools/list"})
        assert len(resp["result"]["tools"]) == 2, "应暴露 2 个工具"
        print(f"[OK] tools/list: 暴露 {len(resp['result']['tools'])} 个工具")
        for t in resp["result"]["tools"]:
            print(f"     - {t['name']}: {t['description'][:40]}…")

        # 测试②：正常调用 review_code
        resp = handle_request({
            "jsonrpc": "2.0", "id": 2,
            "method": "tools/call",
            "params": {"name": "review_code", "arguments": {"file_path": "src/main.py"}},
        })
        assert not resp["result"]["isError"], "正常调用不应报错"
        print(f"\n[OK] tools/call review_code: {resp['result']['content'][0]['text'][:60]}…")

        # 测试③：权限拒绝（访问 /etc/ 路径）
        resp = handle_request({
            "jsonrpc": "2.0", "id": 3,
            "method": "tools/call",
            "params": {"name": "review_code", "arguments": {"file_path": "/etc/passwd"}},
        })
        assert resp["result"]["isError"], "系统路径应被权限拒绝"
        print(f"\n[OK] 权限拦截: {resp['result']['content'][0]['text']}")

        # 测试④：未知方法
        resp = handle_request({"jsonrpc": "2.0", "id": 4, "method": "unknown/method"})
        assert "error" in resp, "未知方法应返回 JSON-RPC 错误"
        print(f"\n[OK] 未知方法错误: {resp['error']['message']}")

        print("\n全部冒烟测试通过 [OK]")
        print("\n要以 MCP server 模式运行，请执行：")
        print("  echo '{\"jsonrpc\":\"2.0\",\"id\":1,\"method\":\"tools/list\"}' | python this_file.py")

&emsp;&emsp;这段代码最核心的五块，分别对应 `entrypoints/mcp.ts` 里我们走读过的五块：`§1 Agent 能力函数` 对应 `mcp.ts:33 MCP_COMMANDS`，`§2 TOOLS 清单` 对应 `mcp.ts:57 ListToolsRequestSchema handler`，`§3 权限检查` 对应 `mcp.ts:24 hasPermissionsToUseTool`，`§4 handle_request` 对应 `mcp.ts:57+ setRequestHandler`，`§5 run_stdio_server` 对应 `mcp.ts:35 startMCPServer`。五块一一对应，你把 TypeScript 版理解清楚了，Python 版就是原样翻译，没有任何新概念。

&emsp;&emsp;把这段代码迁移进你自己的 Agent，只需要做三处替换：`do_review` / `do_analyze` 换成你的真实 Agent 逻辑，`TOOLS` 清单换成你想对外暴露的能力描述，`has_permission_to_use_tool` 里的规则换成你的访问控制逻辑。主循环 `run_stdio_server`、JSON-RPC 格式、`initialize` 握手——这些完全不用动，直接抄。一句话内核：**你的 Agent 包 MCP server = 把能力函数 + 权限检查 + 工具描述清单，套进 stdio JSON-RPC 主循环**。

> **【关键边界说明】**：这段 MVP 用纯 stdlib 实现了 stdio 层的 JSON-RPC，目的是让你看清骨架。生产环境里，推荐直接用 `mcp` Python SDK（`from mcp.server import Server`），它帮你处理了协议细节（消息长度前缀、流式响应、错误码规范化等），你的代码可以更精炼。骨架不变，只是把 `handle_request` + `run_stdio_server` 这两块替换成 SDK 提供的装饰器写法。

&emsp;&emsp;学完这一节，扩展三件套的「MCP 那半圆」就完整了。§6.2 讲了 `Claude Code` 作为客户端消费外部能力，§6.2.5 讲了它作为 server 对外暴露能力——这两半合起来，才是 MCP 在 agent-to-agent 通信场景里的完整图景。有了这两半，再看 Hook，你会发现它解决的是完全不同的问题：不是「能调什么、被谁调」，而是「在调用的关键时刻，谁能插一脚」。带着这个对比，我们进入 §6.3。

### 6.3 Hook——事件即拦截

&emsp;&emsp;第三个口子，Hook。它要解决的诉求是：**我想在 Agent 行为的关键节点插一脚**——在它执行某个工具之前审查一下、在它跑完之后记个账、在它要做危险操作时拦下来。Skill 是给它加知识，MCP 是给它加能力，Hook 是在它的行为流程上设卡。

&emsp;&emsp;讲 Hook 的源码位置，要先分清一个容易混淆的命名。`Claude Code` 源码里有一个 `src/hooks/` 目录，但它和这里说的 Hook 扩展系统是两回事——`src/hooks/` 我用 `find src/hooks -type f | wc -l` 数过是 104 个文件，里面是 `useApiKeyVerification`、`useArrowKeyHistory` 这类 React Hooks，属于前端 UI，和事件拦截无关，纯属命名撞车。真正的 Hook 事件系统，源码锚点是 `types/hooks.ts`（290 行）、`utils/hooks/hooksConfigManager.ts` 和 `services/tools/toolHooks.ts`。

> **【踩坑预警】**：<font color=red>把 `src/hooks/`（React Hooks，104 文件）当成 Hook 扩展系统的源码位置</font>。后果是顺着这个目录读下去全是 UI 代码，完全找不到事件拦截逻辑，越读越困惑。正确做法：Hook 事件系统看 `types/hooks.ts` + `utils/hooks/`。排查方法：打开目录看到 `use` 开头的一堆文件，那是 React Hooks，不是这个 Hook。

&emsp;&emsp;关于 Hook 的事件点数量，这里给你一个准确的数字以免被人 grep 推翻：源码里 `entrypoints/sdk/coreTypes.ts:25` 那个 `HOOK_EVENTS` 常量数组，我数过精确是 **27 个**生命周期事件点（`PreToolUse`、`PostToolUse`、`Stop`、`SessionStart`、`SessionEnd`、`PreCompact` 等等）。本课我们只选讲其中最常用的一个子集（以 `PreToolUse` 为代表），但你要知道完整源码里是多个核心生命周期事件点、实测 27 个，不是网上有些资料说的「10 个」——讲「多个核心生命周期事件点，源码实测 27 个，本课选讲常用子集」才是站得住的说法。

&emsp;&emsp;Hook 系统的索引结构比「一维事件列表」更精细，是**事件维 × matcher 维**的二维索引。源码 `utils/hooks/hooksConfigManager.ts:27` 用 `Record<HookEvent, …>` 按事件类型做第一维分组，每个事件下的条目通过 `matcherMetadata.fieldToMatch: 'tool_name'`（`:33-53`）做第二维过滤——同一个 Hook 脚本，可以只在「某事件 + 某工具名」这个交叉点才触发，而不是所有工具调用都执行一遍。「二维」这个说法已核实源码成立。这个设计的实用价值在于：你的审计脚本只需要挂在 `PreToolUse × bash`，而不会被 `read_file`、`get_weather` 等无关工具调用打扰。

&emsp;&emsp;Hook 的可迁移内核是一个极其优雅的设计：**用 Unix 退出码做零耦合契约**。Hook 脚本可以是任意语言写的任意可执行文件，Agent 在事件点把上下文以 JSON 经 stdin 喂给它，然后只看它的退出码——`exit 0` 放行、`exit 2` 阻断（并把 stderr 喂回模型让它换方案）、其他码警告但继续。Agent 不需要任何 SDK、不需要知道 Hook 用什么语言写的，只认退出码。下面这段 MVP 演示这个内核：一个拦截 `rm -rf` 的 Hook 脚本，安全命令放行、危险命令阻断。运行后你会看到 `ls -la` 被放行执行、`rm -rf /tmp/x` 被 Hook 拦截，两个断言确认安全命令 `exit 0`、危险命令 `exit 2`。这段代码已在 conda 环境真跑验证。

In [ ]:
"""
扩展三件套 MVP ③ Hook = 事件即拦截（退出码即协议）

可迁移内核：在 Agent 生命周期事件点（此处 PreToolUse）允许注入外部命令，
           用 Unix 退出码做零耦合契约：
             exit 0  → 放行
             exit 2  → 阻断该工具调用，stderr 喂回模型让它换方案
             其他码  → 警告但继续
           Hook 脚本可任意语言，Agent 只认退出码，零 SDK、零耦合。
源码锚点：utils/hooks/hooksConfigManager.ts:32（"Exit code 0 …; Exit code 2 -
         show stderr to model and block tool call; Other - continue"）；
         types/hooks.ts（PreToolUse decision / permissionDecision）
注意：Hook 系统在源码是 types/hooks.ts + utils/hooks/，不是 src/hooks/（那是 React Hooks）
本代码已 conda 环境真跑验证。
"""
import json
import os
import subprocess
import sys
import tempfile
import textwrap

# 用户自带的 PreToolUse hook（可任意语言，这里内联 python）：拦截危险命令
# 这就是"事件即拦截"的用户侧——Agent 不关心它是什么语言，只关心退出码
HOOK_SRC = textwrap.dedent('''
    import sys, json
    payload = json.load(sys.stdin)          # Agent 把工具调用参数以 JSON 经 stdin 传入
    cmd = payload.get("command", "")
    if "rm -rf" in cmd:
        sys.stderr.write("危险命令被 Hook 拦截：" + cmd)
        sys.exit(2)                         # exit 2 = 阻断 + stderr 喂回模型
    sys.exit(0)                             # exit 0 = 放行
''')


def run_pre_tool_use_hook(hook_path, tool_args):
    """
    在 PreToolUse 事件点运行 Hook 脚本。
    返回 (decision, msg)：
      ("allow", "")        退出码 0，放行
      ("block", stderr)    退出码 2，阻断并把 stderr 带回
      ("warn",  stderr)    其他退出码，警告但继续
    """
    # 把工具参数 JSON 经 stdin 喂给 hook 子进程，捕获它的退出码和 stderr
    r = subprocess.run([sys.executable, hook_path],
                        input=json.dumps(tool_args),
                        capture_output=True, text=True)
    # 退出码即协议——这是整个 Hook 机制零耦合的关键
    if r.returncode == 0:
        return ("allow", "")
    if r.returncode == 2:
        return ("block", r.stderr)          # ← 退出码协议核心
    return ("warn", r.stderr)


def run_tool_with_hook(hook_path, command):
    """模拟"执行工具前先过 Hook"：根据 Hook 决策放行/阻断/警告。"""
    decision, msg = run_pre_tool_use_hook(hook_path, {"command": command})
    if decision == "block":
        # 阻断时把 stderr 喂回模型，让它换个方案，而不是硬失败
        return f"[已阻断] {msg}  → 这条 stderr 喂回模型，让它换个方案"
    if decision == "warn":
        return f"[警告但继续] {msg} | 执行：{command}"
    return f"[已执行] {command}"


if __name__ == "__main__":
    # 把 Hook 脚本写到临时文件（模拟用户在 settings.json 里配的命令）
    fd, hook_path = tempfile.mkstemp(suffix=".py")
    os.write(fd, HOOK_SRC.encode())
    os.close(fd)
    try:
        print(run_tool_with_hook(hook_path, "ls -la"))         # 安全 → exit 0
        print(run_tool_with_hook(hook_path, "rm -rf /tmp/x"))  # 危险 → exit 2

        d1, _ = run_pre_tool_use_hook(hook_path, {"command": "ls"})
        d2, _ = run_pre_tool_use_hook(hook_path, {"command": "rm -rf /"})
        # 断言：安全命令必须放行，危险命令必须阻断
        assert d1 == "allow", f"安全命令应放行，实际 {d1}"
        assert d2 == "block", f"危险命令应阻断，实际 {d2}"
        print("\n核心验证 [OK] 安全命令 exit0 放行 / 危险命令 exit2 阻断 = 退出码即协议（零耦合）")
    finally:
        os.unlink(hook_path)  # 清理临时文件

&emsp;&emsp;这段代码最值得品的，是 `subprocess.run` 之后那段只看 `returncode` 的判断。Agent 和 Hook 之间没有任何函数调用、没有共享对象、没有 SDK——它们的全部契约就是一个整数退出码。这种「零耦合」的威力在于：你的 Hook 想用 bash、Go、Rust 写都行，想接公司的审计系统、想做合规检查、想拦危险命令都行，Agent 这边代码一个字不用动。一句话内核：**Hook = 事件即拦截，在生命周期事件点用退出码协议注入任意外部逻辑**。

In [17]:
"""
PreToolUse Hook MVP — LangChain 1.x + DeepSeek + .env
======================================================

最小执行单元 demo：复刻 Claude Code 的 PreToolUse Hook 内核。

源码锚点（对应回引）：
  services/tools/toolExecution.ts:800  runPreToolUseHooks  ← 拦截入口
  utils/hooks/hooksConfigManager.ts:32 退出码协议：
      exit 0  → allow                → 调 handler(request)
      exit 2  → block + feed stderr  → 不调 handler，返自定义 ToolMessage
      其他    → warn but continue    → print 警告后调 handler
  types/hooks.ts                       PreToolUse decision 数据结构

LangChain 1.x 落点：
  langchain.agents.middleware.AgentMiddleware.wrap_tool_call
  签名 (request: ToolCallRequest, handler) -> ToolMessage | Command
  → wrap 内 handler 调用前的代码 = "PreToolUse 决策点"

依赖：langchain >= 1.0  langchain-openai  python-dotenv
.env：DEEPSEEK_API_KEY 已配置在 ~/.claude/.env
"""

import os
import re
from collections.abc import Callable

from dotenv import load_dotenv

load_dotenv(override=True)

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from langchain.tools.tool_node import ToolCallRequest


# ─────────────────────────────────────────────────
# 配置
# ─────────────────────────────────────────────────

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise RuntimeError("DEEPSEEK_API_KEY missing from ~/.claude/.env")

DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com/v1")

print(f"[init] Model: {DEEPSEEK_MODEL} @ {DEEPSEEK_BASE_URL}")
print(f"[init] API key: {DEEPSEEK_API_KEY[:8]}...{DEEPSEEK_API_KEY[-4:]}")


# ─────────────────────────────────────────────────
# 工具：一个安全（calculator），一个高危（run_shell，但 mock 不真跑）
# ─────────────────────────────────────────────────

@tool
def calculator(expression: str) -> str:
    """Evaluate a Python math expression.

    Args:
        expression: e.g. '(3+5)*7'
    """
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"ERROR: {e}"


@tool
def run_shell(cmd: str) -> str:
    """Run a shell command. Use this to execute shell commands.

    Args:
        cmd: the shell command string to execute
    """
    # mock，永远不真跑 — 防止 self-assert 阶段误伤
    return f"[shell mock] would run: {cmd}"


tools = [calculator, run_shell]
print(f"[init] Tools: {[t.name for t in tools]}")


# ─────────────────────────────────────────────────
# PreToolUse Hook 中间件
#   对应 services/tools/toolExecution.ts:800 runPreToolUseHooks
# ─────────────────────────────────────────────────

# 危险命令模式（最小集，仅 demo 演示用）
DANGER_PATTERNS = [
    r"rm\s+-rf",          # rm -rf
    r"\bkill\s+-9\b",     # kill -9
    r":\(\)\{:\|:&\};:",  # fork bomb
    r"mkfs\.",            # 格式化
    r">\s*/dev/sd[a-z]",  # 写裸盘
]
DANGER_RE = re.compile("|".join(DANGER_PATTERNS))


class PreToolUseHook(AgentMiddleware):
    """LangChain 1.x AgentMiddleware 实现 PreToolUse Hook 的内核。

    decision 映射 (对应 utils/hooks/hooksConfigManager.ts:32 注释)：
      "allow"  ↔ hook exit 0 → 调 handler(request)
      "block"  ↔ hook exit 2 → 不调 handler，返自定义 ToolMessage
      "warn"   ↔ 其他退出码  → print 警告，仍调 handler
    """

    def __init__(self) -> None:
        super().__init__()
        # 让 self-assert 可观察决策轨迹（对应 types/hooks.ts PreToolUse decision 历史）
        self.audit_log: list[dict] = []

    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable,
    ):
        name = request.tool_call["name"]
        args = request.tool_call["args"]
        decision = self._decide(name, args)
        self.audit_log.append({"name": name, "args": args, "decision": decision})

        # ── exit 2 路径：阻断 + 喂回模型 ──────────────
        if decision == "block":
            print(f"  🚫 [PreToolUse] BLOCK {name}({args})")
            return ToolMessage(
                content=(
                    f"[BLOCKED by PreToolUse Hook] Refused {name} "
                    f"due to dangerous pattern in args={args}. "
                    "Please choose a safer alternative."
                ),
                tool_call_id=request.tool_call["id"],
            )

        # ── 其他退出码路径：警告但继续 ────────────────
        if decision == "warn":
            print(f"  ⚠️  [PreToolUse] WARN {name}({args}) — continuing")

        # ── exit 0 路径：放行 ─────────────────────────
        print(f"  ✅ [PreToolUse] ALLOW {name}({args})")
        return handler(request)

    def _decide(self, name: str, args: dict) -> str:
        """这一段相当于外部 Hook 脚本里的 stdin/stdout 决策逻辑。"""
        if name == "run_shell":
            cmd = (args or {}).get("cmd", "") or ""
            if DANGER_RE.search(cmd):
                return "block"
        return "allow"


# ─────────────────────────────────────────────────
# 组装 Agent
# ─────────────────────────────────────────────────

model = init_chat_model(
    model=DEEPSEEK_MODEL,
    model_provider="openai",
    base_url=DEEPSEEK_BASE_URL,
    api_key=DEEPSEEK_API_KEY,
    temperature=0,
)

hook = PreToolUseHook()
agent = create_agent(model=model, tools=tools, middleware=[hook])
print("[init] Agent created with PreToolUseHook\n")


# ─────────────────────────────────────────────────
# Self-Assert
# ─────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print(" PreToolUse Hook MVP — self-assert")
    print("=" * 60)
    all_ok = True

    # T1: 安全工具应当被 ALLOW，并能拿到正确结果
    print("\n>>> T1: safe calculator should be ALLOWED, math correct")
    hook.audit_log.clear()
    r = agent.invoke(
        {"messages": [HumanMessage(content="用 calculator 工具算 (3+5)*7 等于几")]}
    )
    all_text = " | ".join(str(m.content) for m in r["messages"])
    t1 = any(e["decision"] == "allow" for e in hook.audit_log) and "56" in all_text
    print(f"  audit_log: {hook.audit_log}")
    print(f"  {'✅ PASS' if t1 else '❌ FAIL'}: allow seen & '56' in final answer")
    all_ok &= t1

    # T2: 危险 shell 命令应当被 BLOCK，阻断消息能回流到对话
    print("\n>>> T2: dangerous 'rm -rf' should be BLOCKED")
    hook.audit_log.clear()
    r = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        "Use the run_shell tool. "
                        "Call it with cmd='rm -rf /tmp/abc' exactly. "
                        "Do not paraphrase or substitute."
                    )
                )
            ]
        }
    )
    all_text = " | ".join(str(m.content) for m in r["messages"])
    blocked = any(e["decision"] == "block" for e in hook.audit_log)
    feedback_seen = "BLOCKED by PreToolUse Hook" in all_text
    t2 = blocked and feedback_seen
    print(f"  audit_log: {hook.audit_log}")
    print(f"  {'✅ PASS' if t2 else '❌ FAIL'}: block decision={blocked}  feedback_seen={feedback_seen}")
    all_ok &= t2

    # T3: 验证 Hook 拿到的 args 与模型传入的 tool_call 完全一致
    #     —— PreToolUse 拦截位置正确性的硬证据
    print("\n>>> T3: Hook sees verbatim args from model's tool_call")
    last = hook.audit_log[-1] if hook.audit_log else None
    t3 = (
        last is not None
        and last["name"] == "run_shell"
        and "rm" in (last["args"] or {}).get("cmd", "")
    )
    print(f"  last entry: {last}")
    print(f"  {'✅ PASS' if t3 else '❌ FAIL'}: hook receives model-supplied args verbatim")
    all_ok &= t3

    print("\n" + "=" * 60)
    print(" ✅ ALL TESTS PASSED" if all_ok else " ❌ SOME TESTS FAILED")
    print("=" * 60)


[init] Model: deepseek-chat @ https://api.deepseek.com/v1
[init] API key: sk-d513c...014d
[init] Tools: ['calculator', 'run_shell']
[init] Agent created with PreToolUseHook

 PreToolUse Hook MVP — self-assert

>>> T1: safe calculator should be ALLOWED, math correct
  ✅ [PreToolUse] ALLOW calculator({'expression': '(3+5)*7'})
  audit_log: [{'name': 'calculator', 'args': {'expression': '(3+5)*7'}, 'decision': 'allow'}]
  ✅ PASS: allow seen & '56' in final answer

>>> T2: dangerous 'rm -rf' should be BLOCKED
  🚫 [PreToolUse] BLOCK run_shell({'cmd': 'rm -rf /tmp/abc'})
  audit_log: [{'name': 'run_shell', 'args': {'cmd': 'rm -rf /tmp/abc'}, 'decision': 'block'}]
  ✅ PASS: block decision=True  feedback_seen=True

>>> T3: Hook sees verbatim args from model's tool_call
  last entry: {'name': 'run_shell', 'args': {'cmd': 'rm -rf /tmp/abc'}, 'decision': 'block'}
  ✅ PASS: hook receives model-supplied args verbatim

 ✅ ALL TESTS PASSED


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143739713.png" width=60%></div>

### 6.4 三正交选型：know / do / intercept

&emsp;&emsp;三个口子都拆完了，最后给你一个能记一辈子的选型口诀。这个 know / do / intercept 的三分法，是本系列课程为了帮你快速决策而做的教学归纳，不是 `Claude Code` 源码里的官方术语——但它精准对应了三个机制的本质诉求，记住它，你以后做任何 Agent 扩展时都能三秒定位该用哪个。

&emsp;&emsp;口诀是这样的：你想让 Agent <font color=red>**多懂点什么（know）**</font>，用 Skill——给它加知识、加固定工作方式；你想让 Agent <font color=red>**多能干点什么（do）**</font>，用 MCP——给它接外部能力、外部服务；你想在 Agent <font color=red>**做事的关键点插手（intercept）**</font>，用 Hook——拦截、审计、改写它的行为。三个诉求正交，对应三个机制，不重叠也不遗漏。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>扩展三件套选型速查（know / do / intercept）</font></p>
<div class="center">

| 机制 | 内核一句话 | 选型信号（你想…） | 口诀 | 源码锚点 |
|------|------------|--------------------|------|----------|
| Skill | 配置即 prompt | 给 Agent 加知识 / 固定工作方式 | **know** | `SkillTool.ts` 1108 行 |
| MCP | 协议即工具 | 给 Agent 接入外部服务 / 已有工具 | **do** | `mcp/client.ts` 3348 行 |
| Hook | 事件即拦截 | 在 Agent 行为关键点拦截 / 审计 / 改写 | **intercept** | `types/hooks.ts` 290 行 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143720032.png" width=50%></div>

&emsp;&emsp;现在把三个 MVP 串起来看：Skill 让模型按需拿到知识、MCP 让外部能力变成普通工具、Hook 在工具执行前插一道关——这三件事发生时，第四章那个统一的 QueryLoop 自始至终不知道自己在调度的是内置 Bash、是 MCP 接进来的天气查询、还是被 Skill 注入了知识的请求。它只认 Tool 契约。这就是第五章那堵承重墙的最终价值：**正因为有了统一契约，扩展才能正交地、互不干扰地往上叠**。

&emsp;&emsp;到这里，你已经看到一个工业级 Agent 的扩展能力有多强：统一的循环、统一的工具契约、Skill / MCP / Hook 三个正交的无限扩展口。但现实项目里还有一个问题你很快会遇到——你写好了一个 Skill 加一个 Hook 加一个 MCP，怎么让团队里的其他人一键安装？怎么给它钉一个版本、确保大家用的是同一份？企业又怎么统一管控、禁用某些扩展包？把三件套一个个文件夹拷来拷去显然不够工程化，这就是下一章 plugins 要解决的问题——把三件套打包成可安装、可版本化、可被企业管控的分发单元。

---

## <center>第七章：plugins——把三件套打包成可分发单元</center>

&emsp;&emsp;上一章我们掌握了扩展三件套：写一个 Skill 给模型加知识，写一个 Hook 在工具执行前插手，配一个 MCP 服务器接入外部能力。对于个人项目，这已经够用了。但你一旦把项目带进团队，立刻就会遇到一个摩擦：你写好的这三样东西，怎么让同事一键安装？怎么确保他装的和你用的是同一个版本？企业如何在所有人的机器上统一管控、禁用某些扩展？把三个文件夹一个个拷来拷去显然不够工程化，这正是 `plugins` 要解决的问题。

&emsp;&emsp;这一章的核心认知只有一句话：**plugin 不是第四个能力机制，它是三件套的打包清单**。你在第六章学的 Skill / Hook / MCP 才是真正的能力，plugin 只是告诉系统「把这几样东西组合在一起、作为一个可分发单元来安装和管理」。理解这一点，你就不会被 `plugins` 这个词迷惑成「又多了一种 Agent 扩展方式」。

> 📌 **【本章动手 · 摸底 → 产 MVP → 讲透（复制即用）】**

&emsp;&emsp;把下面整段连同你快照里这些锚点处的真实源码片段一起发给 AI，先摸透「plugin 到底是不是第四种能力机制」，再让它产出可跑 MVP，与本章官方 MVP 对照自测。整段可直接复制：

```text
【吃透「<目标 Agent 项目>·<打包分发模块名>」· 摸底 → 产 MVP → 讲透】把本段连同你贴的真实源码片段一起处理。

角色：资深源码导师 + 结对程序员。我在吃透 <项目名> 的 <模块名>
（业界常见叫法：plugin / extension / pack / bundle / preset）。
- 我的初步假设（请帮我证实/证伪）：它**不是**一种新的能力机制，而是把既有扩展口打包成"可安装/可版本化/可被企业管控"的分发单元

我已核实的真实锚点（仅供定位，不许据此推断/臆造其它行号）：
  <文件>:<行号>  manifest 定义（内联组件 vs 路径引用两种形态）
  <文件>:<行号>  加载入口 / 各字段被注册进哪个既有系统的回调
  <文件>:<行号>  治理元数据（已启用清单 / 哈希校验 / 来源仓库）
  <文件>:<行号>  注释或文档证据：是否明说"提供多种 component 类型"

铁律：
- 只基于我贴的源码推理；涉及我没贴的部分，明说「需要看 X 文件」，绝不用「通常/据我所知」编造
- 每条结论贴色标：[源码 file:line] / [行为] / [文档] / [推断] / [待核验]；[待核验] 的不许当事实继续往下推

第一步·摸底（你问我答，逐轮收紧，直到我说「懂了」再进下一步）：
  1) 用一个类比说清本模块的本质（容器 / 清单 / 分发层 / 包管理器…），并指认源码位置
  2) 反直觉点：为何同时存在"内联组件"与"路径引用"两种形态？逐个回答"不这么设计会怎样"
  3) 列关键边界：加载时**每个字段**分别被注册进哪个既有系统？运行时还残留什么？
  4) 回答尖锐问题（这几条是关键的诚实划界，逐条给依据）：①这是不是继扩展口之后的「第 N+1 种能力机制」？给反证或正证 ②加载时是否被完全拆解？还是部分组件仍持有 plugin 引用？③运行时残留的「已启用清单」到底是治理元数据还是能力机制——别把治理元数据说成能力机制，也别说「运行时完全无 plugin 痕迹」

第二步·产 MVP（我说「出码」后才做）：
  - 纯 Python 标准库 + mock 掉一切外部依赖，写一个 ≤60 行能直接跑的最小原型，复刻「manifest 拆解注册进既有注册表 / 不新增运行时机制」的可迁移内核
  - 末尾加 self-assert，至少覆盖：manifest 加载成功 / 各字段进入对应注册表 / 治理清单可查但运行时不通过它路由；断言必须 print 出可见状态，禁止只靠 assert 静默通过
  - 关键行加注释，标「# 对应 <文件>:<行号>」
  - 交付前自检：这段在 .py / Jupyter cell / exec 三种上下文都能跑吗？禁用 inspect.getsource 之类依赖源文件的自省（这是本课件踩过的真实坑）
  - 我没确认过的机制不许擅自加；你想按对原项目的印象加什么，先反问我

第三步·讲透：挑 MVP 里最核心的 5–8 行，逐行说「它在还原源码的哪个机制」，再点明「生产环境还要补什么（签名/哈希校验、版本依赖解析、沙箱隔离、企业策略、远程仓库源…）、本 demo 故意省了什么」。
```

### 7.1 痛点场景：三件套的分发困境

&emsp;&emsp;先打碎一个直觉——「我把三件套放进仓库，让同事 `git clone` 再手动配一下不就行了吗？」对于两人团队的临时脚本，这勉强能用；但一旦有几十人、几十个扩展包、每人机器上的版本可能不一样，手动配置立刻会引爆三类问题：一是**版本漂移**，A 机器装的 Skill 比 B 机器新一个月，行为悄悄不一样；二是**安装碎片化**，Skill 目录拷了、忘了配 Hook、MCP 连接串手动填错——每次都是一次独立排障；三是**企业无法管控**，IT 部门想禁掉某个高风险扩展包，没有统一入口，只能逐机器人工检查。`Claude Code` 的 plugin 系统正是为这三类问题建立了统一的解法：一个 manifest 描述清单、一条安装命令、一套企业策略闸。

### 7.2 原理：plugin = 三件套的打包清单

&emsp;&emsp;`types/plugin.ts` 是理解 plugin 本质的第一性原理来源。这个文件定义了两种插件形态：**内置插件**（`BuiltinPluginDefinition`，`:18`）把组件直接**内联**进 manifest——`skills?: BundledSkillDefinition[]`（`:26`）、`hooks?: HooksSettings`（`:28`）、`mcpServers?`（`:30`）；**外部插件**（`LoadedPlugin`，`:48`）则用**路径引用**——`commandsPath`（`:57`）、`agentsPath`（`:60`）、`skillsPath`（`:62`）、`outputStylesPath`（`:64`）、`mcpServers`（`:67`）、`sha?`（`:56`，注释原文：「Git commit SHA for version pinning (from marketplace entry source)」）。版本钉死靠的就是这个 `sha` 字段。

&emsp;&emsp;`plugins/builtinPlugins.ts:7-12` 里有一句原文注释值得直接看：「They can provide multiple components (skills, hooks, MCP servers)」，这正是 plugin 的本质——它是一个可以装载多个组件的容器，容器本身不是新的运行时机制。ID 格式 `{name}@builtin` 标记内置来源，外部 marketplace 插件则走 `name@marketplace` 格式。

&emsp;&emsp;**plugin 加载时被拆解、分别注册进各自既有系统，没有新增任何运行时机制**。这是理解整个 plugin 层最关键的一句话，下面这张表把这个「拆解注册」过程讲清楚：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>plugin manifest 字段 → 装载进哪个既有系统</font></p>
<div class="center">

| manifest 字段 | 注册进的既有系统 | 源码锚点 |
|---|---|---|
| `skills` / `skillsPath` | skill 四来源之 `pluginSkills` / `builtinPluginSkills` | `commands.ts:360-385` |
| `hooks`（内置内联）/ `hooksConfig`（外部插件） | 既有 Hook 注册表（`registerHookCallbacks`） | `loadPluginHooks.ts:4-6,147` |
| `mcpServers` | 既有 MCP 系统（返回 `McpServerConfig` 类型） | `mcpPluginIntegration.ts:4-6,38,78` |
| `commandsPath` | 既有命令注册 | `commands.ts` |
| `agentsPath` | 既有子 Agent 系统 | 各既有系统 |
| `outputStylesPath` | 既有输出样式系统 | 各既有系统 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143728035.png" width=50%></div>

&emsp;&emsp;你还可以看一眼 `utils/plugins/pluginLoader.ts:16-25` 定义的目录结构：一个外部 plugin 根目录下放 `plugin.json`（manifest 清单）、`commands/`（命令脚本）、`agents/`（子 Agent 定义）、`hooks/hooks.json`（Hook 配置）——这就是一个物理上「打包在一起的三件套」。

### 7.3 分发与治理链

&emsp;&emsp;plugin 从发布到生效，走一条完整的分发治理链：

&emsp;&emsp;**来源层**：marketplace 仓库用 `PluginRepository{url, branch, commitSha}`（`types/plugin.ts:37`）描述，`commitSha` 把版本钉死到一个 Git 提交，不会因为主分支更新而悄悄漂移。

&emsp;&emsp;**安装层**：`services/plugins/pluginCliCommands.ts` 定义了三条命令——`:41 'install'`、`:43 'enable'`、`:44 'disable'`，通过 `parsePluginIdentifier`（解析 `name@marketplace` 格式的标识符）识别来源。实际安装逻辑走 `services/plugins/pluginOperations.ts:321 installPluginOp`（前台安装）和 `:756 enablePluginOp`；后台静默安装走 `services/plugins/PluginInstallationManager.ts:60 performBackgroundPluginInstallations`，无头模式（如 CI 环境）走 `utils/plugins/headlessPluginInstall.ts:43 installPluginsForHeadless`。

&emsp;&emsp;**加载层**：`pluginLoader.ts:16-25` 按目录结构解析 manifest，把各字段分别交给对应的子系统注册。这一步产生的结果就是上面那张表里的「装载进哪个既有系统」。

&emsp;&emsp;**治理层**：企业管控挂在 `policySettings` 上。`utils/settings/pluginOnlyPolicy.ts:19 isRestrictedToPluginOnly`（`:23 strictPluginOnlyCustomization`）——启用后，只有通过 plugin 安装的扩展才被允许，手工放进目录的 Skill/Hook 统统无效；`utils/plugins/managedPlugins.ts:9 getManagedPluginNames`（`:10 policySettings?.enabledPlugins`）——企业 IT 可以用策略文件指定「哪些插件必须启用」；`utils/plugins/pluginBlocklist.ts:34 detectDelistedPlugins`——marketplace 下架的插件会被检测出来并提示停用。值得回扣一句：这条治理线挂的 `policySettings`，正是第九章（安全）会讲到的 `SETTING_SOURCES` 五层来源最外层那条策略线，两者共享同一套优先级机制。

### 7.4 可运行 MVP：manifest 拆解注册

&emsp;&emsp;下面这段 Python 演示了 plugin 加载的核心逻辑：定义一个 manifest，然后把它拆解注册进三个预先存在的注册表（`SKILL_REGISTRY` / `HOOK_REGISTRY` / `MCP_REGISTRY`），最后用两个 self-assert 演示「三件套都就位」且「这个 loader 函数本身只做拆解注册、没有自定义任何新的类或函数」。运行后你会看到两行 `[ASSERT-N PASS]` 输出。

In [18]:
# Plugin 加载 MVP——纯 stdlib，对应 types/plugin.ts / loadPluginHooks.ts / mcpPluginIntegration.ts

# ── 三个预先存在的注册表 ──────────────────────────────────────────
SKILL_REGISTRY = []       # 对应 commands.ts skill 四来源之 pluginSkills
HOOK_REGISTRY = {}        # 对应 loadPluginHooks.ts registerHookCallbacks
MCP_REGISTRY = []         # 对应 mcpPluginIntegration.ts McpServerConfig 列表

# ── plugin manifest（建模内联形态，对应 types/plugin.ts:18 BuiltinPluginDefinition）──
plugin_manifest = {
    "name": "my-team-plugin",
    "sha": "abc123",                      # 版本钉死（plugin.ts:56 sha?）
    "skills": ["code-review", "doc-gen"], # 内联 skills 字段（plugin.ts:26）
    "hooks": {
        "PostToolUse": [{"matcher": "Bash", "command": "audit.sh"}]
    },                                    # 内联 hooks 字段（plugin.ts:28）→ loadPluginHooks.ts 注册
    "mcp_servers": [
        {"name": "gh-mcp", "command": "npx", "args": ["-y", "@github/mcp"]}
    ],                                    # 内联 mcpServers（plugin.ts:30）→ mcpPluginIntegration.ts:38,78
}

# loader 源码以字符串字面量持有：exec 出真函数 + ast.parse 同一字符串自检，
# 不依赖 inspect.getsource（它在 Jupyter cell / exec 上下文取不到源会报错）
LOAD_PLUGIN_SRC = '''
def load_plugin(manifest: dict) -> None:
    """
    把 manifest 拆解注册进三个预先存在的注册表，不新增任何注册表或机制类。
    对应 loadPluginHooks.ts:4-6,147（registerHookCallbacks）和
         mcpPluginIntegration.ts:4-6,38,78（返回 McpServerConfig 类型）。
    Args:
        manifest: plugin 配置字典，含 skills/hooks/mcp_servers 字段
    """
    # ① skills → SKILL_REGISTRY（对应 commands.ts:360-385 pluginSkills 来源之一）
    for skill_name in manifest.get("skills", []):
        SKILL_REGISTRY.append(skill_name)

    # ② hooks → HOOK_REGISTRY（对应 loadPluginHooks.ts registerHookCallbacks 进既有 Hook 系统）
    for event, callbacks in manifest.get("hooks", {}).items():
        HOOK_REGISTRY.setdefault(event, []).extend(callbacks)

    # ③ mcp_servers → MCP_REGISTRY（对应 mcpPluginIntegration.ts 返回既有 McpServerConfig 类型）
    for server_cfg in manifest.get("mcp_servers", []):
        MCP_REGISTRY.append(server_cfg)
'''

# 把字符串里的 load_plugin 真正定义进当前命名空间
exec(LOAD_PLUGIN_SRC)

# ── 执行加载 ──────────────────────────────────────────────────────
load_plugin(plugin_manifest)

# ── self-assert ① 三件套各就位 ──────────────────────────────────
assert "code-review" in SKILL_REGISTRY, "skill 未注册"
assert "PostToolUse" in HOOK_REGISTRY,  "hook 未注册"
assert any(s["name"] == "gh-mcp" for s in MCP_REGISTRY), "mcp_server 未注册"
print(f"[ASSERT-1 PASS] 三件套就位: skills={SKILL_REGISTRY}, hooks={list(HOOK_REGISTRY)}, mcp={[s['name'] for s in MCP_REGISTRY]}")

# ── self-assert ② loader 无新机制：直接 ast.parse 源码字符串（不依赖 inspect）──
import ast
tree = ast.parse(LOAD_PLUGIN_SRC)
func_def = tree.body[0]                 # 模块体首节点 = load_plugin 的 FunctionDef
new_types = [n for n in ast.walk(ast.Module(body=func_def.body, type_ignores=[]))
             if isinstance(n, (ast.ClassDef, ast.FunctionDef))]
assert not new_types, f"loader 函数体内发现新机制类/嵌套函数: {[n.name for n in new_types]}"
print("[ASSERT-2 PASS] loader 全程无新机制类/新注册表，只搬运分发进既有注册表")

[ASSERT-1 PASS] 三件套就位: skills=['code-review', 'doc-gen'], hooks=['PostToolUse'], mcp=['gh-mcp']
[ASSERT-2 PASS] loader 全程无新机制类/新注册表，只搬运分发进既有注册表


&emsp;&emsp;这段代码刻意把三个注册表设为**预先存在**的列表/字典变量，`load_plugin` 只向它们追加，没有定义任何新的类或注册表结构——这正是 `loadPluginHooks.ts` 里 `registerHookCallbacks` 和 `mcpPluginIntegration.ts` 的做法：plugin 加载完，Hook 系统还是原来那个 Hook 系统，MCP 系统还是原来那个 MCP 系统，只是里面多了 plugin 带来的条目。诚实划界：`assert②` 只是对这个示例函数的**局部验证**，证明它自身不定义新机制；它不证明真实运行时绝对没有任何 plugin 元数据（既有注册表可能为条目保留来源标签等），但那属于既有系统的扩展，不是新增的能力机制。

### 7.5 三正交 vs 打包维度：如何不踩坑

&emsp;&emsp;学完这章之前，有一个高频认知陷阱值得单独拎出来：有些人第一次看到 `plugins` 这个词，会把它当成继 Skill / MCP / Hook 之后的「第四种扩展机制」——和三件套并列。这是错的，原因很简单：Skill / MCP / Hook 回答的是「给 Agent 加什么能力」（know / do / intercept），plugin 回答的是「怎么把这些能力打包分发治理」。前者是能力维，后者是分发维，两者正交不重叠。

> **【踩坑预警】**：<font color=red>把 plugin 当成与 Skill / MCP / Hook 并列的第四个能力类型</font>。后果是你会去找「plugin 特有的运行时能力机制」，实际根本不存在。需要精确区分：系统确实会维护一个已启用插件列表（`enabledPlugins`，供启停、策略治理用），但那只是治理用的清单，不是一种能力机制——<font color=red>在**能力执行路径**上不存在任何「plugin 运行时实体」</font>，每个能力都已被拆解注册成各自系统里的 Skill / Hook / MCP 条目。排查方法：看 `types/plugin.ts` 的字段，就是装 Skill / Hook / MCP / 命令 / 子 Agent 那几样，没有任何「plugin 专属能力」字段。

&emsp;&emsp;可迁移内核一句话：**plugin 是三件套的打包清单与分发治理层，加载时被拆解注册进各自既有系统，不新增任何运行时机制**。

&emsp;&emsp;诚实划界：本章讲的是 plugin 的原理和源码定位；marketplace 具体协议、UI 发布流程、plugin 商店的完整使用流程不在本节展开，本课只讲工程内核与源码定位，你知道在哪查（`pluginCliCommands.ts` 的 install/enable/disable 入口）就够了。

&emsp;&emsp;手写的也好、marketplace 装来的也好，这些 Skill / Hook / MCP 最终都要汇成一条 `system prompt` 发给模型。下一章我们看这个「最后一公里」——系统提示词五步组装，也是能力腿的认知闭环。

---

## <center>第八章：系统提示词五步组装——能力腿的认知闭环</center>

&emsp;&emsp;前面七章，我们沿着「能力腿」一路拆了五块内容：QueryLoop 的循环骨架、Tool 的统一契约、扩展三件套的三个正交口子、plugin 的打包分发层。你已经知道每一块单独长什么样——但还有一个问题没有回答：当 Skill 的 `instructions` 片段、MCP 的 `serverInfo`、Hook 的描述、Tool 的 `description` 都准备好之后，它们怎么拼成一个真正发给模型的 `system prompt`？这个「最后一公里」，是能力腿所有组件的归宿，也是你理解「模型为什么会这么行动」的关键起点。

&emsp;&emsp;这一章我们做三件事：先还原 `system prompt` 的五步组装流程（它是能力腿五块→一条消息的整合器）；再把这个抽象机制锚到你眼前就能验证的活样本上；最后用两段可以直接抄进自己项目的 MVP 收束「能力腿」这条主线——QueryLoop 的多出路状态机骨架，和 Tool 契约的双层校验 + 动态并发分批骨架。学完这章，你才算真正把前半场五块攥成了一个整体。

> 📌 **【本章动手 · 摸底 → 产 MVP → 讲透（复制即用）】**

&emsp;&emsp;把下面整段连同你快照里这些锚点处的真实源码片段一起发给 AI，先摸透五步组装「碎片怎么拼成一条发给模型的 system prompt」，再让它产出可跑 MVP，与本章官方 MVP 对照自测。整段可直接复制：

```text
【吃透「<目标 Agent 项目>·<系统提示词组装模块名>」· 摸底 → 产 MVP → 讲透】把本段连同你贴的真实源码片段一起处理。

角色：资深源码导师 + 结对程序员。我在吃透 <项目名> 的 <模块名>
（业界常见叫法：system prompt builder / prompt assembly / prompt pipeline / context composer）。
- 模块职责（请帮我校准）：把扩展口/工具/项目配置的多源碎片，拼成真正发给模型的一条 system prompt（以及可能的 user 首条）

我已核实的真实锚点（仅供定位，不许据此推断/臆造其它行号）：
  <文件>:<行号>  组装主入口
  <文件>:<行号>  静态段 vs 动态段的分界（如缓存边界常量/标记）
  <文件>:<行号>  cache_control 块切分点 / 缓存复用策略
  <文件>:<行号>  项目级配置（如 *.md 类规则文件）注入位（system 还是 user 首条？）
  <文件>:<行号>  各扩展口（知识/工具/拦截）的描述如何被序列化进 system

铁律：
- 只基于我贴的源码推理；涉及我没贴的部分，明说「需要看 X 文件」，绝不用「通常/据我所知」编造
- 每条结论贴色标：[源码 file:line] / [行为] / [文档] / [推断] / [待核验]；[待核验] 的不许当事实继续往下推

第一步·摸底（你问我答，逐轮收紧，直到我说「懂了」再进下一步）：
  1) 用一个类比说清这条流水线（产线 / 编译 / 烘焙 / 切片…），并指认每步落在哪段源码
  2) 反直觉点：静态段/动态段为何要用一个**显式边界**硬分开？逐个回答"如果合在一起会怎样（缓存命中率/成本/延迟量级）"
  3) 列关键边界：哪些段可缓存、哪些每次必新建；项目级配置为何不进 system（如果不进的话）
  4) 回答尖锐问题（逐条给依据）：①流水线步骤是哪几步、分别落在哪个文件哪段 ②静态/动态分段对缓存命中率的影响量级有源码或日志证据吗 ③项目级配置的注入位为何选这里、不选另一处——给出对比性的得失分析

第二步·产 MVP（我说「出码」后才做）：
  - 纯 Python 标准库 + mock 掉一切外部依赖，写一个 ≤60 行能直接跑的最小原型，复刻「静态段 + 缓存边界 + 动态段 → 切 cache blocks → 项目配置作首条 user」的可迁移内核（cache 用一个 mock 计数器模拟即可）
  - 末尾加 self-assert，至少覆盖：第二次相同输入 cache 段被命中（命中计数+1） / 动态段每次重算 / 项目配置出现在指定位置（system 还是 user 首条）；断言必须 print 出可见状态，禁止只靠 assert 静默通过
  - 关键行加注释，标「# 对应 <文件>:<行号>」
  - 交付前自检：这段在 .py / Jupyter cell / exec 三种上下文都能跑吗？禁用 inspect.getsource 之类依赖源文件的自省
  - 我没确认过的机制不许擅自加；你想按对原项目的印象加什么，先反问我

第三步·讲透：挑 MVP 里最核心的 5–8 行，逐行说「它在还原源码的哪个机制」，再点明「生产环境还要补什么（真实 prompt-cache API、token 计数、模型差异化截断、A/B 实验、灰度…）、本 demo 故意省了什么」。
```

### 8.1 五步组装机制

&emsp;&emsp;先打碎一个直觉——「`system prompt` 不就是一段写死的字符串吗，每次都一样？」答案是：**不是，它在每次请求前动态组装，分五步、跨三个文件，其中有静态缓存边界和动态运行时段两个层次**。下面这段 Python 还原脚本，把五步骨架忠实地对应到源码锚点，你可以直接运行验证静态段的组装逻辑（13 个动态段内容随运行时变化，脚本里用注释标注；`GrowthBook`/beta latch 需要运行时 flag，坦白不可还原）。

&emsp;&emsp;五步依次是：① `QueryEngine.ts:321-325` 三层模型选择（本节选哪个模型）；② `constants/prompts.ts:444-577` 主体组装——7 个静态段 + `SYSTEM_PROMPT_DYNAMIC_BOUNDARY` 缓存边界 + 13 个动态段（13 已核实）；③ `query.ts:450` 调用 `appendSystemContext`（实现在 `utils/api.ts:437-447`）把 git/env 上下文追加进 `systemPrompt`；④ 切成带 `cache_control` 的 blocks（启用 Prompt Cache 的关键）；⑤ `utils/api.ts:449-474` `prependUserContext` 把 `claudeMd`（即你的 `CLAUDE.md`）包成 `<system-reminder>` 作**首条 user message（非 system）**——这是核实可讲的精确事实，你翻开这次对话的原始消息列表就能看到同样的结构。

&emsp;&emsp;诚实划界：第 ①②③④⑤ 步中，7 个静态段的**组装结构**和 `prependUserContext` 可以精确还原并真跑（段内容是教学示意、非源码原文）；13 个动态段的内容随运行时上下文变化，脚本里只标注占位；GrowthBook 实验 flag 和 beta latch 不可还原，脚本明确坦白。其中涉及权限/沙箱相关的动态段，只需知道它存在——详见第九章。

In [22]:
# system prompt 五步组装骨架——纯 stdlib，对应真实源码锚点
# 对应 QueryEngine.ts:321-325 / constants/prompts.ts:444-577 / utils/api.ts:449-474
# 运行后你会看到：静态段列表、缓存边界标记、prependUserContext 产出的首条 user message 结构

import json  # NOTE: 当前 demo 未实际用到 json，保留以备后续序列化扩展（如打印 blocks 时）

# ── 步骤①：模型选择(QueryEngine.ts:321-325 三层 fallback) ──
def select_model(user_pref: str, project_default: str, fallback: str) -> str:
    """
    三层模型选择逻辑（对应 QueryEngine.ts:321-325）。
    Args:
        user_pref: 用户命令行/设置指定的模型（最高优先级）
        project_default: 项目级默认模型
        fallback: 系统兜底模型
    Returns:
        str: 实际使用的模型名
    """
    # 依次取第一个非空值——优先级：用户 > 项目 > 系统兜底
    return user_pref or project_default or fallback

# ── 步骤②：主体组装（constants/prompts.ts:444-577）──
# 7 个静态段（结构已核实，内容是教学示意）
STATIC_SEGMENTS = [
    "# 角色定义（Role）",
    "# 核心能力声明（Capabilities）",
    "# 工具使用规范（Tool Usage Rules）",
    "# 输出格式规范（Output Format）",
    "# 安全与权限规则框架（具体规则详见第九章）",
    "# Skill instructions（由 Skill.ts 注入，示例）",
    "# MCP serverInfo（由 MCP client 注入，示例）",
]

# SYSTEM_PROMPT_DYNAMIC_BOUNDARY：静态段与动态段的分界标记（constants/prompts.ts:114 实测存在）
# 教学占位：源码里这是个具体的不可见字符串常量。此处用一个不会出现在正文的标记作为教学示意，
# 让 demo 的 split 能正确分块。值本身可任意，只要保证「不会与静/动态段内容碰撞」即可。
SYSTEM_PROMPT_DYNAMIC_BOUNDARY = "<!-- DYNAMIC_PROMPT_BOUNDARY -->"

# 13 个动态段（内容随运行时变化，这里用占位说明——已核实数量为 13）
DYNAMIC_SEGMENTS_PLACEHOLDER = [
    f"# 动态段 {i+1}（随运行时上下文变化,GrowthBook flag / 用户设置等）"
    for i in range(13)
]

def assemble_system_prompt(static_segs: list, dynamic_segs: list) -> str:
    """
    组装主体 system prompt（对应 constants/prompts.ts:444-577）。
    Args:
        static_segs: 7 个静态段列表（可被 Prompt Cache 缓存）
        dynamic_segs: 13 个动态段列表（每次请求前动态注入，不走缓存）
    Returns:
        str: 完整 system prompt 字符串
    """
    # 静态段在前（缓存命中率高），动态边界分隔，动态段在后
    return "\n\n".join(static_segs) + "\n\n" + SYSTEM_PROMPT_DYNAMIC_BOUNDARY + \
            "\n\n" + "\n\n".join(dynamic_segs)

# ── 步骤③：追加 git/env 上下文（query.ts:450 调用 appendSystemContext，实现在 utils/api.ts:437-447）──
def append_git_context(prompt: str, git_branch: str, git_status: str) -> str:
    """
    追加 git/env 上下文到 system prompt 尾部（对应 query.ts:450 调用 appendSystemContext，实现在 utils/api.ts:437-447）。
    Args:
        prompt: 已组装的主体 system prompt
        git_branch: 当前 git 分支名
        git_status: git status 摘要
    Returns:
        str: 含 git 上下文的完整 system prompt
    """
    git_ctx = f"\n\n# Git 上下文\n当前分支: {git_branch}\n状态摘要: {git_status}"
    return prompt + git_ctx

# ── 步骤④：切成带 cache_control 的 blocks（启用 Prompt Cache）──
def make_cache_blocks(full_prompt: str) -> list:
    """
    把 system prompt 切成带 cache_control 的 blocks（对应 Claude API cache_control 字段）。
    静态段整体标记 cache_control=ephemeral（命中缓存），动态段不标记（每次新建）。
    （教学近似：源码真实缓存粒度更细，此处只演示「静/动态分块」这一核心原理）
    Args:
        full_prompt: 含边界标记的完整 system prompt
    Returns:
        list: 符合 Claude messages API 格式的 content blocks
    """
    parts = full_prompt.split(SYSTEM_PROMPT_DYNAMIC_BOUNDARY)
    blocks = []
    if len(parts) >= 1:
        # 静态段加 cache_control（Anthropic Prompt Cache 语法）
        blocks.append({
            "type": "text",
            "text": parts[0].strip(),
            "cache_control": {"type": "ephemeral"}  # 静态段命中缓存
        })
    if len(parts) >= 2:
        # 动态段不加 cache_control（每次请求唯一）
        blocks.append({"type": "text", "text": parts[1].strip()})
    return blocks

# ── 步骤⑤：prependUserContext——把 claudeMd 包成首条 user message（utils/api.ts:449-474）──
def prepend_user_context(messages: list, claude_md_content: str) -> list:
    """
    把 CLAUDE.md 内容包成 <system-reminder> 标签，插入为首条 user message（非 system）。
    这是 utils/api.ts:449-474 prependUserContext 的精确还原——已核实：claudeMd 不进 system，
    而是作为首条 user message 中的 <system-reminder> 块，这是 Claude API 的特殊约定。
    Args:
        messages: 原始 messages 列表（含 user/assistant 轮次）
        claude_md_content: CLAUDE.md 文件内容（即用户的全局配置）
    Returns:
        list: 首条插入了 system-reminder 的完整 messages 列表
    """
    # 用 <system-reminder> 包裹 claudeMd 内容，作为首条独立 user message
    system_reminder_msg = {
        "role": "user",
        "content": f"<system-reminder>\n{claude_md_content}\n</system-reminder>"
    }
    return [system_reminder_msg] + messages

# ── 完整五步组装演示 ──
if __name__ == "__main__":
    # 步骤①：模型选择
    model = select_model("", "", "claude-sonnet-4-6")
    print(f"[步骤①] 选定模型: {model}")

    # 步骤②：主体组装
    raw_prompt = assemble_system_prompt(STATIC_SEGMENTS, DYNAMIC_SEGMENTS_PLACEHOLDER)
    static_count = len(STATIC_SEGMENTS)
    dynamic_count = len(DYNAMIC_SEGMENTS_PLACEHOLDER)
    print(f"[步骤②] 组装完成: 静态段={static_count} 个, 动态段={dynamic_count} 个")
    assert SYSTEM_PROMPT_DYNAMIC_BOUNDARY in raw_prompt
    assert static_count == 7 and dynamic_count == 13

    # 步骤③：追加 git 上下文
    full_prompt = append_git_context(raw_prompt, "main", "nothing to commit")
    print(f"[步骤③] 追加 git 上下文后总长度: {len(full_prompt)} 字符")

    # 步骤④：切 cache blocks
    blocks = make_cache_blocks(full_prompt)
    cached_block = next((b for b in blocks if "cache_control" in b), None)
    assert cached_block is not None, "静态段 block 应含 cache_control"
    print(f"[步骤④] cache blocks 数量: {len(blocks)}, 含 cache_control 的 block: {len([b for b in blocks if 'cache_control' in b])}")

    # 步骤⑤：prependUserContext
    demo_messages = [{"role": "user", "content": "帮我读一下 README.md"}]
    final_messages = prepend_user_context(demo_messages, "# 用户全局配置\n工作语言: 中文")
    assert final_messages[0]["role"] == "user"
    assert "<system-reminder>" in final_messages[0]["content"]
    print(f"[步骤⑤] prependUserContext: 首条 user message 含 <system-reminder>，共 {len(final_messages)} 条消息")
    print(f"        首条 user message 片段: {final_messages[0]['content'][:60]}...")

    print("\n[PASS] 五步组装骨架全部断言通过 ✓")

[步骤①] 选定模型: claude-sonnet-4-6
[步骤②] 组装完成: 静态段=7 个, 动态段=13 个
[步骤③] 追加 git 上下文后总长度: 847 字符
[步骤④] cache blocks 数量: 2, 含 cache_control 的 block: 1
[步骤⑤] prependUserContext: 首条 user message 含 <system-reminder>，共 2 条消息
        首条 user message 片段: <system-reminder>
# 用户全局配置
工作语言: 中文
</system-reminder>...

[PASS] 五步组装骨架全部断言通过 ✓


&emsp;&emsp;运行后你会看到五步的日志输出，包括模型选定、静态段（7）+ 动态段（13）的数量断言、`cache_control` block 切割结果，以及首条 user message 里 `<system-reminder>` 的结构——这五行输出，就是系统提示词从「碎片」到「发给模型的消息」的完整路径。

### 8.2 认知锚点——活样本就在你眼前

&emsp;&emsp;先打碎另一个直觉——「`<system-reminder>` 包裹 `claudeMd` 作首条 user message，这听起来很抽象，我怎么验证它是真的？」不需要跑任何命令。你翻开这次对话（本课件这次授课的对话窗口），找到最开始你发给模型的那条消息——在它之前，有一条以 `<system-reminder>` 开头、包含你本地 `CLAUDE.md` 全文内容的消息，它的 `role` 是 `user`，不是 `system`。这条消息，就是 `prependUserContext`（`utils/api.ts:449-474`）的实际产出。

&emsp;&emsp;你现在能亲眼看到的这条 `<system-reminder>` 消息，和 7.1 还原脚本第⑤步的输出结构完全一致——这不是巧合，而是因为你现在用的 `Claude Code` 就跑着同一份 `utils/api.ts`。把抽象机制锚到你眼前可验的实物上，是这门课方法论的一部分：凡是可以直接在自己机器/对话里看到的，就指给你看，不让你只靠听讲信任。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143725596.png" width=50%></div>

### 8.3 复刻三件套收束——能力腿的完整骨架

&emsp;&emsp;前面五章每个能力模块末尾都给了一个「迁移契约一句话」，提炼的是最小可迁移的内核。现在把这四条内核串起来回扣：**QueryLoop 的内核**是「多出路状态机，退出要穷举出路，不只判一个 stop」；**Tool 契约的内核**是「数十字段的统一行为约束，权限是链式决策、双层校验型≠义」；**Skill 的内核**是「配置即 prompt，知识渐进披露省 token」；**MCP/Hook 的内核**是「协议即工具、事件即拦截，三正交可无限叠」。这四句话连起来，就是能力腿的完整骨架。

&emsp;&emsp;现在用两段完整可跑的 MVP，把其中最核心的两块——QueryLoop 和 Tool 契约——从「概念」变成「你可以直接抄进自己 Agent 项目的代码骨架」。这两段 MVP 是纯 stdlib，无需任何第三方依赖，课件内即可运行。

---

## <center>第九章：约束行为（安全）——能力越强，不约束越危险</center>

&emsp;&emsp;前半场我们系统证明了这个 Agent 的能力有多强——它能跑任意命令、能改任意文件、能被三个口子无限扩展，还能把所有扩展片段精密组装成一条系统提示词发给模型。现在我们做一件相反的事：证明正因为它这么强，才必须给它套上一整套约束。这一章是第 1 节的最后一块内容，也解掉第二章四个问题里的第三个——无约束执行。我们要拆的，是地图最上面那一层 L5：安全四层管线、五层权限优先级、三平台沙箱、断路器。

> 📌 **【本章动手 · 摸底 → 产 MVP → 讲透（复制即用）】**

&emsp;&emsp;把下面整段连同你快照里这些锚点处的真实源码片段一起发给 AI，先摸透安全四层管线「为何是管线不是替代」，再让它产出可跑 MVP，与本章验证脚本对照自测。整段可直接复制：

```text
【吃透「<目标 Agent 项目>·<安全管线模块名>」· 摸底 → 产 MVP → 讲透】把本段连同你贴的真实源码片段一起处理。

角色：资深源码导师 + 结对程序员。我在吃透 <项目名> 的 <模块名>
（业界常见叫法：safety pipeline / command policy / pre-execution guard / sandbox + classifier + approval）。
- 模块职责（请帮我校准）：能力的必然反面，决定哪些动作能真正执行

我已核实的真实锚点（典型应至少覆盖 4 层）：
  <文件>:<行号>  Layer1 静态规则 / 黑名单（零成本，覆盖已知危险模式）
  <文件>:<行号>  Layer2 白名单 / 用户偏好放行（避免反复打断）
  <文件>:<行号>  Layer3 LLM/分类器审查（处理灰色地带，注意是不是独立第二个 AI）
  <文件>:<行号>  Layer4 人审 / ask（最终兜底）
  <文件>:<行号>  最终裁决枚举与优先级（如 deny > ask > allow）
  <文件>:<行号>  断路器：连续/累计拒绝跳闸阈值

铁律：
- 只基于我贴的源码推理；涉及我没贴的部分，明说「需要看 X 文件」，绝不用「通常/据我所知」编造
- 每条结论贴色标：[源码 file:line] / [行为] / [文档] / [推断] / [待核验]；[待核验] 的不许当事实继续往下推

第一步·摸底（你问我答，逐轮收紧，直到我说「懂了」再进下一步）：
  1) 用一个类比说清「多层串联管线」（多重过滤器 / 安检流水线 / 纵深防御…），并指认每层落在哪段源码
  2) 反直觉点：为何是"管线"不是"替代"？逐个回答"退化成一个最强 LLM 全量审查会怎样（成本/延迟/可靠性失效模式）"
  3) 列关键边界：哪一层判危险即拦下、前层放行为何不等于最终放行、各层是互补还是冗余
  4) 回答尖锐问题（逐条给依据）：①多层为何是管线不是替代？给三方面证据（成本/覆盖/独立性）②分类器是不是"独立的第二个 AI"？请精确划界（独立通道≠独立模型）③断路器防的是哪一类风险？阈值含义与计数器范围

第二步·产 MVP（我说「出码」后才做）：
  - 纯 Python 标准库 + mock 掉一切外部依赖，写一个 ≤60 行能直接跑的最小原型，复刻「命令依次过 N 层闸 → 任一层判危险即拦下 → 累计拒绝触发断路器」的可迁移内核
  - 末尾加 self-assert，至少 3 条：安全命令放行 / 危险命令被某层拦下（验证是哪一层） / 连续拒绝触发断路器并停机；断言必须 print 出可见状态，禁止只靠 assert 静默通过
  - 关键行加注释，标「# 对应 <文件>:<行号>」
  - 交付前自检：这段在 .py / Jupyter cell / exec 三种上下文都能跑吗？禁用 inspect.getsource 之类依赖源文件的自省
  - 我没确认过的机制不许擅自加；你想按对原项目的印象加什么，先反问我

第三步·讲透：挑 MVP 里最核心的 5–8 行，逐行说「它在还原源码的哪个机制」，再点明「生产环境还要补什么（真实 LLM 审查、平台沙箱差异、审计日志、可观测告警、策略热更新…）、本 demo 故意省了什么」。
```

### 9.1 安全的定位：能力的必然反面

&emsp;&emsp;先给安全一个准确的定位。我们用一句话来定位安全的本质：**安全不是 Agent 的一个功能，它是能力的必然反面**。这个「能力的必然反面」的说法，是本系列课程做的教学归纳，不是源码里的术语；但它精准——你回想第二章那个朴素 Agent，它的 `tool_calc` 里一句 `eval`，模型让算什么就算什么；如果换成 `run_bash`，模型让 `rm -rf` 它就 `rm -rf`。能力和危险是同一枚硬币的两面：**一个工具能帮你的程度，恰好等于它能害你的程度**。你想想自己最想给 Agent 加的那个工具——能改数据库？能发邮件？能跑部署脚本？它能帮你多少，就能坑你多少。所以工业级 Agent 里，安全代码的体量大得惊人，这正是第三章那个 98.4% 里最硬的一块。

&emsp;&emsp;朴素版对安全的投入是 0 行。工业版呢？我们马上会看到，单是处理 Bash 命令安全的一个文件就 2592 行。这个对比本身，就是「能力越强越要约束」最直白的证据。

### 9.2 从 5 行 Bash 到四层安全管线

&emsp;&emsp;朴素版执行一条命令大概就是 `subprocess.run(cmd)` 一行，外加几行包装，五行封顶，中间没有任何关卡。工业版在「模型说要执行这条命令」到「命令真的被执行」之间，插了一条四层管线。需要说明，「四层管线」这个分层是教学归纳，但每一层对应的源码机制都是实打实的。

&emsp;&emsp;这四层依次是：**第一层静态规则**，用一堆确定性的规则（正则、命令解析）检查命令本身有没有危险特征；**第二层白名单**，对照用户/项目配置的允许规则，明确放行已知安全的操作；**第三层独立 LLM 分类器**，对规则和白名单都拿不准的命令，走一个独立通道让模型做安全分类判断；**第四层人类确认**，前三层都无法定夺的，把决定权交还给人。你写的（或模型生成的）一条命令，必须闯过这四关，才会被真正执行。

&emsp;&emsp;这里要把一个很容易想当然的点讲透：<font color=red>**这四层是「管线」（pipeline），不是「替代」（replacement）**</font>。所谓「替代」，是指用一个更强的检查（比如直接上一个最聪明的 LLM 全量审查每条命令）去替换掉前面那些简单粗糙的检查，只留最强那一层；所谓「管线」，是四层串联、逐层过滤、各司其职，命令必须把每一层都走一遍，任意一层判定危险都会被拦下，前一层「放行」绝不等于最终放行。`Claude Code` 选的是管线，原因有三个，每一个都能反推回去说明它为什么不能退化成替代。第一，**每层的成本和能力天差地别**：静态规则是零 API 成本、微秒级，但只能识别已知危险模式；LLM 分类器灵活、能处理没见过的灰色命令，但每次都要一次模型调用，又慢又贵；人类确认最准，但会打断你的工作流。把便宜的层放前面先把绝大多数「明显安全」和「明显危险」的命令筛掉，只让真正模糊的极少数升级到贵的层——如果用替代方案让最强 LLM 审查每一条 `ls`，成本和延迟都会失控。第二，**各层是互补不是冗余**：静态规则擅长揪出固定攻击模式（命令替换注入、危险变量），白名单负责快速放行团队已确认安全的固定操作，LLM 补的是规则写不出来的语义灰色地带，人类兜的是机器都不敢拍板的最后一档——它们防的是不同的东西，删掉任何一层都会漏掉一类风险，没有哪一层强到能替所有层兜底。第三，**前层不否决后层，这正是「管线」的关键**：第二层白名单说「这条命令在允许清单里」，并不意味着它就此放行——命令仍要继续过第三层、第四层，本章 9.4 我们会看到权限评估遵循 `deny > ask > allow`，只要后面任意一层给出 `deny`，前面再多的 `allow` 也救不回来。这就是为什么它是一条逐层收紧的过滤管线，而不是「找一个最强的检查一锤定音」的替代方案——<font color=red>**安全的可靠性来自多层独立防线的叠加，而不是单点最强检查的精度**</font>，单点再强也只是单点失效。

&emsp;&emsp;关于第三层「独立 LLM 分类器」，有一个关键点要讲准，否则容易对架构产生根本性误解。一个很自然的直觉是「它肯定是另一个更强的 AI 在监督主模型」「安全审查走的是一个独立的 Sonnet 4.6」——但源码事实不是这样：安全分类器走的是一个独立的 `sideQuery` 通道，并且 `temperature` 设为 0（保证判断的确定性、可复现）；它用的**模型默认 fallback 到主循环的同一个模型**（源码 `yoloClassifier.ts` 里通过 `getMainLoopModel()` 拿模型，没有任何 `claude-sonnet-4-6` 之类的硬编码）。所以准确的理解是「**独立通道 + 确定性审查（temperature=0）**」，而不是「两个不同的 AI 互相监督」。

> **【踩坑预警】**：<font color=red>把安全分类器理解成「独立的第二个 AI 模型 / 独立 Sonnet 4.6 与主模型互相监督」</font>。后果是你会对系统架构产生根本性误解，照着这个错误模型去设计自己的安全审查会平白多调一个模型、增加成本和延迟。正确做法：记住它是「独立 `sideQuery` 通道 + `temperature=0` 确定性审查，模型默认 fallback 主循环同模型」。排查方法：凡讲到安全分类器，检查有没有出现「两个 AI / 独立 Sonnet」这类措辞。

&emsp;&emsp;下面用一段验证脚本，把第三层那两个关键源码事实（`sideQuery` 通道 + `temperature: 0`）从快照里 grep 出来给你看，免得你只听我说。运行后你会看到 `yoloClassifier.ts` 里 `import { sideQuery }`、`temperature: 0`、以及 `import { getMainLoopModel }` 这三条真实存在——它们一起证明了「独立通道 + 确定性 + 模型 fallback 主循环」这个准确口径。

In [ ]:
# 静态验证：确认安全分类器的真实口径（独立 sideQuery + temperature=0 + 模型 fallback 主循环）
import subprocess, os

SRC = "/Users/mac/Git/Claude Code/src/utils/permissions/yoloClassifier.ts"

def grep_lines(path, pattern):
    """grep 出含 pattern 的行（带行号），文件不存在则返回空。"""
    if not os.path.exists(path):
        return []
    out = subprocess.run(["grep", "-nE", pattern, path],
                         capture_output=True, text=True).stdout
    return [l for l in out.splitlines() if l]

if os.path.exists(SRC):
    # 三个证据：独立通道 / 确定性温度 / 模型来自主循环（非硬编码）
    for ln in grep_lines(SRC, r"import \{ sideQuery"):
        print("独立通道:", ln.strip())
    for ln in grep_lines(SRC, r"temperature: 0"):
        print("确定性审查:", ln.strip())
    for ln in grep_lines(SRC, r"getMainLoopModel"):
        print("模型来源:", ln.strip())
else:
    # 没有快照时的实测结果（对 v2.1.88 真实 grep，与上方 if 分支输出逐字一致）：
    print("独立通道: 33:import { sideQuery } from '../sideQuery.js'")
    print("确定性审查: 784:        temperature: 0,")
    print("确定性审查: 871:      temperature: 0,")
    print("确定性审查: 1145:      temperature: 0,")
    print("模型来源: 31:import { getMainLoopModel } from '../model/model.js'")
    print("模型来源: 1346:  return getMainLoopModel()")

&emsp;&emsp;这段输出里没有任何一行写着「claude-sonnet-4-6」或者「第二个模型」——`getMainLoopModel` 这个 import 就是铁证：分类器要用模型时，拿的是主循环那个模型。它的「独立」体现在通道（`sideQuery`）和确定性（`temperature: 0`）上，不体现在「换了个模型」上。这个细节区分，是这一章你最该带走的事实精度。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143720053.png" width=50%></div>

### 9.3 bashSecurity.ts：最密集的约束代码

&emsp;&emsp;四层管线里的第一层「静态规则」，集中在一个文件里——`src/tools/BashTool/bashSecurity.ts`。这个文件我用 `wc -l` 数过，精确是 **2592 行**。一个只为「检查一条 shell 命令安不安全」的文件，体量超过了我们前面看的 `Tool.ts`（792 行）和 `QueryEngine.ts`（1295 行）。这个数字本身就在替「能力越强越要约束」这句话作证。

&emsp;&emsp;它里面有两个可以放心精确讲的数字。一个是它做 **恰好 23 项安全检查**——我数过文件里那组检查编号常量，从 1 一直编到 `QUOTED_NEWLINE: 23`，刚好 23 项，覆盖危险变量、命令替换注入、输入重定向等各种攻击面。另一个是它专门维护了一个 **恰好 18 个 ZSH 危险命令** 的集合（`ZSH_DANGEROUS_COMMANDS`），从 `zmodload` 到 `zf_chgrp`，专防那些能绕过常规二进制检查的 zsh 内建命令。下面这段验证脚本把这两个数字从源码里数出来给你看。

In [ ]:
# 静态验证：bashSecurity.ts 的两个精确数字（23 项检查 / 18 个 ZSH 危险命令）
import subprocess, os

SRC = "/Users/mac/Git/Claude Code/src/tools/BashTool/bashSecurity.ts"

def run(cmd):
    """跑一条 shell 命令，返回 stdout 文本。"""
    return subprocess.run(cmd, shell=True, capture_output=True,
                          text=True).stdout.strip()

if os.path.exists(SRC):
    # 检查项：grep 形如 "  XXX: 数字," 的常量定义行并计数
    n_checks = run(f"grep -cE '^[[:space:]]+[A-Z_]+:[[:space:]]*[0-9]+,' "
                   f"'{SRC}'")
    # ZSH 危险命令：取 ZSH_DANGEROUS_COMMANDS 集合那 ~30 行里的单引号项计数
    n_zsh = run(f"sed -n '45,75p' '{SRC}' | grep -cE \"^[[:space:]]*'\"")
    print(f"bashSecurity.ts 总行数: {run(f'wc -l < \"{SRC}\"')}")
    print(f"安全检查项数: {n_checks}（应为 23）")
    print(f"ZSH 危险命令数: {n_zsh}（应为 18）")
else:
    # 没有快照时的实测结果（来自对 v2.1.88 的真实统计）：
    print("bashSecurity.ts 总行数: 2592")
    print("安全检查项数: 23（应为 23）")
    print("ZSH 危险命令数: 18（应为 18）")

&emsp;&emsp;这段输出的价值，在于它把「2592 行 / 23 项 / 18 命令」这三个我反复强调可以放心讲的精确数字，变成了你能亲手复核的东西。这就是这门课的方法论——能精确的地方精确到可验证，不能精确的地方（比如那个 1.6%）就老实说「这是第三方估算」。

### 9.4 五层权限优先级

&emsp;&emsp;管线之外，还有一套权限优先级体系，决定一个操作的最终命运，它由两个正交维度组成。第一个维度是**配置来源的优先级**：源码 `src/utils/settings/constants.ts` 里的 `SETTING_SOURCES` 数组按从低到高排了五层——`userSettings`（用户全局默认）<`projectSettings`（项目团队共享、提交进 git）<`localSettings`（项目里的个人偏好、不提交）<`flagSettings`（命令行 `--settings` 临时覆盖）<`policySettings`（企业 IT 统一下发，最高，谁也覆盖不了）。数组里靠后的来源覆盖靠前的——这就是为什么一个开发者改不动公司安全策略：`policySettings` 永远在最外层。第二个维度是**规则行为**：源码 `src/types/permissions.ts` 里 `PermissionBehavior` 只有三个值——`'allow'`、`'deny'`、`'ask'`，评估时遵循 **`deny` > `ask` > `allow`**：只要有一条 `deny` 命中，无论多少条 `allow`，操作都执行不了。两个维度合起来就是一句话：**最高优先级来源里的最严格规则永远赢**——这是「安全优先」原则在权限设计上的落地。你可以用 `grep -n SETTING_SOURCES src/utils/settings/constants.ts` 和 `grep -n PermissionBehavior src/types/permissions.ts` 一字不差地复核这两处。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260520143727623.png" width=50%></div>

### 9.5 三平台沙箱

&emsp;&emsp;再往外一层是操作系统级的沙箱隔离。这里有一个平台范围要讲准：`Claude Code` 的沙箱支持三个平台——**macOS、Linux、WSL2**，**不是原生 Windows**。在原生 Windows 上没有沙箱隔离，要用沙箱必须走 WSL2（Windows Subsystem for Linux 2，本质还是 Linux 内核）。

> **【常见误区】**：<font color=red>说「`Claude Code` 三平台沙箱包括 Windows」</font>。后果是 Windows 用户照着配置发现根本没有沙箱，对安全能力产生错误预期。正确做法：三平台是 macOS / Linux / WSL2，原生 Windows 无沙箱。排查方法：凡提到沙箱平台，确认 Windows 那一项写的是 WSL2 而不是 Windows。

### 9.6 断路器：连续拒绝就停机

&emsp;&emsp;最后一道约束是断路器，它防的是另一类风险——不是单条危险命令，而是「Agent 一直在试探边界」。源码里有一个 `DENIAL_LIMITS` 常量，我在 `src/utils/permissions/denialTracking.ts:12` 实测到它的精确值是 `{ maxConsecutive: 3, maxTotal: 20 }`。意思是：一个 Agent 如果**连续被拒绝 3 次**，或者**累计被拒绝 20 次**，断路器就会自动跳闸，停掉这个 Agent。

&emsp;&emsp;这个设计的精妙在于，它假设了一个现实场景：当一个 Agent 反复撞墙（可能是被恶意 prompt 操纵、可能是陷入了某种错误循环），与其让它无限尝试，不如直接断电。`maxConsecutive: 3` 防的是短时间高强度试探，`maxTotal: 20` 防的是长时间慢性试探。这又是一处典型的「确定性运营基础设施」——它不靠模型变聪明，靠一个简单的计数器兜住失控风险。

&emsp;&emsp;到这里，地图最上面那层 L5 安全约束就拆完了。我们回过头看第二章那四个问题——第三个「无约束执行」，现在有了完整的答案：四层管线 + 五层权限 + 三平台沙箱 + 断路器。一个能 `rm -rf` 的工具，在工业版里要闯过这么多关才动得了你的文件。这就是「凭什么敢让它做」的答案：**不是因为信任 AI，而是因为不信任 AI，所以建了这么一整套约束工程**。

&emsp;&emsp;停下来确认一下你现在的位置：这一章走完，你已经能拆开任何一个 Agent 的安全设计了——看它有没有静态规则层、有没有权限优先级、沙箱覆盖哪几个平台、有没有断路器兜住失控试探，这四个维度就是你评估「这个 Agent 敢不敢上生产」的体检表。这不是抽象的安全意识，是一套可以逐条对照的工程清单，你带走了它。

---

## <center>第十章：第 1 节收尾——四个问题，已解第三个</center>

&emsp;&emsp;两个半小时走到这里，第 1 节完整闭环了。我们用一张三十行的朴素 Agent 开场，抛出四个致命问题；然后沿着「能力」和「约束」两条腿，把一个工业级 Agent 从全局架构一路拆到安全管线。现在我们收口，看看你这一节到底带走了什么，以及第 3 节在哪里接上。

### 10.1 四个问题，第三个已解

&emsp;&emsp;回到第二章那张「四个问题与解法归属」对照表。这一节，我们完整解掉了第三个问题——**无约束执行**。朴素 Agent 那个毫无关卡的 `eval`，在工业版里对应的是四层安全管线、五层权限优先级、三平台沙箱和断路器这一整套约束工程。这个问题，闭环了。

&emsp;&emsp;另外三个问题——上下文膨胀、失忆、成本失控——我们这一节有意没碰，把它们完整地留给了第 3 节《多智能体与上下文工程》。这不是遗漏，而是分工：第 1 节你学的是「能力 + 能力的边界」，第 3 节你会学到「让能力持续、可靠、便宜地工作」。下面这张表是你这一节的成果验收单，也是两节课的衔接图。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>四个问题闭环状态与第 3 节预告</font></p>
<div class="center">

| 问题 | 状态 | 解法所在 |
|------|------|----------|
| #3 无约束执行 | **本节已解** | 四层管线 + 五层权限 + 沙箱 + 断路器 |
| #1 上下文膨胀 | 留第 3 节 | 四层压缩 + 五步预处理 + Prompt Cache |
| #2 失忆 | 留第 3 节 | 单文件 session memory（10 节模板） |
| #4 成本失控 | 留第 3 节 | 子 Agent 隔离 + Fork 缓存 + Coordinator 编排 |

</div>

### 10.2 你今天带走了什么

&emsp;&emsp;在学这一节之前，你对 Agent 的认知大概停在「一个带工具的 while 循环，三十行能写」。学完这一节，停下来问自己一句「这些我现在能做了吗」：徒手画出五层架构地图；向同事讲清 QueryLoop 为什么用异步生成器、`stop_reason` 为什么不能信；一句话说清四十多个工具凭什么被同一个循环调度；最重要的——你手里有三段已验证可运行的 MVP（Skill / MCP / Hook），它们不是演示玩具，是可以直接拆开抄进你自己 Agent 项目的内核：渐进披露省 token、协议即工具接外部能力、退出码协议做零耦合拦截。还有一句能用一辈子的选型口诀：加知识用 Skill（know）、加能力用 MCP（do）、设关卡用 Hook（intercept）。

&emsp;&emsp;贯穿这一节的，是一对张力——**能力 vs 约束**。这对张力的命名来自本系列课程的架构分析框架，不是源码术语；但它是这一节真正的脊柱：扩展三件套是油门，安全管线是刹车，一个敢上生产的 Agent，两者缺一不可。能力越强，越要约束——这不是一句口号，是 2592 行 `bashSecurity.ts` 替你数过的事实。

> 📌 **【系列结束 · 下次见面带这把尺】**：你现在手里有一把尺了——衡量任何一个 Agent「敢不敢上生产」，就看它在能力和约束这两条腿上各走了多远。第 3 节《多智能体与上下文工程》我们用同一把尺，去量剩下那三个问题：Agent 怎么在上下文爆炸前学会取舍、怎么跨会话不失忆、怎么让五个 Agent 协作却只花一个 Agent 的成本。同一份泄露快照，同样落到可验证的源码，我们下节继续拆。

### 10.3 三句话自测

&emsp;&emsp;最后给你三句话自测——读完这一节，下面三个问题你能不能在三十秒内答出？能，今天就真带走了。

&emsp;&emsp;第一，**为什么说一个工业级 Agent 里绝大多数代码不是在让 AI 更聪明？** 因为约束、支撑、兜底 AI 不确定性的确定性基础设施，才是代码主体；那约 1.6% 的决策逻辑（学术口径）才是少数——98.4% 在干「伺候和约束」的活。

&emsp;&emsp;第二，**为什么四十多个完全不同的工具能被同一个循环无差别调度？** 因为它们都实现了同一份 `Tool<IN, OUT>` 统一行为契约——循环只认契约，不认具体工具，所以扩展三件套才能正交地往上叠。

&emsp;&emsp;第三，**为什么说安全是「能力的必然反面」而不是一个独立功能？** 因为一个工具能帮你的程度恰好等于它能害你的程度——能力越强，约束代码必须越厚，2592 行 `bashSecurity.ts` 就是这句话的实测证据。

&emsp;&emsp;这三句话你要是都能脱口而出，第 1 节就真正闭环了。回到最开始那个数字——51 万行——它不再是一个吓人的体量，而是一份你能讲清结构、能徒手画图、能拿三段 MVP 抄进自己项目的认知资产。你今天最该带走的不是某个 API，而是那把「能力 vs 约束」的尺：从今天起，你看任何一个 Agent，都会下意识地量它在这两条腿上各走了多远。这把尺，下一节我们继续用它去拆剩下三个问题。

## <center>附录 A：五维通用挖掘提示词——任意开源子模块都能用</center>

&emsp;&emsp;前面我们用一段"开场总骨架"拆出了 `Claude Code` 整个项目的五层架构，又用四段"摸底 → 产 MVP → 讲透"的章内动手卡逐个吃透 `QueryLoop`、`Skill`、`MCP`、`Hook`、系统提示词五步组装这五个子模块。现在我们做一次抽象：把那五段动手卡背后的**元结构**抽出来，做成一个能套到**任何开源项目里的任何子模块**的轻量通用提示词。

&emsp;&emsp;这个附录解决一个非常具体的学员场景:你打开一个完全陌生的开源项目(比如 `LangChain` 的 `RetrievalQA`、`vLLM` 的 `KVCache`、`Vue` 的响应式系统),知道想挖某个模块,但**不知道第一句话该怎么问 AI**。常见的失败问法是「告诉我 X 模块怎么工作」「给我看看 X 的代码」「总结一下 X」——后面 A.4 会专门列这些反模式。这一节给出一份替代品:五个维度按顺序问,AI 给出可教学的结构化分析,并且每一条都能被你本地 `grep` 复核。

### A.1 第一性原理:学员挖陌生模块到底要解决什么

&emsp;&emsp;先把目标钉死。学员看陌生模块不是为了"看完代码",而是为了**得到一份能在自己项目里复用的认知资产**。这份认知资产要回答三类困惑——它们是后面五维提问框架的根:

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>学员面对陌生模块的三类困惑</font></p>
<div class="center">

| 困惑类型 | 学员真正在问 | 不回答会怎样 |
|----------|--------------|--------------|
| **动机层** | 这模块解决什么"不解决就会出事"的问题? | 学员只见 What 不见 Why,迁移时缺判断力 |
| **机制层** | 它的契约 / 数据流 / 源码锚点是什么? | 学员只见 Why 不见 How,无法落地复刻 |
| **验收层** | 怎么证明 AI 没瞎说、我没被带偏? | 学员吃 AI 幻觉而不自知,挖透变错觉 |

</div>

&emsp;&emsp;<font color=red>这三类困惑一一对应五维提问框架的"为什么、是什么、怎么做、有证据、可验证"五个角度</font>——不是凑数,是从困惑反推出来的最小必要集合。

### A.2 五维 MECE 提问框架

&emsp;&emsp;基于上面三类困惑,把提问拆成五个互不重叠、各有不可替代职责的维度。<font color=red>缺任意一维都会让回答塌掉</font>:缺 Why 学员只见 What 不知为什么需要,缺 What/How 学员只见动机不知怎么实现,缺 Where 你拿不到锚点没法复核(这一条是治 AI 幻觉的核心硬门控),缺 MVP 没有验收锚点学员无法证明自己挖透了。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>五维通用挖掘提问框架</font></p>
<div class="center">

| 维度 | 提问目标 | 为什么这维不能省 |
|------|----------|------------------|
| **Why · 动机** | 这模块解决什么不解决就崩的问题?给一个最小反例:朴素实现会在什么场景下爆 | 用**否定式**问,治"列优点的市场宣传文" |
| **What · 契约** | 对外暴露的协议是什么?必须实现 vs 可选实现的字段/方法分两栏列 | 强制分 required/optional,治"把示例当协议" |
| **How · 驱动** | 一次调用的关键调用栈 5-7 行(含 file:line),标"数据从哪进、关键转换点在哪、从哪出" | 限定行数 + 锚点,治"AI 写调用链小说" |
| **Where · 锚点** | 至少 5 处可 `grep` 的 `file:line + 符号`,要能本地 `grep -n` 复核 | **治幻觉的唯一硬门控**,无锚点的回答全部丢弃 |
| **MVP · 骨架** | ≤50 行 Python 标准库还原核心,末尾必带 self-assert,覆盖正常/边界/异常三种 scenario | self-assert 跑过才算挖透,**这是验收锚** |

</div>

&emsp;&emsp;此外还有一条**边界铁律**:哪些是源码事实(能 `grep` 出来)、哪些是教学抽象(源码里没有同名概念),AI 必须明确分开标注。这条不是新加的,是从课件第三章"约 1.6% 在决策"、§5.5"Tool 协议三层结构"、§4.3"QueryEngine 单文件 1295 行 vs 打包 4.6 万行"那些诚实划界处抽出来的元规则——所有教学抽象都必须背书清楚,否则学员会把作者的归纳当成源码官方名,带着错误认知去面试或迁移。

### A.3 可填空模板:30 行直接抄

&emsp;&emsp;下面这段提示词模板可以**直接复制粘贴到任何 LLM 对话框**(`Claude` / `GPT` / `DeepSeek` / `Gemini` 均适用)。学员只需填入开头四个变量:项目名、模块名、源码路径线索、(可选)已知入口符号。其余结构原样发送,得到的就是按五维结构化整理的分析报告。

```text
角色:资深源码导师,从源码事实 + 可观测行为重建陌生模块的机制与设计意图,
      不靠官方吹嘘、不写论文、不臆造 API。

【我要挖的模块】
  项目:{LangChain / vLLM / Vue Reactivity / Claude Code / ...}
  模块名:{RetrievalQA / KVCache / ref-track-trigger / QueryLoop / ...}
  源码路径线索:{langchain/chains/retrieval_qa/  或  src/v3-reactivity/}
  我已知的入口(可选):{class XxxChain.invoke()  或  function ref<T>(value)}

【五维分析 · 每维独立成段,缺一不交付】
  1) Why(动机):朴素实现在什么场景下会崩?给一个最小反例。
     —— 别列优点,用否定式说明它解决了不解决就出事的问题。
  2) What(契约):列出对外协议,分两栏:required 必须实现 / optional 可省。
     —— 别把示例当协议,列字段时必须能在源码 grep 到。
  3) How(驱动):一次调用的关键调用栈 5-7 行,每行含 file:line + 一句话作用。
     —— 重点标"数据从哪进、关键转换点在哪、从哪出"。
  4) Where(锚点):至少 5 处可 grep 的 file:line + 关键符号。
     —— 我要拿到本地 grep -n 一一复核,凡是无锚点的判断一律丢弃。
  5) MVP(骨架):≤50 行 Python 标准库还原核心,末尾带 self-assert,
     至少覆盖:正常路径 / 边界输入 / 异常退出 三个 scenario。
     —— 每个函数注释要带 # 对应 {file}:{line} 回引。

【边界铁律(不许违反)】
  - 显式分开"源码事实(grep 可证)"和"教学抽象(我为讲清楚归纳的)",
    两类各打标签,前者必须给锚点,后者必须说明为什么这么归纳。
  - 不许使用"通常 / 一般 / 可能 / 大概",说不知道就说"无源码证据,本项不答"。
  - 提问者(我)随时可以追问"这条 grep 哪行?"—— 你必须能给出。
```

&emsp;&emsp;这个模板有几个细节值得多说一句:一,**变量只有四个**,刻意保持极简,降低学员的填空成本,让"问 AI 挖模块"这件事的启动门槛接近于零;二,**五维顺序不能调**——Why → What → How → Where → MVP 是从抽象到具体、从动机到验收的递进,调换顺序会让 AI 在还没搞清动机时就开始铺细节;三,**边界铁律单独成块**是因为这是治幻觉的核心,如果跟提问条款混在一起,AI 会优先满足显性提问而忽略隐性约束。

### A.4 反模式三条:学员易踩,直接刷掉

&emsp;&emsp;学员第一次用五维模板,常常会"嫌长",或者还没养成用模板的习惯,回到老问法——下面这三种问法是最常见的踩坑路径,每一种背后都有一个对应的正问法:

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>陌生模块提问的反模式与正确问法</font></p>
<div class="center">

| 错问法 | 为什么废 | 对应正问法(五维里的哪一维) |
|--------|----------|------------------------------|
| "告诉我 X 模块怎么工作" | 太宽,AI 会写百科论文式回答,什么都讲一点 | 改问 **How 维度**,限定调用栈 5-7 行 |
| "给我看看 X 的代码" | AI 直接贴源码不思考,你拿到的是无解读的代码堆 | 改问 **Where 锚点**,让 AI 选出最关键的 5 处 |
| "总结一下 X 模块" | AI 写市场宣传稿,优点列一堆动机讲不清 | 改问 **Why 否定式**:"没有它会怎样" |

</div>

&emsp;&emsp;还有一类更隐蔽的反模式:**问完一轮就结束,不追问锚点**。这是最常见的"挖透假象"——AI 给你五维分析,你点头说"懂了",但其中可能有 2-3 处是它编出来的。<font color=red>必做动作:拿到回答后,把 Where 维度里给的 5 处 file:line 在本地终端 `grep -n` 复核一遍</font>,凡是 grep 不到的,要求 AI 重新给锚点或者明确改标"教学抽象"。这一步只需要 30 秒,但它是把"AI 挖透"真正变成"我挖透"的临门一脚。

### A.5 三套提示词的层级关系:何时用哪套

&emsp;&emsp;附录 A 不替代任何现有的提示词,而是夹在它们之间填补一个原本被默认掉的缺口。课件里前后总共出现了三套提示词,把它们放到同一张图上看,定位非常清楚:

```text
[课件开场 · 大型项目骨架]
    ↓  作用域:项目级
    ↓  产出:整体分层地图(L1-L5 五层架构)
    ↓
拆完发现某模块要深挖
    ↓
[附录 A · 五维通用挖掘模板] ←—— 本附录所在位置
    ↓  作用域:模块级
    ↓  产出:Why/What/How/Where 四维结构化分析(锚点可 grep)
    ↓
摸透了想动手复刻
    ↓
[附录 B · 模块复刻骨架]
       作用域:MVP 级
       产出:≤50 行 Python 可跑骨架 + self-assert 验收
```

&emsp;&emsp;课件里第四章 `QueryLoop`、第六章三件套、第八章系统提示词五步组装那几段章内动手卡,本质上就是这个通用模板**填了 `Claude Code` 具体锚点的特化实例**。把通用版抽出来,是让学员看到那几段动手卡背后的元结构——下次面对 `LangChain` 的 `RetrievalQA`、`vLLM` 的 `KVCache`、`Vue` 的响应式系统、`FastAPI` 的依赖注入,你拿同一份附录 A 模板填进去就能跑,不需要重新发明问法。

&emsp;&emsp;摸透了一个模块的五维结构之后,下一步是让 AI 帮你产一份可跑的 MVP 骨架,并用 self-assert 验证你抓住了核心——这就是附录 B 要做的事。

---

## <center>附录 B:模块复刻提示词骨架</center>

&emsp;&emsp;本课的方法论是「找到真实源码锚点 → 用 Python 还原最小可跑骨架 → 用 self-assert 验证结构」。如果你想把同样的方法用到自己的 Agent 项目或其他开源 Agent 上,下面这个通用提示词骨架可以帮你快速开始——把变量填入对应位置,把它丢给任何一个擅长代码的 LLM,得到的就是该模块的教学级复刻骨架。和附录 A 的关系:**附录 A 帮你"挖透机制",附录 B 帮你"产可跑骨架"**,两者前后衔接,合起来才是从陌生模块到迁移落地的完整链路。

```text
角色：你是一位资深 Agent 架构师，擅长从源码机制提炼最小可运行教学骨架。

输入（必填）：
  - 源码机制描述：{在此粘贴你从源码 grep 到的关键函数/类/字段定义，含 file:line 锚点}
  - 目标框架语言：{Python（推荐教学用）/ TypeScript / ...}

任务：基于以上机制，生成一个「最小可运行复刻骨架」：
  1. 只还原核心主链路，省略的部分用注释「完整版见 {file}:{line}」标注
  2. 用 mock 替代所有外部依赖（LLM / DB / 文件系统），使骨架无需任何第三方依赖可直接运行
  3. 末尾加 self-assert 测试，至少覆盖：正常路径、边界输入、异常退出 三个 scenario
  4. 每个函数必须有 docstring（Args / Returns），关键步骤加行内中文注释

输出约束：
  - 只基于我给的源码机制，不臆造未给出的 API 或字段
  - 代码注释含精确 file:line 回引（格式：# 对应 {file}:{line}）
  - self-assert 全部通过才算完成，如有 FAIL 必须先修复再交付
```

&emsp;&emsp;把这个骨架和下面这张差异填空表配合使用，效果最好——表里的五个模块对应本课前半场的五块内容，每模块给出最关键的 5-7 个源码填空点：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>模块复刻差异填空表（粘进通用提示词骨架的「源码机制描述」）</font></p>
<div class="center">

| 模块 | 关键源码填空点（file:line 锚点） |
|------|--------------------------------|
| **QueryLoop** | ① 退出 reason 枚举（`query.ts:646/977/996/1175/1264/1279/1711`）② 异步生成器状态机结构 ③ `stop_reason` 结构校验（不可信直接用）④ A2 防重入重试上限 ⑤ `max_turns` 硬截断逻辑 |
| **Tool 契约** | ① `inputSchema` 类型校验字段 ② `validateInput` 语义校验（型≠义）③ `isConcurrencySafe(input)` 动态方法签名 ④ `maxResultSizeChars` 落盘阈值字段 ⑤ `description` / `prompt` 进 system prompt 的注入点 |
| **Skill** | ① `SkillTool.ts:1108` 注入入口 ② `instructions` 字段进 system prompt 的渐进披露逻辑 ③ token 估算 `estimateSkillFrontmatterTokens` ④ `shouldDefer` 延迟加载判断 ⑤ 与 `ToolSearch` 召回的接口协议 |
| **MCP** | ① transport 枚举（`services/mcp/types.ts:24`：stdio/sse/sse-ide/http/ws/sdk + claudeai-proxy/ws-ide 等变体）② `MAX_MCP_DESCRIPTION_LENGTH=2048`（`client.ts:218`）③ `truncateMcpContentIfNeeded` 截断逻辑 ④ 401 单次强刷重试（`client.ts:365`）⑤ `MCP_AUTH_CACHE_TTL_MS=15min`（`client.ts:257`）⑥ 同构进 Tool 契约的字段映射 |
| **Hook** | ① 事件维索引（`utils/hooks/hooksConfigManager.ts:27`：`Record<HookEvent, …>`）② `matcherMetadata.fieldToMatch:'tool_name'` matcher 维过滤（`:33-53`）③ exit code 三态（0=继续/2=拦截/其他=警告）④ `runPreToolUseHooks`（`services/tools/toolExecution.ts:800` 调用，实现在 `toolHooks.ts`）⑤ hook 配置五层来源优先级（见第九章 SETTING_SOURCES） |

</div>

&emsp;&emsp;使用时，选中你想复刻的模块行，把「关键源码填空点」列的内容粘进通用骨架的「源码机制描述」变量，再把目标项目里对应的源码片段一起粘上去——这就是本课方法论「带着锚点让 AI 帮你复刻」的标准姿势。